# ETL VALIDATOR v10 · QA v2 (Comfandi / Quind)

v9 + celda de parámetros (FLUJOS_JSON, EJECUTAR_TXT, CORRIDA_QA) para lanzarlo con RunNotebook sin copias por lote.
Las mejoras v10 que corren al recoger el veredicto (roles reales desde el código y el _delta_log, pruebas reclasificadas,
analítica recalculada y el cotejo de flujo) las aplica el agente etl-validador desde local (`python -m pc etl recoger`).
Este notebook sirve también para mostrar una ejecución en vivo en Fabric.


# ETL VALIDATOR — versión Fabric · v9.0 (resultados/reportes_v3)

**Versión 2.0** · Comfandi / Quind S.A.S.

Ejecuta y valida flujos por su **nombre de negocio**, en el ambiente que se
indique. No necesita ningún archivo adicional: todo el motor vive en este
notebook y habla con Fabric por su API REST con el token de la sesión.

Cada celda declara su número de paso en la primera línea, así que para saber
cuál correr no hay que contarlas: basta leer el encabezado.

---

## Qué cambia en v9 · Variable Library

Hasta v8 muchas rutas salían como `{vl_core_...}` o apuntaban a la variable
equivocada, y el validador informaba «no existe en OneLake» sobre archivos que sí
existían. Tenía cuatro causas, y las cuatro quedan corregidas:

1. **Alias sin traducir.** Un Copy referencia `pipeline().libraryVariables.vl_core_x`
   y la variable se llama `x` dentro de `vl_core`. Ahora cada flujo resuelve su bloque
   `libraryVariables` (alias → biblioteca + variable).
2. **Bibliotecas mezcladas.** Todas se juntaban en un diccionario plano, y el mismo
   nombre vive en varias con valores distintos (`sftp_local_temp_dir`). Ahora cada
   alias va a *su* biblioteca, en el workspace del flujo: primero el suyo, luego
   orchestration y luego ingestion, con el *value set* activo aplicado.
3. **Flujos invocados.** Un `orq_*` baja a sus hijos, pero resolvía las
   expresiones del hijo con las variables del padre. Ahora cada actividad usa las
   del pipeline al que pertenece.
4. **Expresiones anidadas.** `split(...)[0]`, `replace`, `substring(..., add(...))`
   se partían por comas. Ahora hay un evaluador de expresiones de Data Factory. Lo
   que solo existe al correr (`item()`, `activity()`) queda vacío, como antes.

Además:

- Un `esquema.tabla` cuyo esquema no existe en el lakehouse (`credito`,
  `corporativo`, `SSF`, `Cliente`: Postgres) ya no se busca en `Tables/`.
- El nombre real de la tabla se toma sin distinguir mayúsculas
  (`sap_pscd_DFKKOP`).
- El prevuelo solo **bloquea** si la variable falta de verdad en la biblioteca o
  vale *placeholder*. Si el validador no pudo evaluar la expresión, **avisa**.
- El paso 5 muestra de qué biblioteca y variable sale cada valor.

---

## Antes de empezar

**Adjunta un lakehouse.** Panel izquierdo → *Add lakehouse*. Basta con
`lh_transversal` de QA. Sin él, el paso 12 no tiene dónde escribir los reportes
y la revisión se pierde al cerrar la sesión.

Luego edita solo el **paso 1** y ejecuta en el orden de abajo.

---

## Orden de ejecución

### Primera pasada — reconocimiento, no toca nada

Con `EJECUTAR = False`:

| Paso | Qué hace |
|---|---|
| **1** | Parámetros: los flujos, el ambiente y la bandera |
| **2** | Núcleo |
| **3** | Pasos |
| **4** | Resuelve cada nombre a un objeto real de Fabric |
| **5** | Entradas y salidas declaradas |
| **6** | Prevuelo: cierra con `EJECUTABLES: N de M` |

Si el prevuelo no deja todos los flujos ejecutables, para ahí: el paso 5 dice
qué bandera quedó sin resolver.

### Segunda pasada — ejecución

Vuelve al **paso 1**, pon `EJECUTAR = True` y córrelo de nuevo. Luego:

| Paso | Qué hace |
|---|---|
| **7** | Línea base: mide el destino **antes** de escribir |
| **8** | Lanza los flujos y espera |
| **9** | Vuelve a medir y compara |
| **10** | Informe consolidado |

El paso 7 va **antes** del 8. Al revés, el «antes» ya sería el «después» y la
comparación pierde sentido.

### Analítica — solo si hay flujos de analítica

| Paso | Qué hace |
|---|---|
| **11.1** a **11.3** | Funciones de lectura, medición y veredicto |
| **11.4** | La revisión: variables, ambiente, medición, salida y transformación |

Salta los flujos de ingesta y lo dice.

### Cierre

| Paso | Qué hace |
|---|---|
| **12** | Inventario: guarda lo que faltaba y lista cada archivo con su ruta |

---

## Resumen

```
1 → 2 → 3 → 4 → 5 → 6            EJECUTAR = False
1                                 EJECUTAR = True
7 → 8 → 9 → 10
11.1 → 11.2 → 11.3 → 11.4         solo con flujos de analítica
12
```

De los doce pasos, **el 8 es el único que actúa**, y solo con `EJECUTAR = True`.

---

## Qué NO hace

No compara contra Stratio. Mide el destino antes y después de la corrida, y en
analítica contrasta la entrada con la salida. La paridad contra Stratio sale de
los reportes de `resultados/comparaciones/`, que produce la aplicación de
métricas midiendo cada plataforma por separado.

---

## Qué queda en el lakehouse

| Carpeta | Contenido |
|---|---|
| `resultados/flujos/<fecha>/` | Consolidado de la corrida |
| `resultados/flujos/veredictos/` | Veredicto de cada flujo |
| `resultados/flujos/veredictos/<fecha>/` | Los mismos, por fecha |
| `resultados/analitica/<fecha>/` | Revisión individual y general de analítica |
| `resultados/informes/<fecha>/` | Índice con las cifras y la ruta de todo lo demás |

---

### v3 · qué cambia respecto a 2.0

1. **`resultados/reportes_v2/`**: carpeta nueva dentro de la de siempre,
   `Files/resultados/reportes_v2/validador/`. `resultados/` no cambia de nombre y lo que ya
   hay dentro no se mueve ni se toca; las carpetas nuevas se crean solas si no existen. El veredicto por flujo se escribe **una sola vez**, en
   `flujos/<flujo>/<CORRIDA>.json` (una carpeta por flujo, un archivo por corrida); el
   consolidado en `corridas/`, el informe en `informes/`, la analítica en `analitica/<flujo>/`.
2. **Paso 8.1 · Entradas y salidas tras la corrida** (sustituye al antiguo 5.1). Después de
   ejecutar, mide **solo** las rutas que cada flujo declara —sus entradas y sus salidas, ninguna
   otra— y las enmarca en una tabla. Cada ruta se mide una vez aunque la compartan varios flujos.
   El veredicto por flujo gana `entradas_y_salidas_tras_corrida`.
   **Paso 8.2 · Grafo de dependencias**: un interruptor al comienzo de la celda (`GRAFO = True`
   la ejecuta, `GRAFO = False` la salta). Cruza las rutas ya declaradas sin leer OneLake, y usa lo
   medido en 8.1 para `existencia`. El veredicto gana `dependencias` (`upstream`, `downstream`,
   aristas).
3. Salida declarada ilegible tras la corrida → `BLOQUEANTE`; leer filas y dejar las salidas en 0 →
   `BLOQUEANTE`; entrada o salida vacía → `ADVERTENCIA`; upstream con salidas vacías o ilegibles →
   `ADVERTENCIA`; ciclo → `ADVERTENCIA`.
4. Para enlazar con las métricas: el `CORRIDA` de este notebook se declara como
   `CORRIDA_VALIDADOR` en la CONFIG de los notebooks de métricas.

---

### v5.2 · qué cambia respecto a 5.1

1. **El lector se elige por la ubicación de la ruta, no por su extensión.** Lo que cuelga de
   `Files/` se abre primero como **parquet** —que es como la ingesta escribe bronze— y sólo
   después como Delta; lo que cuelga de `Tables/` sigue abriéndose como Delta. Antes, una
   carpeta sin extensión caía siempre en el lector Delta y devolvía `DELTA_TABLE_NOT_FOUND`.
2. **Una lectura fallida ya no se reporta como ausencia.** Antes de anotar el hallazgo se
   pregunta a OneLake si la ruta existe: si existe, el hallazgo es **ADVERTENCIA** («existe
   pero el motor no la pudo perfilar»); sólo si OneLake dice que no está es **BLOQUEANTE**;
   si no se puede confirmar, queda en advertencia y lo dice.
3. **Una salida escrita en partes se mide como carpeta, no como archivo.** Cuando el nombre
   del archivo depende de la iteración —`@concat('VBAP_', item(), '.parquet')`— la resolución
   dejaba el `item()` en blanco y se medía un literal `VBAP_.parquet` que no existe en ninguna
   corrida: el flujo escribe `VBAP_1.parquet`, `VBAP_2.parquet`… Ahora esa salida se resuelve a
   su carpeta, se guarda el patrón (`VBAP_*.parquet`) para el informe y se mide el conjunto.
4. **Una salida en `staging/` ya no se exige permanente.** Es un paso intermedio que el propio
   flujo consume y reescribe particionado —`sj_sappscd_dfkkko_partition_by_cpudt` y sus
   equivalentes—, así que su ausencia al terminar es lo esperado: queda como advertencia que
   remite al destino final, no como bloqueo.
5. **Una salida ilegible que sí existe en OneLake baja a advertencia**, con su número de
   archivos y su tamaño; sólo es bloqueo cuando OneLake confirma que la ruta no está.
6. Se registra con qué lector se leyó cada ruta (`FORMATO_LEIDO`) y si el fallo fue
   únicamente «no es Delta» (`SOLO_NO_DELTA`).

Efecto medido sobre la corrida `20260920T215442` (193 flujos, qa): de los 292 hallazgos
`Files/bronze/… DELTA_TABLE_NOT_FOUND`, **210 estaban comprobados como falso positivo** —la
ruta había sido medida y existía, o la escribió una ingesta que pasó en esa misma corrida— y
82 quedaban sin poder confirmar. Con este cambio 19 flujos de analítica pasan de bloqueados a
aprobados, 9 quedan en advertencia, y el conteo de bloqueos reales baja de 1.191 a 899.

Efecto de los puntos 3 a 5 sobre esa misma corrida: los cuatro flujos de ingesta que
reprobaban por «salida declarada ilegible» —`30-landing-sappscd-zsub-informetipi` (10 archivos
en bronze), `08-landing-sappscd-dfkkko` (998 archivos particionados por CPUDT),
`06-landing-sappscd-zsub-p-det-apo` y `07-landing-sappscd-zsub-p-tab-ctrl`— pasan, y a
`14-landing-sappscd-vbap` (9 archivos) le queda sólo el fallo real de su corrida. Ingesta queda
en 36 de 38 aprobados y la corrida completa en 66.

**Pendiente, no incluido en esta versión:** en cuatro flujos el descriptor se parte mal y
`overwrite` y `|` se toman como nombres de salida.

### v5.1 · qué cambia respecto a 4.0

1. **`_limpiar_ts()` pone la zona horaria que la API omite.** Sin ella, la resta contra
   `datetime.now(timezone.utc)` reventaba con «can't subtract offset-naive and offset-aware
   datetimes» y tumbaba el flujo entero. También arregla `escrita_por_la_corrida`, que fallaba en
   silencio por lo mismo.
2. **`huella()` ya no cae a la lectura por REST cuando Spark está presente.** Esa vía bajaba todos
   los parquet de la carpeta a la RAM del kernel y, con una entrada de bronze de varios GB, el
   nodo mataba el kernel («Ipython kernel exits with code -9»). Si Spark no puede leer, `leer()`
   guarda la causa real en `MOTIVOS` y se sigue. Fuera de Fabric, la lectura por REST tiene tope
   (`LIMITE_MB_LECTURA_REST`, 512 MB).
3. **`esperar()` consulta la corrida por su id**, no el historial completo del ítem cada 15 s.
   Menos memoria y menos llamadas, y ya no puede confundirse con una corrida ajena del mismo
   pipeline.
4. **Paso 8.1 y 8.2** (arriba). **El 5.1 se va.**

---

### v4 · qué cambia respecto a 3.0

1. **`reportes_v3`**: `Files/resultados/reportes_v3/validador/`. Mismas carpetas, nombres nuevos.
2. **Nombres con fecha y versión**, igual que en métricas:
   `analitica_<flujo>_<aaaa-mm-dd>_v<N>.json` para el veredicto de un flujo,
   `dependencias_<aaaa-mm-dd>_v<N>.json` y `informe_<aaaa-mm-dd>_v<N>.json`.
   La versión cuenta por flujo y se resuelve mirando lo que ya hay en la carpeta.
3. **El Paso 5.1 desaparece** y `DEPENDENCIAS_CON_TODOS` con él: era lo lento (una llamada a la
   API por cada flujo del ambiente, más una consulta a OneLake por ruta). Lo sustituyen 8.1 y 8.2,
   que trabajan con lo medido tras la corrida.
4. `corridas/` desaparece: el consolidado por corrida ya no es la unidad. Queda el informe.


---

### v5.3 · qué se arregla

**`TypeError: 'bool' object is not a mapping` en el Paso 12 · Cierre.**

`GRAFO` es el **interruptor** del Paso 8.2 (`GRAFO = True`), no el grafo. El grafo se llama
`RED_DEPENDENCIAS` y no salía de esa celda. El Paso 12 hacía:

```python
if globals().get("GRAFO"):          # True  -> entra
    ... {**_SELLO, **GRAFO, ...}    # **True -> TypeError
```

Ahora la guarda y el desempaquetado miran `RED_DEPENDENCIAS`. Se cambian **las dos**: dejar
`if GRAFO:` y tocar solo el `**` movería el fallo, porque con el interruptor en `True` y el
Paso 8.2 sin ejecutar `RED_DEPENDENCIAS` es `None` y `**None` vuelve a ser `TypeError`.

Efecto: el JSON de dependencias (`nodos`, `aristas` salida→entrada, `orden_topologico`,
`ciclos`, `cobertura`) por fin se escribe. Antes el Paso 12 moría antes de llegar a él.


---

### v5.4 · la medición ya no va en serie

`leer_pie()` hace **dos peticiones de rango** por parquet, y `medir()` las encadenaba sobre
`datos[:60]`: hasta **120 viajes de red seguidos por cada salida**. Ahí se iba el tiempo.

Esto **no es trabajo de Spark**, es HTTP contra OneLake. Los hilos esperan en la red, no en
CPU, así que aquí **no manda el `defaultParallelism`** de la sesión y subir de 8 sigue
rindiendo.

Se paralelizan dos niveles:

1. **los pies de parquet** dentro de una medición (lo gordo);
2. **las entradas y salidas** de un mismo flujo, que no dependen unas de otras.

Anidarlos multiplicaría las peticiones en vuelo (8 × 8 = 64), así que un **semáforo global**
mantiene el techo en `HILOS_RED` pase lo que pase.

`HILOS_RED = 8` en la celda de parámetros. Con `1` vuelve a ir en serie, para comparar
tiempos o si la API empieza a rechazar.

**El orden no cambia.** Los resultados se pliegan en el orden original y los `print` salen
como antes: el informe se lee igual, solo que tarda menos.


---

### v6 · flujos nuevos y dependencias declaradas

`FLUJOS` con los 13 flujos del lote, `AMBIENTE = "qa"`, `EJECUTAR = True`, y encima de la
lista las dependencias conocidas leídas de las definiciones.

**Dos correcciones sobre la lista tal y como llegó**, las dos justificadas por el propio
documento de dependencias:

1. **Faltaba `detalle_matriz_lealtad`** — el paso 5 de la cadena, el que produce
   `gold.comercial_detalle_matriz_lealtad`. Sin él, `metas_puntos_corte_04` vuelve a
   fallar por lo mismo que ya falló.
2. **El orden estaba mal.** El Paso 8 ejecuta `for obj in LISTOS`, es decir en el orden de
   `FLUJOS`, y `metas_puntos_corte_04` (paso 6) venía **antes** que `ingesta_cobertura`
   (paso 4). La cadena va ahora primera y en orden.

`cliente_servicios` se queda en la lista aunque va a fallar: tres de sus entradas no tienen
productor migrado en QA (`salud_pacientes`, `salud_agendamiento`, `zeus/InfoCobertura`).
Verlo fallar documentado es el objetivo; el porqué está escrito encima de la lista.

`ventas` y `clientes_mercadeo` quedan **comentados**: desbloquearían dos de las cinco
entradas de `cliente_servicios`, pero no estaban en el lote pedido.


---

### v7 · ejecución por olas

**`FLUJOS`**: las 143 entradas de la lista quedan en **133 únicas**. Las 10 repetidas son las
de fotos (`00-master-data`, `00-origenes`, `01-destinos`, `01-info-rep-legal`, `03-ape-repleg`,
`04-info-agrupadoras`, `07-info-total-empresas`, `00-info-cities`, `00-ibc`) más
`35-subsidio-en-especie-aforo`. Se conserva la primera aparición.

**Paso 8** deja de ser lanzar → esperar → lanzar → esperar. Ahora:

1. `plan_de_olas()` deriva las dependencias de las **entradas y salidas declaradas** — mismo
   criterio que `grafo_es()` del Paso 8.2: una salida de A es entrada de B si las rutas
   normalizadas coinciden o una es prefijo de la otra.
2. Agrupa en **olas**: dentro de una ola nada depende de nada, así que van **a la vez**
   (`HILOS_FLUJOS`, por defecto 4). Entre olas se espera.
3. Si detecta un **ciclo**, no adivina: lanza lo que queda en serie y lo dice.

**Lo único que supone:** un flujo que no declara entradas ni salidas no se puede probar
dependiente, así que va en paralelo. Sale contado en pantalla.

**Progreso en vivo.** Cada lanzamiento y cada final se imprimen en cuanto ocurren, así que
ves avanzar la ola sin esperar a que termine:

```
OLA 2/7 · 14 flujo(s) · 4 a la vez
   ▶ lanzado   01-tableros-afiliados-empresas    a1b2c3…
   ✅ 01-tableros-afiliados-empresas    Completed    2m 14s
   ❌ 04-tableros-afiliados-pacs        Failed       0m 31s
OLA 2 terminada: 13 OK · 1 con problema · 6m 02s
```

**El informe no cambia.** Toda la contabilidad (`anotar`, `cronometrar`, el detalle por
flujo) se hace **fuera de los hilos**, en serie y en el orden de la ola: no hace falta un
cerrojo por cada estructura compartida y la salida se lee igual que siempre.

`HILOS_FLUJOS` no lo limita el `defaultParallelism`: aquí solo se espera por corridas que
arrancan en Fabric. El techo es la **capacidad del workspace**. Si te pasas, Fabric encola.

Con `ORDEN_AUTOMATICO = False` vuelve al comportamiento de siempre, de uno en uno.


---

### v8 · Paso 7 en paralelo, y la lista real de Fabric

**`FLUJOS`: los 143 con su nombre real de `orchestration-qa`.** Sacados del CLI: 601 items
inventariados en los 19 workspaces QA. Los 9 pares de fotos vuelven a estar los dos
(`master_data` + `master_data_fotos`…), y `info_cities_foto` va en **singular**, que es como
está en Fabric.

**Paso 7.** Antes: un flujo detrás de otro y, por cada entrada y cada salida, un `count` más
un CRC32 sobre todas las columnas concatenadas. Dos recorridos completos por tabla, en serie.
Ahora, tres cambios que se acumulan:

1. **Las entradas no calculan checksum.** `revisar_entrada()` solo lee `filas` y `columnas`
   — el CRC32 se calculaba para nada. Ahí se va la mitad del trabajo de cada entrada.
2. **Cada ruta se mide una vez.** Una tabla que es salida de un flujo y entrada de otro se
   medía dos veces; ahora hay caché por ruta. El paso dice cuántas mediciones se ahorra.
3. **Las mediciones van en paralelo** (`HILOS_LINEA_BASE`, por defecto 4), con progreso en
   vivo. Los `print` y los `anotar` se hacen después, en serie y en el orden de siempre, así
   que el informe se lee igual.

Aquí sí es Spark, no HTTP: el techo lo pone el `defaultParallelism`. La ganancia de los
hilos es ×2–×3, no ×8 — pero se suma a la del checksum y la de la caché.


## Paso 1 · PARÁMETROS — la única celda que editas

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 1 · PARAMETROS — la única celda que editas
# ════════════════════════════════════════════════════════════════════════════
VERSION = "9.0"        # queda registrada en cada reporte

# ════════════════════════════════════════════════════════════════════════════
#  DEPENDENCIAS CONOCIDAS  ·  leido de las definiciones, no supuesto
# ════════════════════════════════════════════════════════════════════════════
#
#  CADENA DE metas_puntos_corte_04 — el orden de FLUJOS es el orden de ejecucion
#  (Paso 8 recorre `for obj in LISTOS`), asi que esta cadena va primera y en orden:
#
#    1. asignacion_empresas_jsol  ->  silver.comercial_asignacion_empresas_jsol
#    2. pedidos_ventas_jsol       ->  silver.comercial_pedidos_ventas_jsol
#    3. dim_ventas_acum           ->  gold.comercial_dim_ventas_acum        (lee el 2)
#    4. ingesta_cobertura         ->  Files/bronze/raw-files/matriz_lealtad/cobertura
#    5. detalle_matriz_lealtad    ->  gold.comercial_detalle_matriz_lealtad (lee 3 y 4)
#    6. metas_puntos_corte_04     ->  silver.comercial_puntos_corte_matriz_lealtad
#
#  El paso 5 necesita ademas silver.afiliados_trabajadores y silver.afiliados_pacs,
#  que YA existen. Si `detalle_matriz_lealtad` no resuelve, prueba con el nombre con el
#  que aparecia antes en el inventario: "03-detalle_matriz_lealtad".
#
#  cliente_servicios — NO se desbloquea ejecutando: le faltan entradas sin productor.
#
#    silver.mercadeo_transacciones_en_linea_ventas  <- flujo `ventas`  (DataPipeline,
#                                                      orchestration-qa) EJECUTABLE YA
#    silver.mercadeo_clientes_mercadeo              <- `clientes_mercadeo`, solo existe
#                                                      como SparkJobDefinition en
#                                                      mercadeo-qa, SIN pipeline: hay que
#                                                      invocarlo directo o crearselo
#    silver.salud_pacientes                         <- NINGUN flujo migrado en QA
#    silver.salud_agendamiento                      <- NINGUN flujo migrado en QA
#    Files/bronze/zeus/InfoCobertura                <- falta la ingesta de zeus
#
#  Las seis silver.cliente360_cliente_360* que el validador marca como bloqueo NO son
#  entradas: son las SALIDAS del propio cliente_servicios. Salen en rojo porque todavia
#  no existen, y se crean cuando el flujo corra bien. No busques quien las produce.
#
#  Con tres entradas sin productor, cliente_servicios va a fallar. Se deja en la lista
#  porque medirlo y ver el fallo documentado es el objetivo, no un accidente.
# ════════════════════════════════════════════════════════════════════════════

FLUJOS = [
    # ══ Cadena de la matriz de lealtad · EL ORDEN IMPORTA ════════════════════
    #    El motor de olas (Paso 8) la reordena solo si estos flujos declaran sus
    #    entradas y salidas. Se deja primera y en orden como red de seguridad:
    #    si alguno NO las declara, este orden es lo unico que lo salva.
    "asignacion_empresas_jsol",                 # asignacion_empresas_jsol
    "pedidos_ventas_jsol",                      # pedidos_ventas_jsol
    "dim_ventas_acum",                          # 01-dim-ventas_acum
    "ingesta_cobertura",                        # 02_ingesta_cobertura
    "detalle_matriz_lealtad",                   # 03-detalle_matriz_lealtad
    "metas_puntos_corte_04",                    # 04-metas-puntos-corte

    # ══ El resto · en el orden de tu lista; el motor de olas decide ══════════
    "metricas_td",                              # 01-metricas-td
    "keycloak_prod",                            # 01-keycloak-prod
    "familiar_recurrente_fecha",                # 02a-tableros-giass-familiarrecurrente_fecha
    "tableros_aportes_aportes_05",              # 05-tableros-aportes-aportes
    "tableros_aportes_mora_06",                 # 06-tableros-aportes-mora
    "tableros_aportes_pila_13",                 # 13-tableros-aportes-pila
    "prevalidador_2069_29",                     # 29-prevalidador-2069
    "estado_postulaciones_fovis_33",            # 33-estado-postulaciones-fovis
    "postulaciones_rechazadas_fosfec_fovis",    # 43-postulaciones-rechazadas-fosfec-fovis
    "reporte_log_admin_lista_chequeo",          # 47-reporte-log-admin-lista-chequeo
    "riesgos_token_otp",                        # 29-riesgos-token_otp
    "riesgos_cambio_celular",                   # 30-riesgos-cambio-celular
    "reporte_cuota_anulados_ssf",               # 02-reporte-cuota-anulados-ssf
    "postulaciones_detalle_chequeo_fosfec_fovis",# 48-postulaciones-detalle-chequeo-fosfec-fovis
    "rq1_bd_inicial_tt_afiliados",              # rq1_bd_inicial_tt_afiliados
    "composicion_periodo_cierre_pago",          # composicion_periodo_cierre_pago
    "td_estado_postulaciones_sfv_02",           # 02-td-estadopostulaciones-sfv
    "td_fosfec_estadoPostulaciones_05",         # 05-td-fosfec-estadopostulaciones
    "td_fosfec_postulacionesDevueltas_06",      # 06-td-fosfec-postulacionesdevueltas
    "subsidio_en_especie_aforo",                # 35-subsidio-en-especie-aforo
    "td_afiliaciones",                          # 11-td-afiliaciones
    "ra2_usuarios_sve_reporte_con_filtros_14",  # 14-ra2-usuarios-sve-reporte-con-filtros
    "ra2_solicitudes_afiliaciones_sin_trazabilidad",# 15-ra2-solicitudes-afiliaciones-(sin trazabilidad)
    "ra2_inspektor_afiliaciones_17",            # 17-ra2-inspektor-afiliaciones
    "ra2_usuarios_sus_18",                      # 18-ra2-usuarios-sus
    "ra2_consulta_inspektor",                   # 20-ra2-consulta-inspektor
    "reporte_ugpp",                             # reporte_ugpp
    "subsidio_en_especie_aforo",                # 35-subsidio-en-especie-aforo
    "envios",                                   # 04-envios
    "reenvios",                                 # 05-reenvios
    "info_cities",                              # 00-info-cities
    "ibc",                                      # 00-ibc
    "model_excecution",                         # 01-model-excecution
    "subsidios_aportes_deficit_postgres",       # 05-subsidios-aportes-deficit-postgres
    "calculo_trab_ben_cuota_monetaria",         # 06-calculo-trab-ben-cuota-monetaria
    "start_fotos",                              # start
    "cliente_servicios",                        # cliente_servicios
    "empresa_servicios",                        # empresa_servicios
    "visitas_gcv_empresa",                      # visitas_gcv_empresa
    "pedidos_sin_cerrar_jsol",                  # pedidos_sin_cerrar_jsol
    "transformacion_contact_comercial",         # transformacion_contact_comercial
    "educacion",                                # 05_educacion
    "educacion_estado",                         # 06_educacion_estado
    "turismo",                                  # 10_turismo
    "recreacion",                               # 14_recreacion
    "calendario",                               # 03-calendario
    "exogena_ingresos",                         # exogena-ingresos
    "medios_pago",                              # 01-medios-pago
    "core_validaciones_validador_subsidio_06b", # 06b-core-validaciones-validador-subsidio
    "flujo_control_ppto",                       # 01-flujo-control-ppto
    "ibc_fotos",                                # 00-ibc
    "info_cities_foto",                         # 00-info-cities
    "ssf_afiliados_ssf_empresas_01",            # 01-ssf-afiliados-ssf-empresas
    "ssf_afiliados_ssf_afiliados_02",           # 02-ssf-afiliados-ssf-afiliados
    "ssf_afiliados_ssf_pacs_03",                # 03-ssf-afiliados-ssf-pacs
    "ssf_afiliados_ssf_empresas_historico_04",  # 04-ssf-afiliados-ssf-empresas-historico
    "ssf_afiliados_ssf_afiliados_historico_05", # 05-ssf-afiliados-ssf-afiliados-historico
    "ssf_afiliados_ssf_pacs_historico_06",      # 06-ssf-afiliados-ssf-pacs-historico
    "poblacion_afiliada_empresas_13",           # 13-poblacion-afiliada-empresas
    "poblacion_afiliada_afiliados_14",          # 14-poblacion-afiliada-afiliados
    "poblacion_afiliada_pacs_15",               # 15-poblacion-afiliada-pacs
    "poblacion_afiliada_fact_16",               # 16-poblacion-afiliada-fact
    "tableros_afiliados_empresas_01",           # 01-tableros-afiliados-empresas
    "tableros_afiliados_contactabilidad_02",    # 02-tableros-afiliados-contactabilidad
    "tableros_afiliados_trabajadores_03",       # 03-tableros-afiliados-trabajadores
    "tableros_afiliados_pacs_04",               # 04-tableros-afiliados-pacs
    "tableros_subsidios_subsidio_07",           # 07-tableros-subsidios-subsidio
    "tableros_servicios_postulaciones_fosfec",  # 08-tableros-servicios-postulaciones-fosfec
    "tableros_servicios_beneficios_fosfec_09",  # 09-tableros-servicios-beneficios-fosfec
    "tableros_afiliados_trabajadores_fosfec_10",# 10-tableros-afiliados-trabajadores-fosfec
    "tableros_web_service_16",                  # 16-tableros-web-service
    "tableros_fovis_prevalidadorFovis_21",      # 21-tableros-fovis-prevalidadorfovis
    "tableros_servicios_beneficios_fovis_23",   # 23-tableros-servicios-beneficiosfovis
    "tableros_cartera_26",                      # 26-tableros-cartera
    "tableros_vivienda_negocio_27",             # 27-tableros-vivienda-negocio
    "sabana_sub_especie_dinamico",              # 34-sabana-sub-especie-dinamico
    "reporte_credito_subsidio_saldo_periodo_46",# 46-reporte-credito-subsidio-saldo-periodo
    "reporte_cuota_consolidado_ssf",            # 03-reporte-cuota-consolidado-ssf
    "reporte_cuota_detalle_ssf",                # 01-reporte-cuota-detalle-ssf
    "periodo_contable_cierre",                  # periodo-contable-cierre
    "tabla_base_reporte_aportes_v6",            # tabla-base-reporte-aportes
    "reporte_unidad_publica_empleo",            # reporte-unidad-publica-empleo
    "reporte_upe_empresas",                     # reporte-upe-empresas
    "ra2_consulta_afiliados_reporte_16",        # 16-ra2-consulta-afiliados-reporte
    "ra2_consulta_afiliados_beneficiarios_sin_info_contacto",# 21-ra2-consulta-afiliados-beneficiarios-sin-info-contacto
    "ra2_reporte_pensionados",                  # 22-ra2-reporte-pensionados
    "ra2_reporte_historico_afiliaciones",       # 26-ra2-reporte-historico-afiliaciones
    "ra2_reporte_consulta_subsidio",            # 28-ra2-reporte-consulta-subsidio
    "core_validaciones_afiliados_02",           # 02-core-validaciones-afiliados
    "core_validaciones_pac_03",                 # 03-core-validaciones-pac
    "core_validaciones_empresas_04",            # 04-core-validaciones-empresas
    "core_validaciones_alertas_cm_07",          # 07-core-validaciones-alertascm
    "vivienda_solicitudes_01",                  # 01-vivienda-solicitudes
    "vivienda_reporte_acompaniamiento_02",      # 02-vivienda-reporte-acompaniamiento
    "ssf_dimensiones_19",                       # 19-ssf-dimensiones
    "ssf_facts_20",                             # 20-ssf-facts
    "info_rep_legal",                           # 01-info-rep-legal
    "ape_repleg",                               # 03-ape-repleg
    "info_agrupadoras",                         # 04-info-agrupadoras
    "info_total_empresas",                      # 07-info-total-empresas
    "origenes",                                 # 00-origenes
    "destinos",                                 # 01-destinos
    "master_data",                              # 00-master-data
    "Tabla_cosechas",                           # tabla_cosechas
    "trusted_datarmat_credito_clientes",        # 01-trusted-datarmat-credito-clientes
    "trusted_datarmat_credito_empresa",         # 02-trusted-datarmat-credito-empresa
    "trusted_datamart_credito_detalle_prestamo",# 03-trusted-datamart-credito-detalle-prestamo
    "trusted_datarmat_credito_prestamo",        # 04-trusted-datarmat-credito-prestamo
    "trusted_datarmat_credito_cartera_fact",    # 05-trusted-datarmat-credito-cartera(fact)
    "trusted_datamart_credito_niveles_fact",    # 06-trusted-datamart-credito-niveles(fact)
    "mdt_final",                                # mdt_final
    "predicciones_k_means",                     # predicciones_k_means
    "afiliados",                                # 01_afiliados
    "empresas",                                 # 03_empresas
    "ventas",                                   # 09_ventas
    "locaciones_centro_costos",                 # 01-locaciones-centro-costos
    "recaudos",                                 # recaudos
    "informe_autorretenciones",                 # informe-autorretenciones
    "clientes_mercadeo",                        # 02-clientes-mercadeo
    "productos_articulo",                       # 02-productos-articulo
    "mercadeo_ventas",                          # 01_ventas
    "reporte_ya_preinscriptos_cm_27B",          # 27b-reporte-ya-preinscriptos-cm
    "reporte_proximos_preinscribir_cm_27a",     # 27a-reporte-proximos-preinscribir-cm
    "master_data_fotos",                        # 00-master-data
    "origenes_fotos",                           # 00-origenes
    "destinos_fotos",                           # 01-destinos
    "info_rep_legal_fotos",                     # 01-info-rep-legal
    "vacaciones",                               # 01-vacaciones
    "last_partner",                             # 02-last-partner
    "tasas",                                    # 02-tasas
    "ape_repleg_fotos",                         # 03-ape-repleg
    "moves",                                    # 03-moves
    "sabana_pila",                              # 03-sabana-pila
    "info_agrupadoras_fotos",                   # 04-info-agrupadoras
    "info_general_empresas",                    # 05-info-general-empresas
    "info_salarial",                            # 06-info-salarial
    "info_total_empresas_fotos",                # 07-info-total-empresas
]

AMBIENTE = "qa"        # qa | dev | prod
EJECUTAR = True        # True solo cuando el prevuelo salga EJECUTABLE
# A partir de aqui la duracion es SOSPECHOSA, pero no se deja de contar: una
# corrida larga sigue siendo una corrida, y abandonar el conteo es justo lo que
# impide saber cuanto tardo de verdad.
TOPE_SOSPECHA = 1800   # segundos: se avisa, se sigue esperando y contando
TOPE_ESPERA = 14400    # segundos: techo absoluto, para no esperar sin fin

# Raiz de los resultados. Sale al lakehouse adjunto.
# reportes_v2: carpeta NUEVA dentro de la de siempre. Files/resultados/ no cambia de nombre y
# lo que ya hay dentro no se mueve ni se toca; estas se crean solas si no existen.
#   .../reportes_v2/validador/flujos/<flujo>/<CORRIDA>.json   el veredicto por flujo, UNA vez
#   .../reportes_v2/validador/corridas/corrida_<CORRIDA>_<N>flujos.json
#   .../reportes_v2/validador/dependencias/<CORRIDA>.json     el grafo entradas/salidas
#   .../reportes_v2/validador/informes/informe_<CORRIDA>.json
#   .../reportes_v2/validador/analitica/<flujo>/<CORRIDA>.json
RUTA_RESULTADOS = "/lakehouse/default/Files/resultados/reportes_v3/validador"

# Dependencias entre flujos por coincidencia de SALIDAS de uno con ENTRADAS de otro.
# True  -> se leen las entradas/salidas de TODOS los DataPipeline y SparkJobDefinition de los
#          workspaces del ambiente (solo descriptores, sin Spark), para que el grafo tenga con
#          que cruzar. Es lo que hace falta para saber de quien depende un flujo.
# False -> solo se cruzan los flujos de FLUJOS entre si.  <- por defecto: es lo rapido.
# En True lee el descriptor de TODOS los flujos del ambiente (una llamada a la API por flujo):
# da el grafo completo pero tarda varios minutos con 40+ flujos.
# Comprobar contra OneLake que cada entrada y salida declarada EXISTE (HEAD al DFS, sin Spark).
VERIFICAR_EXISTENCIA = True

# Sin las comas, Python pega los literales y los N nombres se vuelven uno solo:
# el validador buscaria un flujo inexistente de 500 caracteres y resolveria por
# aproximacion a uno cualquiera. Es un error mudo, asi que aqui se corta.
_raros = [f for f in FLUJOS if len(f) > 60 or f.count("-landing-") > 1]
if _raros:
    raise ValueError(
        f"{len(_raros)} entrada(s) de FLUJOS parecen varios nombres pegados: "
        f"revisa que cada linea termine en coma.\n  " + "\n  ".join(
            f[:90] + ("..." if len(f) > 90 else "") for f in _raros))
if len(set(FLUJOS)) != len(FLUJOS):
    print(f"aviso: hay nombres repetidos en FLUJOS ({len(FLUJOS)} entradas, "
          f"{len(set(FLUJOS))} distintas)")
# Hilos para la MEDICION. No es trabajo de Spark: es HTTP contra OneLake, asi que aqui
# no manda el defaultParallelism de la sesion —los hilos esperan en la red, no en CPU— y
# subir de 8 sigue rindiendo. Es un techo GLOBAL: aunque se paralelice por flujo y por
# parquet a la vez, nunca hay mas de HILOS_RED peticiones de pie en vuelo.
#   1 = como antes, todo en serie (para comparar tiempos o si la API empieza a rechazar)
# Flujos que se lanzan A LA VEZ. No es Spark de este notebook: son corridas que arrancan
# en Fabric y aqui solo se espera por ellas, asi que el limite no es el defaultParallelism
# sino la CAPACIDAD del workspace. Si te pasas, Fabric las encola: no fallan, pero tampoco
# van mas rapido. 4 es prudente para empezar; subelo mirando la capacidad.
# Hilos del PASO 7 (linea base). Aqui SI es Spark: cuenta y hashea de verdad, asi que el
# techo lo pone el defaultParallelism de la sesion, no la red. Con 4 sobre 8 slots se llenan
# los huecos que hoy quedan ociosos mientras una tabla hace su count; no esperes x8.
# 8 = los slots de tarea de la sesion. Por encima los jobs hacen cola en el planificador y
# cada uno mantiene su plan vivo en el driver, que es donde se cae una sesion. No subir a 16.
HILOS_LINEA_BASE = 8

# El techo aqui NO es este numero: es la CAPACIDAD (CU) del workspace. Si no caben, Fabric
# los encola —no fallan, pero tampoco van mas rapido—. Y si varios flujos escriben la misma
# tabla Delta, mas concurrencia = mas conflictos de escritura. 8 es seguro; 16, segun SKU.
HILOS_FLUJOS = 8

# True  = el orden lo decide el motor de olas, leyendo las entradas y salidas declaradas.
# False = se lanzan en el orden de FLUJOS, de uno en uno (el comportamiento de siempre).
ORDEN_AUTOMATICO = True

# HTTP puro contra OneLake: los hilos esperan en la red, no en CPU ni en slots de Spark.
# Es el unico de los tres que escala de verdad por encima de 8.
HILOS_RED = 16

print(f"ETL Validator {VERSION} · {len(FLUJOS)} flujo(s) solicitados en {AMBIENTE}")
print(f"Medicion en paralelo: {HILOS_RED} hilo(s) de red"
      + ("  (en serie)" if int(HILOS_RED) <= 1 else ""))
print("EJECUTAR = " + ("True · se lanzarán los flujos" if EJECUTAR
                       else "False · solo lectura"))

In [ ]:
# QA v2 · PARAMETROS (celda "parameters": RunNotebook los sobreescribe)
FLUJOS_JSON = ""      # lista JSON de flujos; vacio = la lista del Paso 1
EJECUTAR_TXT = ""     # "true" | "false"; vacio = lo del Paso 1
CORRIDA_QA = ""       # id de la corrida QA v2 que lanza esto (trazabilidad)


In [ ]:
# QA v2 · aplica los parametros recibidos sobre el Paso 1
import json as _json_qa
if FLUJOS_JSON:
    FLUJOS = _json_qa.loads(FLUJOS_JSON)
if EJECUTAR_TXT:
    EJECUTAR = EJECUTAR_TXT.strip().lower() == "true"
print(f"QA v2 · corrida {CORRIDA_QA or '-'} · {len(FLUJOS)} flujo(s) · EJECUTAR={EJECUTAR}")


## Paso 2 · NÚCLEO — no editar

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 2 · NÚCLEO — no editar
# ════════════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════════
#  NÚCLEO — no requiere editarse. Todo el motor del validador vive aquí.
# ══════════════════════════════════════════════════════════════════════════
import base64, json, re, time
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone

API = "https://api.fabric.microsoft.com/v1"
EJECUTABLES = ("DataPipeline", "SparkJobDefinition", "Notebook")
_TIPO_TRABAJO = {"DataPipeline": "Pipeline", "SparkJobDefinition": "sparkjob",
                 "Notebook": "RunNotebook"}
HALLAZGOS = []          # dicts: nivel, flujo, prueba, objeto, mensaje, esperado, obtenido
TIEMPOS = {}            # flujo -> {etapa: segundos}
ORDEN = {"OK": 0, "OMITIDO": 1, "ADVERTENCIA": 2, "BLOQUEANTE": 3}
_FLUJO = ""


def en_flujo(nombre):
    """Marca a que flujo se atribuyen los hallazgos siguientes."""
    global _FLUJO
    _FLUJO = str(nombre or "")


def anotar(nivel, objeto, mensaje, *, prueba="", esperado=None, obtenido=None):
    """Registra un hallazgo.

    `esperado` y `obtenido` se guardan aparte del mensaje para que el informe
    por flujo pueda decir que se comprobo y contra que, sin volver a parsear
    texto.
    """
    HALLAZGOS.append({"nivel": nivel, "flujo": _FLUJO, "prueba": prueba,
                      "objeto": objeto, "mensaje": mensaje,
                      "esperado": esperado, "obtenido": obtenido})
    marca = {"OK": "  ok  ", "OMITIDO": " omit ",
             "ADVERTENCIA": " adv  ", "BLOQUEANTE": "BLOQUE"}[nivel]
    print(f"  [{marca}] {objeto}: {mensaje}")


def cronometrar(flujo, etapa, segundos):
    TIEMPOS.setdefault(flujo, {})[etapa] = round(float(segundos), 2)


def veredicto(hallazgos=None):
    """El peor nivel del conjunto. OK si no hay nada."""
    fuente = HALLAZGOS if hallazgos is None else hallazgos
    return max((h["nivel"] for h in fuente), key=lambda n: ORDEN[n], default="OK")


def de_flujo(nombre):
    return [h for h in HALLAZGOS if h["flujo"] == nombre]


def resultado(hallazgos):
    """Responde "paso o no" en una palabra, que es lo que se quiere leer.

    `veredicto()` devuelve el peor nivel, y ahi un OMITIDO estructural -como el
    origen SFTP que no se puede leer desde aqui- tapa las once comprobaciones
    que si pasaron. Este separa las dos cosas: si algo fallo, y si algo quedo
    sin verificar.
    """
    niveles = {h["nivel"] for h in hallazgos}
    if not hallazgos:
        return "SIN_PROBAR"
    if "BLOQUEANTE" in niveles:
        return "NO_PASA"
    if "ADVERTENCIA" in niveles:
        return "PASA_CON_AVISOS"
    if "OMITIDO" in niveles:
        return "PASA_CON_OMISIONES"
    return "PASA"


def pasa(hallazgos):
    """True si nada bloquea. Un OMITIDO no reprueba: avisa que no se verifico."""
    return not any(h["nivel"] == "BLOQUEANTE" for h in hallazgos)


# ─────────────────────────────────────────────────────────── API REST
_CLI = {}


def _token_cli(scope):
    """Token via la CLI `fab`, para poder correr este notebook fuera de Fabric.

    Dentro del servicio nunca se llega aqui: notebookutils responde primero.
    """
    import shutil
    import subprocess
    if scope in _CLI:
        return _CLI[scope]
    fab = shutil.which("fab")
    if not fab:
        return ""
    with open(fab, encoding="utf-8", errors="ignore") as f:
        piso = f.readline().strip()
    py = piso[2:].strip() if piso.startswith("#!") else "python3"
    guion = ("from fabric_cli.core import fab_constant as con\n"
             "from fabric_cli.core.fab_auth import FabAuth\n"
             f"print(FabAuth().get_access_token(con.{scope}, "
             "interactive_renew=False) or '')")
    try:
        r = subprocess.run([py, "-c", guion], capture_output=True, text=True, timeout=120)
        salida = [l for l in r.stdout.strip().splitlines() if l.startswith("ey")]
        _CLI[scope] = salida[-1] if salida else ""
    except Exception:
        _CLI[scope] = ""
    return _CLI[scope]


def _token(recurso="pbi"):
    """Token de la sesion. Dentro de Fabric lo da notebookutils; fuera, la CLI."""
    for mod in ("notebookutils", "mssparkutils"):
        try:
            return __import__(mod).credentials.getToken(recurso)
        except Exception:
            continue
    scope = ("SCOPE_ONELAKE_DEFAULT" if recurso == "storage"
             else "SCOPE_FABRIC_DEFAULT")
    tok = _token_cli(scope)
    if tok:
        return tok
    raise RuntimeError(
        "Sin token de Fabric. Dentro del servicio es automatico; "
        "fuera, autentica con `fab auth login`.")


def _cuerpo_json(crudo):
    """La API a veces responde vacio, o `null`, o una lista. Aqui se normaliza."""
    if not str(crudo).strip():
        return {}
    try:
        datos = json.loads(crudo)
    except Exception:
        return {}
    return datos if isinstance(datos, dict) else {"value": datos}


def api_bruta(ruta, metodo="GET", cuerpo=None, timeout=120):
    """Llamada cruda. Devuelve (codigo, json, cabeceras) y no interpreta el 202."""
    datos = json.dumps(cuerpo).encode() if cuerpo is not None else None
    pet = urllib.request.Request(
        f"{API}/{ruta.lstrip('/')}", data=datos, method=metodo,
        headers={"Authorization": f"Bearer {_token()}",
                 "Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(pet, timeout=timeout) as r:
            return r.status, _cuerpo_json(r.read().decode()), dict(r.headers)
    except urllib.error.HTTPError as e:
        return e.code, (_cuerpo_json(e.read().decode()) or {"error": str(e)}), dict(e.headers)
    except Exception as e:
        return 0, {"error": str(e)}, {}


def api(ruta, metodo="GET", cuerpo=None, timeout=120, seguir=True):
    """Llamada a la API de Fabric. Devuelve (codigo, json). Nunca lanza.

    Un 202 con `x-ms-operation-id` es una operacion asincrona y se espera, que
    es lo que hace falta para el getDefinition de items grandes. Un 202 sin esa
    cabecera no lo es: al lanzar un trabajo, el 202 ES la respuesta buena y lo
    que llega es la corrida, no una operacion que consultar.
    """
    codigo, respuesta, cab = api_bruta(ruta, metodo, cuerpo, timeout)
    op = cab.get("x-ms-operation-id")
    if codigo == 202 and seguir and op:
        espera = cab.get("Retry-After")
        return _esperar_operacion(op, float(espera) if espera else 2.0)
    return codigo, respuesta


def _esperar_operacion(op, espera=2.0, tope=180):
    """Espera una operacion asincrona y devuelve su resultado."""
    limite = time.time() + tope
    while time.time() < limite:
        codigo, estado = api(f"operations/{op}", seguir=False)
        if codigo != 200:
            return codigo, estado
        situacion = str(estado.get("status", "")).lower()
        if situacion == "succeeded":
            return api(f"operations/{op}/result", seguir=False)
        if situacion in ("failed", "undefined"):
            return 500, {"error": f"operacion {situacion}",
                         "detalle": estado.get("error")}
        time.sleep(max(1.0, min(espera, 10.0)))
    return 0, {"error": f"operacion {op} sin terminar tras {tope}s"}


class FabricNoResponde(RuntimeError):
    """La API no respondio.

    Se separa a proposito de "respondio y no hay nada": un listado que falla y
    un workspace vacio son cero items en las dos lecturas, y confundirlos hace
    que el validador declare ausente un flujo que si esta desplegado.
    """


_REINTENTABLES = (0, 429, 500, 502, 503, 504)


def _insistir(ruta, intentos=4):
    """GET con reespera creciente. Lanza si no logra un 200."""
    espera = 2.0
    ultimo = ""
    for _ in range(intentos):
        codigo, cuerpo = api(ruta)
        if codigo == 200:
            return cuerpo
        ultimo = f"HTTP {codigo} {json.dumps(cuerpo, ensure_ascii=False)[:160]}"
        if codigo not in _REINTENTABLES:
            raise FabricNoResponde(f"{ruta}: {ultimo}")
        time.sleep(espera)
        espera = min(espera * 2, 30)
    raise FabricNoResponde(f"{ruta}: sin respuesta usable tras {intentos} intentos ({ultimo})")


def paginar(ruta):
    """Recorre continuationToken y devuelve todos los `value`."""
    fuera, token = [], None
    while True:
        sep = "&" if "?" in ruta else "?"
        pagina = _insistir(
            ruta + (f"{sep}continuationToken={urllib.parse.quote(token, safe='')}"
                    if token else ""))
        fuera += pagina.get("value", [])
        token = pagina.get("continuationToken")
        if not token:
            return fuera


# ─────────────────────────────────────────────────────── inventario
_WS, _ITEMS = {}, {}


def workspaces(refrescar=False):
    """{nombre: id} de todos los workspaces visibles."""
    if _WS and not refrescar:
        return _WS
    encontrados = {str(w.get("displayName")): str(w.get("id"))
                   for w in paginar("workspaces")}
    _WS.clear()
    _WS.update(encontrados)
    return _WS


def workspaces_de(ambiente):
    return {n: i for n, i in workspaces().items()
            if n.startswith("fabric-comfandi-") and n.endswith(f"-{ambiente}")}


def items(ws_nombre):
    """[{id,displayName,type}] del workspace. Solo se cachea el exito."""
    if ws_nombre in _ITEMS:
        return _ITEMS[ws_nombre]
    wid = workspaces().get(ws_nombre)
    if not wid:
        raise FabricNoResponde(f"workspace no visible: {ws_nombre}")
    _ITEMS[ws_nombre] = paginar(f"workspaces/{wid}/items")
    return _ITEMS[ws_nombre]


def definicion(ws_nombre, item_id, sufijo):
    """Una parte de la definición del item, decodificada por el final de su ruta."""
    wid = workspaces().get(ws_nombre)
    codigo, cuerpo = api(f"workspaces/{wid}/items/{item_id}/getDefinition", "POST", {})
    if codigo not in (200, 202):
        return None
    for parte in (cuerpo.get("definition") or {}).get("parts", []):
        if str(parte.get("path", "")).endswith(sufijo):
            try:
                return json.loads(base64.b64decode(parte.get("payload", "")).decode())
            except Exception:
                return None
    return None


_VARS = {}


def bibliotecas(ambiente):
    """{nombre visible: (workspace, id)} de cada Variable Library del ambiente."""
    fuera = {}
    for ws in workspaces_de(ambiente):
        for it in items(ws):
            if it.get("type") == "VariableLibrary":
                fuera[str(it.get("displayName"))] = (ws, it["id"])
    return fuera


def variables(ambiente, nombres=None):
    """{variable: valor} de las Variable Library del ambiente.

    `nombres` acota a las bibliotecas que el flujo declara usar, que es lo
    normal: cargar todas obliga a un getDefinition por biblioteca y no aporta.
    """
    disponibles = bibliotecas(ambiente)
    pedidas = sorted(set(nombres) & set(disponibles)) if nombres else sorted(disponibles)
    fuera = {}
    for nombre in pedidas:
        clave = (ambiente, nombre)
        if clave not in _VARS:
            ws, ident = disponibles[nombre]
            # v9 · con el value set ACTIVO de la biblioteca, no solo el valor por defecto
            _VARS[clave] = dict(valores_de_biblioteca(ws, ident))
        fuera.update(_VARS[clave])
    return fuera


def bibliotecas_del_pipeline(defi):
    """Nombres de las Variable Library que el pipeline declara usar."""
    decl = (defi.get("properties") or defi).get("libraryVariables") or {}
    return {str(v.get("libraryName")) for v in decl.values()
            if isinstance(v, dict) and v.get("libraryName")}


# ────────────────────────────────────── expresiones de Data Factory
def _piezas(cuerpo, vars_):
    out = []
    for parte in re.split(r",(?![^()]*\))", cuerpo):
        p = parte.strip()
        if p.startswith("'") and p.endswith("'"):
            out.append(p[1:-1])
        else:
            v = re.search(r"libraryVariables\.([A-Za-z_0-9]+)", p)
            out.append(str(vars_.get(v.group(1), "{" + v.group(1) + "}")) if v else "")
    return "".join(out)


def _expr(e, vars_):
    m = re.match(r"\s*concat\((.*)\)\s*$", e, re.S)
    return _piezas(m.group(1), vars_) if m else _piezas(e, vars_)


def lit(valor, vars_):
    """Resuelve un campo de Data Factory a su valor real.

    Un mismo campo llega en tres formas y las tres se dan en los pipelines de
    Comfandi: literal, texto con `@{expr}` embebido, y el objeto
    `{"value": "@concat(...)", "type": "Expression"}`.
    """
    if isinstance(valor, dict):
        valor = valor.get("value", "")
    s = str(valor or "")
    if not s:
        return ""
    # v9 · antes se partia la expresion por comas y se pegaban los trozos: toda
    # funcion anidada (split(...)[0], replace, substring(..., add(...))) salia mal.
    if "@{" in s:
        partes, ultimo = [], 0
        for ini, fin, expr in _interpolaciones(s):
            partes.append(s[ultimo:ini])
            partes.append(_evaluar_texto(expr, vars_))
            ultimo = fin
        partes.append(s[ultimo:])
        return "".join(partes)
    if s.startswith("@@"):
        return s[1:]
    return _evaluar_texto(s[1:], vars_) if s.startswith("@") else s


def texto_crudo(valor):
    """El campo tal como lo declara el descriptor, todavia sin resolver."""
    if isinstance(valor, dict):
        return str(valor.get("value", ""))
    return str(valor or "")


# ────────────────────────────── v9 · evaluador de expresiones de Data Factory
# Sustituye al troceo por comas de `_piezas`: aquel partia `concat(a, b)` por las
# comas de primer nivel y pegaba lo que reconocia, asi que cualquier funcion
# anidada —split(...)[0], replace(...), substring(..., add(...))— salia mal o
# repetida. Esto es un analizador de verdad, pequeño, que evalua lo que se puede
# saber ANTES de ejecutar y deja vacio lo que solo existe durante la corrida.


class _Hueco(str):
    """Texto vacio que marca un valor de tiempo de corrida (item(), activity()…).
    Dentro de un concat se comporta como "" —lo mismo que hacia v8—, y el que
    quiera saber que hubo algo dinamico lo reconoce por su tipo."""


_TOK = re.compile(r"""
    \s*(?:
      (?P<cad>'(?:[^']|'')*')          |
      (?P<num>-?\d+(?:\.\d+)?)         |
      (?P<id>[A-Za-z_@][A-Za-z_0-9]*)  |
      (?P<sim>[(),.\[\]?])
    )""", re.X)


def _tokens(texto):
    pos, fuera = 0, []
    texto = texto.strip()
    while pos < len(texto):
        m = _TOK.match(texto, pos)
        if not m or m.end() == pos:
            raise ValueError(f"expresion no reconocida cerca de: {texto[pos:pos + 30]!r}")
        pos = m.end()
        tipo = m.lastgroup
        v = m.group(tipo)
        if tipo == "cad":
            v = v[1:-1].replace("''", "'")
        fuera.append((tipo, v))
    return fuera


def _net_a_strftime(fmt):
    """Formato .NET (yyyy-MM-dd HH:mm:ss) a strftime; cubre lo que usan los flujos."""
    reemplazos = (("yyyy", "%Y"), ("yy", "%y"), ("MM", "%m"), ("dd", "%d"),
                  ("HH", "%H"), ("hh", "%I"), ("mm", "%M"), ("ss", "%S"), ("fff", "%f"))
    out, i = "", 0
    while i < len(fmt):
        for a, b in reemplazos:
            if fmt.startswith(a, i):
                out += b
                i += len(a)
                break
        else:
            out += fmt[i]
            i += 1
    return out


def _fecha(v):
    if isinstance(v, datetime):
        return v
    s = str(v).replace("Z", "+00:00")
    try:
        return datetime.fromisoformat(s)
    except ValueError:
        return datetime.now(timezone.utc)


def _texto(v):
    if isinstance(v, bool):
        return "true" if v else "false"
    if isinstance(v, datetime):
        return v.strftime("%Y-%m-%dT%H:%M:%S.%f0Z")
    return "" if v is None else str(v)


def _num(v):
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return v
    try:
        f = float(_texto(v) or 0)
    except ValueError:
        return 0
    return int(f) if f.is_integer() else f


_FUNCIONES = {
    "concat": lambda *a: "".join(_texto(x) for x in a),
    "split": lambda s, sep: _texto(s).split(_texto(sep)),
    "replace": lambda s, a, b: _texto(s).replace(_texto(a), _texto(b)),
    "tolower": lambda s: _texto(s).lower(),
    "toupper": lambda s: _texto(s).upper(),
    "trim": lambda s: _texto(s).strip(),
    "substring": lambda s, i, n=None: (_texto(s)[int(_num(i)):] if n is None
                                       else _texto(s)[int(_num(i)):int(_num(i)) + int(_num(n))]),
    "indexof": lambda s, x: _texto(s).find(_texto(x)),
    "lastindexof": lambda s, x: _texto(s).rfind(_texto(x)),
    "length": lambda s: len(s if isinstance(s, (list, dict)) else _texto(s)),
    "startswith": lambda s, x: _texto(s).startswith(_texto(x)),
    "endswith": lambda s, x: _texto(s).endswith(_texto(x)),
    "contains": lambda s, x: _texto(x) in (s if isinstance(s, (list, dict)) else _texto(s)),
    "equals": lambda a, b: (_texto(a).lower() == _texto(b).lower()
                            if isinstance(a, str) or isinstance(b, str) else a == b),
    "not": lambda a: not a,
    "and": lambda *a: all(a),
    "or": lambda *a: any(a),
    "empty": lambda s: not s,
    "coalesce": lambda *a: next((x for x in a if x not in (None, "")), None),
    "first": lambda s: (s[0] if s else None),
    "last": lambda s: (s[-1] if s else None),
    "string": _texto,
    "int": lambda s: int(_num(s)),
    "float": lambda s: float(_num(s)),
    "bool": lambda s: _texto(s).lower() in ("true", "1"),
    "json": lambda s: s,
    "join": lambda s, sep: _texto(sep).join(_texto(x) for x in s),
    "add": lambda a, b: _num(a) + _num(b),
    "sub": lambda a, b: _num(a) - _num(b),
    "mul": lambda a, b: _num(a) * _num(b),
    "div": lambda a, b: (_num(a) // _num(b) if isinstance(_num(a), int) and isinstance(_num(b), int)
                         else _num(a) / _num(b)),
    "mod": lambda a, b: _num(a) % _num(b),
    "max": lambda *a: max(_num(x) for x in (a[0] if len(a) == 1 and isinstance(a[0], list) else a)),
    "min": lambda *a: min(_num(x) for x in (a[0] if len(a) == 1 and isinstance(a[0], list) else a)),
    "greater": lambda a, b: _num(a) > _num(b),
    "greaterorequals": lambda a, b: _num(a) >= _num(b),
    "less": lambda a, b: _num(a) < _num(b),
    "lessorequals": lambda a, b: _num(a) <= _num(b),
    "createarray": lambda *a: list(a),
    "array": lambda a: [a],
    "take": lambda s, n: s[:int(_num(n))],
    "skip": lambda s, n: s[int(_num(n)):],
    "union": lambda *a: [x for lst in a for x in lst],
    "utcnow": lambda fmt=None: (datetime.now(timezone.utc).strftime(_net_a_strftime(fmt))
                                if fmt else datetime.now(timezone.utc)),
    "formatdatetime": lambda d, fmt=None: (_fecha(d).strftime(_net_a_strftime(fmt)) if fmt
                                           else _texto(_fecha(d))),
    "dayofmonth": lambda d: _fecha(d).day,
    "dayofweek": lambda d: (_fecha(d).weekday() + 1) % 7,
    "dayofyear": lambda d: _fecha(d).timetuple().tm_yday,
}


def _tiempo(d, n, unidad, fmt=None):
    u = str(unidad).lower().rstrip("s")
    dias = {"second": 1 / 86400, "minute": 1 / 1440, "hour": 1 / 24, "day": 1, "week": 7}.get(u)
    base = _fecha(d)
    if dias is not None:
        base = base + timedelta(days=dias * int(_num(n)))
    elif u in ("month", "year"):
        import calendar
        meses = int(_num(n)) * (12 if u == "year" else 1)
        m = base.month - 1 + meses
        y, m = base.year + m // 12, m % 12 + 1
        base = base.replace(year=y, month=m, day=min(base.day, calendar.monthrange(y, m)[1]))
    return _FUNCIONES["formatdatetime"](base, fmt)


_FUNCIONES.update({
    "addtotime": _tiempo,
    "subtractfromtime": lambda d, n, u, fmt=None: _tiempo(d, -int(_num(n)), u, fmt),
    "getpasttime": lambda n, u, fmt=None: _tiempo(datetime.now(timezone.utc), -int(_num(n)), u, fmt),
    "getfuturetime": lambda n, u, fmt=None: _tiempo(datetime.now(timezone.utc), int(_num(n)), u, fmt),
    "adddays": lambda d, n, fmt=None: _tiempo(d, n, "day", fmt),
    "addhours": lambda d, n, fmt=None: _tiempo(d, n, "hour", fmt),
    "addminutes": lambda d, n, fmt=None: _tiempo(d, n, "minute", fmt),
    "addseconds": lambda d, n, fmt=None: _tiempo(d, n, "second", fmt),
    "converttimezone": lambda d, _a, _b=None, fmt=None: _FUNCIONES["formatdatetime"](d, fmt),
    "convertfromutc": lambda d, _z, fmt=None: _FUNCIONES["formatdatetime"](d, fmt),
    "startofday": lambda d, fmt=None: _FUNCIONES["formatdatetime"](
        _fecha(d).replace(hour=0, minute=0, second=0, microsecond=0), fmt),
    "startofmonth": lambda d, fmt=None: _FUNCIONES["formatdatetime"](
        _fecha(d).replace(day=1, hour=0, minute=0, second=0, microsecond=0), fmt),
})
# Lo que solo existe mientras corre el pipeline.
_DINAMICAS = {"item", "activity", "variables", "iterationindex", "trigger", "guid", "rand"}


class _Evaluador:
    def __init__(self, tokens, contexto):
        self.t, self.i, self.ctx = tokens, 0, contexto

    def _ver(self, k=0):
        j = self.i + k
        return self.t[j] if j < len(self.t) else (None, None)

    def _tomar(self, valor=None):
        tok = self._ver()
        if valor is not None and tok[1] != valor:
            raise ValueError(f"se esperaba {valor!r} y llego {tok[1]!r}")
        self.i += 1
        return tok

    def expresion(self):
        tipo, v = self._tomar()
        if tipo == "cad":
            val = v
        elif tipo == "num":
            val = float(v) if "." in v else int(v)
        elif tipo == "id":
            low = v.lower()
            if low in ("true", "false"):
                val = low == "true"
            elif low == "null":
                val = None
            elif self._ver()[1] == "(":
                self._tomar("(")
                args = []
                while self._ver()[1] != ")":
                    args.append(self.expresion())
                    if self._ver()[1] == ",":
                        self._tomar(",")
                self._tomar(")")
                val = self._llamar(low, args)
            else:
                raise ValueError(f"identificador suelto: {v}")
        else:
            raise ValueError(f"simbolo inesperado: {v}")
        return self._accesos(val)

    def _llamar(self, nombre, args):
        if nombre == "pipeline":
            return {"__pipeline__": True}
        if nombre == "if":
            return args[1] if args[0] else args[2]
        if nombre in _DINAMICAS:
            return _Hueco()
        f = _FUNCIONES.get(nombre)
        if f is None:
            raise ValueError(f"funcion no soportada: {nombre}")
        return f(*args)

    def _accesos(self, val):
        while self._ver()[1] in (".", "[", "?"):
            s = self._tomar()[1]
            if s == "?":
                continue
            if s == ".":
                prop = self._tomar()[1]
                val = self._propiedad(val, prop)
            else:
                idx = self.expresion()
                self._tomar("]")
                val = self._propiedad(val, idx)
        return val

    def _propiedad(self, val, prop):
        if isinstance(val, _Hueco) or val is None:
            return _Hueco()
        if isinstance(val, dict) and val.get("__pipeline__"):
            p = str(prop).lower()
            if p in ("libraryvariables", "parameters", "globalparameters"):
                return {"__rama__": p}
            return _Hueco()        # RunId, TriggerTime, Pipeline… solo existen al correr
        if isinstance(val, dict) and "__rama__" in val:
            return self.ctx(val["__rama__"], str(prop))
        if isinstance(val, list):
            i = int(_num(prop))
            return val[i] if -len(val) <= i < len(val) else None
        if isinstance(val, dict):
            return val.get(prop)
        return _Hueco()


def evaluar(expr, contexto):
    """Valor de una expresion (sin la @ inicial). `contexto(rama, nombre)` resuelve
    pipeline().libraryVariables.X / parameters.X / globalParameters.X."""
    ev = _Evaluador(_tokens(expr), contexto)
    val = ev.expresion()
    if ev.i != len(ev.t):
        raise ValueError(f"sobra texto tras la expresion: {ev.t[ev.i:][:3]}")
    return val


# ────────────────────────────── v9 · Variable Library por flujo, no plana
# Hasta v8, `variables()` juntaba TODAS las bibliotecas en un diccionario plano
# por nombre de variable, y `lit()` buscaba ahi el ALIAS del pipeline:
#   pipeline().libraryVariables.vl_core_ssf_afiliados_csv_filename
# cuando la variable se llama `ssf_afiliados_csv_filename` dentro de `vl_core`.
# Resultado: toda ruta de un Copy quedaba como `{vl_core_...}` y el validador
# informaba «no existe en OneLake» sobre archivos que si existian. Ademas, el
# mismo nombre vive en varias bibliotecas con valores distintos
# (sftp_local_temp_dir = Files/tmp en vl_core, Files/bronze/... en vl_mercadeo):
# en un diccionario plano gana la ultima que se cargo.
#
# Ahora cada flujo resuelve SU bloque `libraryVariables` (alias -> biblioteca +
# variable), contra la biblioteca de SU workspace —orchestration o ingestion—,
# con el value set ACTIVO de esa biblioteca aplicado.

_VL_RESUELTAS = {}
_VL_ORDEN_WS = ("orchestration", "ingestion")


def definicion_partes(ws_nombre, item_id):
    """Todas las partes de la definicion del item, {ruta: contenido decodificado}."""
    wid = workspaces().get(ws_nombre)
    codigo, cuerpo = api(f"workspaces/{wid}/items/{item_id}/getDefinition", "POST", {})
    if codigo not in (200, 202) or not isinstance(cuerpo, dict):
        return {}
    fuera = {}
    for parte in (cuerpo.get("definition") or {}).get("parts", []):
        try:
            fuera[str(parte.get("path", ""))] = base64.b64decode(parte.get("payload", "")).decode("utf-8")
        except Exception:
            continue
    return fuera


def valores_de_biblioteca(ws_nombre, item_id):
    """{variable: valor} de UNA biblioteca, con el value set activo aplicado.

    El valor por defecto de `variables.json` solo vale si el value set activo es
    el por defecto; si el workspace activo otro (qa, prod…), sus overrides de
    `valueSets/<nombre>.json` mandan.
    """
    clave = (ws_nombre, item_id)
    if clave in _VL_RESUELTAS:
        return _VL_RESUELTAS[clave]
    partes = definicion_partes(ws_nombre, item_id)
    try:
        declaradas = json.loads(partes.get("variables.json") or "{}").get("variables") or []
    except ValueError:
        declaradas = []
    valores = {str(v["name"]): ("" if v.get("value") is None else str(v.get("value")))
               for v in declaradas if isinstance(v, dict) and v.get("name")}
    activo = ""
    wid = workspaces().get(ws_nombre)
    cod, props = api(f"workspaces/{wid}/variableLibraries/{item_id}")
    if cod == 200 and isinstance(props, dict):
        activo = str((props.get("properties") or {}).get("activeValueSetName") or "")
    aplicado = ""
    if activo and activo.lower() not in ("default value set", "default"):
        for ruta, txt in partes.items():
            if not ruta.startswith("valueSets/"):
                continue
            try:
                vs = json.loads(txt)
            except ValueError:
                continue
            nombre_vs = str(vs.get("name") or ruta.rsplit("/", 1)[-1][:-5])
            if nombre_vs.lower() == activo.lower():
                for ov in vs.get("variableOverrides") or ():
                    if isinstance(ov, dict) and ov.get("name") in valores:
                        valores[str(ov["name"])] = "" if ov.get("value") is None else str(ov["value"])
                aplicado = nombre_vs
    _VL_RESUELTAS[clave] = valores
    _VL_RESUELTAS[("__activo__",) + clave] = {"activo": activo or "Default value set",
                                              "aplicado": aplicado or "Default value set"}
    return valores


def bibliotecas_por_workspace(ambiente):
    """{nombre de biblioteca: [(workspace, id), …]}, el mismo nombre puede estar en varios."""
    fuera = {}
    for ws in workspaces_de(ambiente):
        for it in items(ws):
            if it.get("type") == "VariableLibrary":
                fuera.setdefault(str(it.get("displayName")), []).append((ws, it["id"]))
    return fuera


def _candidatas(nombre_lib, ambiente, ws_flujo):
    """Donde buscar una biblioteca: primero el workspace del flujo (Fabric resuelve
    `libraryName` contra el propio workspace), luego orchestration e ingestion."""
    lugares = bibliotecas_por_workspace(ambiente).get(nombre_lib, [])

    def prioridad(par):
        ws = par[0]
        if ws == ws_flujo:
            return 0
        for i, marca in enumerate(_VL_ORDEN_WS, 1):
            if f"-{marca}-" in ws:
                return i
        return 9
    return sorted(lugares, key=prioridad)


class VariablesDelFlujo(dict):
    """Lo que ve `lit()`: alias del pipeline -> valor, y ademas nombre real -> valor.

    Es un dict para que todo lo que ya recibia `vars_` siga funcionando; lo nuevo va
    en atributos: de donde salio cada alias, cuales no se encontraron y los
    parametros del pipeline con su valor por defecto.
    """

    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.origen, self.faltantes, self.parametros = {}, {}, {}
        self.ambiente = None


def variables_del_pipeline(defi, ambiente, ws_flujo=""):
    """Las variables que el pipeline puede referenciar, ya resueltas para el ambiente."""
    props = defi.get("properties") or defi
    decl = props.get("libraryVariables") or {}
    fuera = VariablesDelFlujo()
    cargadas = {}

    def biblioteca(nombre):
        if nombre not in cargadas:
            cargadas[nombre] = None
            for ws, ident in _candidatas(nombre, ambiente, ws_flujo):
                vals = valores_de_biblioteca(ws, ident)
                if vals:
                    cargadas[nombre] = (ws, vals)
                    break
        return cargadas[nombre]

    # 1 · lo que el pipeline declara: alias -> (biblioteca, variable)
    for alias, d in decl.items():
        if not isinstance(d, dict):
            continue
        lib, var = str(d.get("libraryName") or ""), str(d.get("variableName") or alias)
        hallada = biblioteca(lib) if lib else None
        if hallada and var in hallada[1]:
            fuera[alias] = hallada[1][var]
            fuera.origen[alias] = f"{lib}.{var} ({hallada[0]})"
        else:
            fuera.faltantes[alias] = (f"{lib}.{var}: " +
                                      ("la biblioteca no existe en el ambiente" if not hallada
                                       else "la variable no existe en la biblioteca"))

    # 2 · alias usados pero NO declarados (en analitica de QA el bloque viene vacio):
    #     el alias es <biblioteca>_<variable>, se prueba el prefijo mas largo que sea
    #     el nombre de una biblioteca del ambiente.
    usados = set(re.findall(r"libraryVariables\.([A-Za-z_0-9]+)", json.dumps(props)))
    libs_ambiente = sorted(bibliotecas_por_workspace(ambiente), key=len, reverse=True)
    for alias in sorted(usados - set(fuera)):
        for lib in libs_ambiente:
            pref = lib + "_"
            if alias.startswith(pref):
                hallada = biblioteca(lib)
                var = alias[len(pref):]
                if hallada and var in hallada[1]:
                    fuera[alias] = hallada[1][var]
                    fuera.origen[alias] = f"{lib}.{var} ({hallada[0]}, alias no declarado)"
                    fuera.faltantes.pop(alias, None)
                    break
        else:
            fuera.faltantes.setdefault(alias, "alias sin declarar y sin biblioteca que lo explique")

    # 3 · nombres reales de las bibliotecas usadas, sin pisar los alias: los usa
    #     `resolver_alias` y cualquier referencia directa por nombre.
    for lib in sorted({str(d.get("libraryName")) for d in decl.values() if isinstance(d, dict)}
                      | {o.split(".", 1)[0] for o in fuera.origen.values()}):
        hallada = biblioteca(lib)
        if hallada:
            for k, v in hallada[1].items():
                fuera.setdefault(k, v)

    for nombre, p in (props.get("parameters") or {}).items():
        if isinstance(p, dict) and "defaultValue" in p:
            fuera.parametros[nombre] = p.get("defaultValue")
    fuera.ambiente = ambiente
    return fuera


def _contexto_de(vars_):
    parametros = getattr(vars_, "parametros", {}) or {}

    def contexto(rama, nombre):
        if rama == "libraryvariables":
            return str(vars_[nombre]) if nombre in vars_ else "{" + nombre + "}"
        if rama == "parameters":
            v = parametros.get(nombre)
            return v if v is not None else "{" + nombre + "}"
        return "{" + nombre + "}"
    return contexto


def _interpolaciones(s):
    """[(inicio, fin, expresion)] de cada @{…} con llaves balanceadas y comillas respetadas."""
    fuera, i = [], 0
    while True:
        i = s.find("@{", i)
        if i < 0:
            return fuera
        if i > 0 and s[i - 1] == "@":          # @@{ es un literal
            i += 2
            continue
        prof, j, en_cad = 1, i + 2, False
        while j < len(s) and prof:
            c = s[j]
            if c == "'":
                en_cad = not en_cad
            elif not en_cad and c == "{":
                prof += 1
            elif not en_cad and c == "}":
                prof -= 1
            j += 1
        fuera.append((i, j, s[i + 2:j - 1]))
        i = j


_DEF_HIJOS = {}      # (workspace, pipeline invocado) -> su definicion
_VARS_HIJOS = {}
_CARPETAS_LH = {}


def vars_de_actividad(info, vars_):
    """Las variables con que se resuelve UNA actividad: las del pipeline al que
    pertenece. Un flujo invocado declara su propio bloque libraryVariables, y
    resolverlo con el del padre dejaba sin valor todos sus alias —o, peor, los
    resolvia quitando prefijos contra otra variable de nombre parecido."""
    org = info.get("origen_pipeline") if isinstance(info, dict) else None
    ambiente = getattr(vars_, "ambiente", None)
    if not org or not ambiente:
        return vars_
    if org not in _VARS_HIJOS:
        d = _DEF_HIJOS.get(org)
        _VARS_HIJOS[org] = variables_del_pipeline(d, ambiente, org[0]) if d else vars_
    return _VARS_HIJOS[org]


def _carpetas_lakehouse(ws_id, lh_id, rel):
    """{nombre en minusculas: nombre real} de las carpetas directas bajo `rel`.
    None si no se pudo listar (entonces se ancla como en v8, sin adivinar)."""
    clave = (ws_id, lh_id, rel)
    if clave not in _CARPETAS_LH:
        res = None
        leer, dfs = globals().get("_onelake"), globals().get("DFS")
        if leer and dfs and ws_id and lh_id:
            try:
                cod, cuerpo = leer(f"{dfs}/{ws_id}?recursive=false&resource=filesystem"
                                   f"&directory={lh_id}/{rel}")
                if cod == 200 and isinstance(cuerpo, dict):
                    res = {}
                    for p in cuerpo.get("paths", []):
                        if str(p.get("isDirectory", "")).lower() == "true":
                            n = str(p.get("name", "")).rsplit("/", 1)[-1]
                            res[n.lower()] = n
                elif cod == 404:
                    res = {}
            except Exception:
                res = None
        _CARPETAS_LH[clave] = res
    return _CARPETAS_LH[clave]


EXPRESIONES_NO_EVALUADAS = {}   # expresion -> motivo: lo que cayo en el metodo de v8


def _evaluar_texto(expr, vars_):
    """Una expresion a texto. Si el evaluador no la entiende, se vuelve a lo de v8
    y queda anotada en EXPRESIONES_NO_EVALUADAS, para que no pase inadvertida."""
    try:
        return _texto(evaluar(expr, _contexto_de(vars_)))
    except Exception as e:
        EXPRESIONES_NO_EVALUADAS[str(expr)[:300]] = f"{type(e).__name__}: {e}"
        return _expr(expr, vars_)


# v5.2 · una salida que se escribe en partes no es un archivo, es un conjunto.
# El descriptor la declara con `item()` dentro del nombre —VBAP_@{item()}.parquet—
# y al resolver la expresion ese `item()` se iba en blanco, dejando un literal
# "VBAP_.parquet" que no existe en ninguna corrida: el flujo escribe VBAP_1,
# VBAP_2 … Medir ese literal reprobaba flujos que habian escrito bien.
ITERADOR = re.compile(r"item\s*\(\s*\)", re.I)
POR_ITERACION = re.compile(
    r"item\s*\(\s*\)|activity\s*\(|utcnow\s*\(|formatDateTime\s*\(|addDays\s*\(", re.I)


def por_iteracion(campo):
    """True si el valor del campo cambia en cada vuelta del bucle o con la fecha."""
    return bool(POR_ITERACION.search(texto_crudo(campo)))


def patron_de(campo, vars_):
    """El nombre resuelto con `*` donde va el valor de la iteracion: VBAP_*.parquet."""
    crudo = texto_crudo(campo)
    if not crudo:
        return ""
    return lit(ITERADOR.sub("'*'", crudo), vars_)


# Lo que cuelga de staging/ es un paso intermedio: el propio flujo lo consume y
# lo reescribe particionado. Exigir que siga ahi al terminar la corrida es pedir
# que no haya hecho su trabajo.
INTERMEDIA = re.compile(r"(^|/)(staging|tmp|temp|scratch)(/|$)", re.I)


def es_intermedia(carpeta):
    return bool(INTERMEDIA.search(str(carpeta or "")))


def variables_referidas(*campos):
    """Nombres de Variable Library que aparecen en los campos dados."""
    nombres = []
    for c in campos:
        nombres += re.findall(r"libraryVariables\.([A-Za-z_0-9]+)", texto_crudo(c))
    return sorted(set(nombres))


_RASTROS = (
    (r"\{([A-Za-z_][A-Za-z_0-9]*)\}", "variable sin valor en la Variable Library"),
    # Una variable puede estar "resuelta" y aun asi no valer nada: en QA varias
    # traen la palabra placeholder como valor. Se ve como una ruta plausible, se
    # construye un abfss con ella y la lectura falla por "no existe", cuando el
    # problema real es que nunca se configuro.
    (r"(?i)(^|[/_.-])placehold(er)?([/_.-]|$)", "valor de relleno sin configurar"),
    (r"pipeline\(\)|libraryVariables", "expresion de Data Factory sin evaluar"),
    (r"@\{|@concat|@pipeline|@activity", "expresion sin evaluar"),
    (r"['\"]type['\"]:\s*['\"]Expression['\"]", "campo objeto sin desenvolver"),
)


def rastro_sin_resolver(ruta):
    """Que quedo sin resolver en una ruta, si quedo algo.

    Existe porque una ruta a medio resolver es peor que una ruta ausente: se
    ve como un texto plausible, se construye un abfss con ella y la lectura
    falla por "no existe", cuando el problema real es que nunca se resolvio.
    """
    return sorted({etiqueta for patron, etiqueta in _RASTROS
                   if re.search(patron, str(ruta))})


def abfss(ws_id, artifact_id, raiz, ruta):
    return (f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/"
            f"{artifact_id}/{raiz}/{ruta.lstrip('/')}")


_SJD_CACHE = {}
_LH_NOMBRES = {}


def lakehouse_del_spark_job(ws_id, sjd_id):
    """(workspace_id, lakehouse_id) del lakehouse por defecto del Spark Job.

    Las rutas que un flujo de analitica recibe por argumento son relativas a
    ese lakehouse: `Files/bronze/...` o `esquema.tabla`. Sin resolverlas no se
    puede leer el dato ni afirmar a que ambiente apunta.
    """
    clave = (ws_id, sjd_id)
    if clave in _SJD_CACHE:
        return _SJD_CACHE[clave]
    codigo, cuerpo = api(f"workspaces/{ws_id}/items/{sjd_id}/getDefinition", "POST", {})
    lh = ""
    if codigo in (200, 202):
        for parte in (cuerpo.get("definition") or {}).get("parts", []):
            if not str(parte.get("path", "")).endswith(".json"):
                continue
            try:
                d = json.loads(base64.b64decode(parte.get("payload", "")).decode())
            except Exception:
                continue
            if isinstance(d, dict) and d.get("defaultLakehouseArtifactId"):
                lh = str(d["defaultLakehouseArtifactId"])
                break
    # El lakehouse por defecto puede vivir en OTRO workspace: en Comfandi los
    # Spark Job estan en contactabilidad-qa y escriben en lh_transversal, que
    # es de datalake-qa. Usar el workspace del job daria un abfss que no existe.
    _SJD_CACHE[clave] = (workspace_del_lakehouse(lh) or ws_id, lh)
    return _SJD_CACHE[clave]


_LH_WS = {}


def workspace_del_lakehouse(lh_id):
    """Workspace que aloja ese lakehouse. Vacio si no se encuentra."""
    if not lh_id:
        return ""
    if lh_id in _LH_WS:
        return _LH_WS[lh_id]
    for nombre, wid in workspaces().items():
        codigo, cuerpo = api(f"workspaces/{wid}/items?type=Lakehouse")
        for i in (cuerpo.get("value", []) if codigo == 200 else []):
            _LH_WS.setdefault(str(i.get("id")), wid)
            _LH_NOMBRES.setdefault(str(i.get("id")), str(i.get("displayName")))
        if lh_id in _LH_WS:
            break
    return _LH_WS.get(lh_id, "")


def anclar_en_lakehouse(valor, ws_id, lh_id):
    """abfss de una ruta relativa o de un `esquema.tabla`. Vacio si no aplica."""
    v = str(valor or "").strip()
    if not v or not lh_id or v.startswith("abfss://"):
        return v if v.startswith("abfss://") else ""
    if rastro_sin_resolver(v):
        return ""                      # no se ancla lo que no esta configurado
    if v.startswith("Files/") or v.startswith("Tables/"):
        rel = v
    elif re.match(r"^[A-Za-z_][\w]*\.[A-Za-z_][\w]*$", v):
        # `silver.mi_tabla` es una tabla Delta del lakehouse, no una ruta
        esquema, _, tabla = v.partition(".")
        esquemas = _carpetas_lakehouse(ws_id, lh_id, "Tables")          # v9
        if esquemas is not None:
            if esquema.lower() not in esquemas:
                return ""      # esquema que no existe en el lakehouse: tabla externa
            esquema = esquemas[esquema.lower()]
            tablas = _carpetas_lakehouse(ws_id, lh_id, f"Tables/{esquema}")
            if tablas and tabla.lower() in tablas:
                tabla = tablas[tabla.lower()]
        rel = f"Tables/{esquema}/{tabla}"
    else:
        return ""
    return (f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}/{rel}")

def _nombre_lakehouse(ws_id, lh_id):
    if not lh_id:
        return ""
    if lh_id not in _LH_NOMBRES:
        workspace_del_lakehouse(lh_id)      # puebla nombres y workspaces
    return _LH_NOMBRES.get(lh_id, "")


# ─────────────────────────────────────────────── actividades anidadas
# Un flujo no es una lista plana. Un ForEach, un If, un Switch o un Until
# LLEVAN DENTRO las actividades que hacen el trabajo, y un InvokePipeline
# delega en otro flujo entero. Recorrer solo el primer nivel — que es lo que se
# hacia — deja fuera justo esas: un orquestador de dieciseis sub-flujos se
# validaba como si no tuviera ninguna entrada ni ninguna salida, en silencio.

PROF_MAX_ANIDADAS = 6          # cuantos niveles se bajan

_HIJOS = {
    "foreach":     lambda tp: tp.get("activities") or [],
    "until":       lambda tp: tp.get("activities") or [],
    "ifcondition": lambda tp: ((tp.get("ifTrueActivities") or [])
                               + (tp.get("ifFalseActivities") or [])),
    "switch":      lambda tp: ((tp.get("defaultActivities") or [])
                               + [x for c in (tp.get("cases") or [])
                                  for x in (c.get("activities") or [])]),
}


def _ws_nombre_por_id(wid):
    """El nombre del workspace a partir de su id.

    `definicion()` pide el NOMBRE y un InvokePipeline solo trae el id, asi que
    hay que invertir el mapa para poder bajar al flujo invocado.
    """
    if not wid:
        return None
    for nombre, ident in (workspaces() or {}).items():
        if str(ident) == str(wid):
            return nombre
    return None


def _invocado(a):
    """(workspace, pipeline) del flujo que invoca esta actividad, si invoca uno."""
    if str(a.get("type") or "").lower() not in ("invokepipeline", "executepipeline"):
        return None, None
    tp = a.get("typeProperties") or {}
    pid = tp.get("pipelineId") or (tp.get("pipeline") or {}).get("referenceName")
    return tp.get("workspaceId"), pid


def _rama_de(a, hijo):
    """En que rama del contenedor vive el hijo.

    Importa al leer el reporte: una salida que solo se escribe por la rama falsa
    de un If no es lo mismo que una incondicional.
    """
    tp = a.get("typeProperties") or {}
    n = hijo.get("name")
    for x in (tp.get("ifTrueActivities") or []):
        if x.get("name") == n:
            return "si"
    for x in (tp.get("ifFalseActivities") or []):
        if x.get("name") == n:
            return "no"
    for c in (tp.get("cases") or []):
        for x in (c.get("activities") or []):
            if x.get("name") == n:
                return f"caso {c.get('value')}"
    return None


def iteraciones_declaradas(a):
    """Cuantas vueltas declara un ForEach, si se puede saber sin ejecutarlo.

    Si `items` es una lista literal se cuenta. Si es una expresion —lo normal—
    no se puede saber de antemano: el numero real sale de las actividades de la
    corrida, que si las lista una por una.
    """
    it = (a.get("typeProperties") or {}).get("items")
    if isinstance(it, list):
        return len(it)
    if isinstance(it, dict):
        v = it.get("value")
        if isinstance(v, list):
            return len(v)
        if isinstance(v, str):
            try:
                x = json.loads(v)
                if isinstance(x, list):
                    return len(x)
            except Exception:
                pass
    return None


def actividades_planas(acts, camino=(), prof=0, visitados=None, origen=None):
    """Todas las actividades: contenedores y flujos invocados incluidos.

    Devuelve dicts con la actividad y DONDE vive, porque en un flujo con ForEach
    el nombre solo no distingue nada: hay que poder decir «el Spark job que
    cuelga del ForEach que recorre las ciudades».

    `visitados` corta los ciclos: un flujo que se invoca a si mismo, directa o
    indirectamente, colgaria el recorrido.
    """
    visitados = set() if visitados is None else visitados
    fuera = []
    for a in acts or []:
        tipo = str(a.get("type") or "")
        tp = a.get("typeProperties") or {}
        hijos_de = _HIJOS.get(tipo.lower())
        ws_inv, pid_inv = _invocado(a)
        reg = {
            "actividad": a,
            "nombre": a.get("name"),
            "tipo": tipo,
            "camino": list(camino),
            "dentro_de": " / ".join(camino) or None,
            "profundidad": prof,
            "es_contenedor": bool(hijos_de) or bool(pid_inv),
            "invoca_pipeline": pid_inv,
            "iteraciones": (iteraciones_declaradas(a)
                            if tipo.lower() == "foreach" else None),
            "rama": None,
            "truncado": None,
            # v9 · None = el pipeline principal; (ws, id) = un flujo invocado. Sus
            # expresiones se resuelven con SU bloque libraryVariables, no el del padre.
            "origen_pipeline": origen,
        }
        fuera.append(reg)
        if prof >= PROF_MAX_ANIDADAS:
            if hijos_de or pid_inv:
                reg["truncado"] = f"no se baja mas de {PROF_MAX_ANIDADAS} niveles"
            continue
        if hijos_de:
            for h in hijos_de(tp):
                sub = actividades_planas([h], tuple(camino) + (a.get("name"),),
                                         prof + 1, visitados, origen)
                r = _rama_de(a, h)
                if r and sub:
                    sub[0]["rama"] = r
                fuera += sub
        elif pid_inv:
            if pid_inv in visitados:
                reg["truncado"] = "ciclo: este flujo ya se recorrio"
                continue
            visitados.add(pid_inv)
            ws_n = _ws_nombre_por_id(ws_inv)
            d_hijo = definicion(ws_n, pid_inv, "pipeline-content.json") if ws_n else None
            if d_hijo is None:
                reg["truncado"] = f"no se pudo leer el flujo invocado {pid_inv}"
                continue
            reg["invocado_resuelto"] = True
            _DEF_HIJOS[(ws_n, pid_inv)] = d_hijo                                # v9
            fuera += actividades_planas(
                (d_hijo.get("properties") or d_hijo).get("activities", []),
                tuple(camino) + (a.get("name"),), prof + 1, visitados, (ws_n, pid_inv))
    return fuera


def resumen_del_arbol(planas):
    """Una linea por actividad, con la sangria del anidamiento."""
    fuera = []
    for x in planas:
        extra = ""
        if x["tipo"].lower() == "foreach":
            n = x.get("iteraciones")
            extra = (f"  ({n} iteraciones declaradas)" if n is not None
                     else "  (iteraciones por expresion: se ven al ejecutar)")
        if x.get("invoca_pipeline"):
            extra += ("  -> flujo invocado, se baja" if x.get("invocado_resuelto")
                      else "  -> flujo invocado")
        if x.get("rama"):
            extra += f"  [rama {x['rama']}]"
        if x.get("truncado"):
            extra += f"  AVISO {x['truncado']}"
        fuera.append("   " * x["profundidad"] + f"· {x['nombre']}  [{x['tipo']}]{extra}")
    return fuera


# ─────────────────────────────────────────────── el tiempo, siempre
# Antes la duracion se perdia en cuanto la corrida no tenia endTimeUtc: se
# imprimia un guion. Y esperar() caia del while sin return, asi que al llegar al
# tope devolvia None y con ello desaparecia la corrida entera del reporte.

def _hms(seg):
    """3800 -> '1 h 03 m 20 s'. Siempre devuelve texto, nunca None."""
    if seg is None:
        return "sin medir"
    try:
        s = int(round(float(seg)))
    except Exception:
        return "sin medir"
    signo = "-" if s < 0 else ""
    s = abs(s)
    h, r = divmod(s, 3600)
    m, x = divmod(r, 60)
    if h:
        return f"{signo}{h} h {m:02d} m {x:02d} s"
    if m:
        return f"{signo}{m} m {x:02d} s"
    return f"{signo}{x} s"


def duracion_detallada(j):
    """Segundos de la corrida, salga de donde salga, diciendo de cual.

    Se prueban las fuentes en orden de fiabilidad, porque «3800 s segun la API»
    y «3800 s segun el reloj del validador» no son lo mismo cuando hay que
    discutir el numero:

      1. startTimeUtc y endTimeUtc      (la buena)
      2. startTimeUtc y AHORA           (parcial, va creciendo)
      3. el reloj del propio validador  (red de seguridad)
    """
    j = j or {}
    try:
        a = datetime.fromisoformat(_limpiar_ts(j.get("startTimeUtc")))
    except Exception:
        a = None
    try:
        b = datetime.fromisoformat(_limpiar_ts(j.get("endTimeUtc")))
    except Exception:
        b = None
    if a and b:
        return {"segundos": int((b - a).total_seconds()),
                "origen": "API (inicio y fin)", "parcial": False}
    if a:
        return {"segundos": int((datetime.now(timezone.utc) - a).total_seconds()),
                "origen": "API (inicio) hasta ahora", "parcial": True}
    if j.get("_segundos_medidos") is not None:
        return {"segundos": int(j["_segundos_medidos"]),
                "origen": "reloj del validador",
                "parcial": bool(j.get("_se_dejo_de_esperar"))}
    return {"segundos": None, "origen": "no hay corrida que medir", "parcial": False}


def duracion_segundos(j):
    """La duracion como numero. None solo si de verdad no hay nada que medir."""
    return duracion_detallada(j)["segundos"]


def texto_duracion(j):
    """La linea que se imprime. Sale SIEMPRE, incluso sin cifra."""
    d = duracion_detallada(j)
    if d["segundos"] is None:
        return f"duracion: sin medir ({d['origen']})"
    t = f"duracion: {_hms(d['segundos'])} ({d['segundos']} s) · {d['origen']}"
    if d["parcial"]:
        t += " · AUN EN CURSO, el numero sigue subiendo"
    if d["segundos"] > TOPE_SOSPECHA:
        t += (f" · AVISO pasa de {_hms(TOPE_SOSPECHA)}: sospechoso, "
              f"confirmar en el monitor de Fabric")
    return t


def io_pipeline(defi, vars_):
    """Entradas y salidas de cada Copy del pipeline, con su procedencia.

    De cada campo se guardan las dos caras: la plantilla que declara el
    descriptor y el valor real tras resolverla. Sin la plantilla no se puede
    saber de donde sale un nombre; sin el valor no se puede ir a buscarlo.
    """
    fuera = []
    for _info in actividades_planas((defi.get("properties") or defi)
                                    .get("activities", [])):
        a = _info["actividad"]
        tp = a.get("typeProperties") or {}
        vars_a = vars_de_actividad(_info, vars_)          # v9 · las de SU pipeline
        for rol in ("source", "sink"):
            ds = ((tp.get(rol) or {}).get("datasetSettings") or {})
            loc = (ds.get("typeProperties") or {}).get("location") or {}
            cruda_carpeta, cruda_archivo = loc.get("folderPath"), loc.get("fileName")
            carpeta = lit(cruda_carpeta, vars_a).rstrip("/")
            archivo = lit(cruda_archivo, vars_a)
            # v5.2 · con `item()` en el nombre, lo que existe es la carpeta entera.
            partes = por_iteracion(cruda_archivo)
            patron = patron_de(cruda_archivo, vars_a) if partes else ""
            if partes:
                archivo = ""
            ruta = "/".join(x for x in (carpeta, archivo) if x)
            if not ruta:
                continue
            ls = ((ds.get("linkedService") or {}).get("properties") or {}).get("typeProperties") or {}
            reg = {"actividad": a.get("name", ""),
                   "dentro_de": _info["dentro_de"],
                   "rama": _info["rama"],
                   "iteraciones": _info["iteraciones"],
                   "rol": "entrada" if rol == "source" else "salida",
                   "tipo": str(loc.get("type", "")),
                   "plantilla_carpeta": texto_crudo(cruda_carpeta),
                   "plantilla_archivo": texto_crudo(cruda_archivo),
                   "carpeta": carpeta, "archivo": archivo, "ruta": ruta,
                   "por_partes": partes, "patron_archivo": patron,
                   "intermedia": es_intermedia(carpeta),
                   "variables": {n: vars_a.get(n) for n in
                                 variables_referidas(cruda_carpeta, cruda_archivo)},
                   # v9 · de que biblioteca y variable sale cada alias
                   "origen_variables": {n: (getattr(vars_a, "origen", {}) or {}).get(n)
                                        or (getattr(vars_a, "faltantes", {}) or {}).get(n)
                                        for n in variables_referidas(cruda_carpeta, cruda_archivo)},
                   "sin_resolver": rastro_sin_resolver(ruta),
                   "lakehouse": (ds.get("linkedService") or {}).get("name", ""),
                   "ws_id": str(ls.get("workspaceId", "")),
                   "artifact_id": str(ls.get("artifactId", "")),
                   "abfss": ""}
            # v9 · una ruta a medio resolver no se ancla: medirla da «no existe»
            # sobre un nombre que nunca existio y se confunde con una falla del flujo
            if reg["ws_id"] and reg["artifact_id"] and not reg["sin_resolver"]:
                reg["abfss"] = abfss(reg["ws_id"], reg["artifact_id"],
                                     ls.get("rootFolder", "Files"), ruta)
            fuera.append(reg)
    return fuera


# ─────────────────────────────────────────────────────────── corridas
_RE_OFFSET_TS = re.compile(r"(?:[+-]\d{2}:?\d{2})$")


def _limpiar_ts(t):
    """Fabric devuelve 7 decimales (ticks .NET); %f admite 6.

    Y a veces devuelve el instante SIN marca de zona ("2026-09-17T18:57:41.2294089"), aunque
    el campo se llame startTimeUtc. Sin marca, fromisoformat da un datetime naive, y restarlo
    de datetime.now(timezone.utc) revienta con "can't subtract offset-naive and offset-aware
    datetimes" y tumba el flujo entero. El campo es UTC por contrato, asi que se le pone la
    marca que le falta en vez de confiar en que venga."""
    t = str(t or "").replace("Z", "+00:00")
    t = re.sub(r"(\.\d{6})\d+", r"\1", t)
    if t and "T" in t and not _RE_OFFSET_TS.search(t):
        t += "+00:00"
    return t


def historial(ws_nombre, item_id, tope=5):
    wid = workspaces().get(ws_nombre)
    js = paginar(f"workspaces/{wid}/items/{item_id}/jobs/instances")
    js.sort(key=lambda j: str(j.get("startTimeUtc") or ""), reverse=True)
    return js[:tope]


def duracion(j):
    """La duracion como texto corto. Devuelve "-" SOLO si no hay nada que medir.

    Antes bastaba con que faltara endTimeUtc —corrida viva, corrida abandonada
    al llegar al tope— para que saliera un guion. Justo los casos en los que uno
    quiere saber cuanto llevaba.
    """
    d = duracion_detallada(j)
    if d["segundos"] is None:
        return "-"
    return f"{d['segundos']:.0f}s" + ("+" if d["parcial"] else "")


def actividades(ws_nombre, corrida_id, margen_horas=24):
    """Detalle por actividad de una corrida de pipeline.

    Es la unica fuente que dice cuantas filas leyo el flujo del origen, y por
    eso es lo que permite validar la entrada de un flujo de ingesta cuyo origen
    (SFTP, JDBC) no se puede leer desde Spark.
    """
    wid = workspaces().get(ws_nombre)
    if not wid or not corrida_id:
        return []
    ahora = datetime.now(timezone.utc)
    codigo, resp = api(
        f"workspaces/{wid}/datapipelines/pipelineruns/{corrida_id}/queryactivityruns",
        "POST",
        {"lastUpdatedAfter": (ahora - timedelta(hours=margen_horas)).isoformat(),
         "lastUpdatedBefore": (ahora + timedelta(hours=1)).isoformat()})
    return resp.get("value", []) if codigo == 200 else []


def lanzar(ws_nombre, item_id, tipo):
    """Dispara el item. Devuelve (id_de_corrida, error)."""
    wid = workspaces().get(ws_nombre)
    trabajo = _TIPO_TRABAJO.get(tipo, "Pipeline")
    codigo, cuerpo, cab = api_bruta(
        f"workspaces/{wid}/items/{item_id}/jobs/instances?jobType={trabajo}", "POST", {})
    if codigo not in (200, 202):
        return None, f"HTTP {codigo}: {json.dumps(cuerpo, ensure_ascii=False)[:200]}"
    # El 202 no trae cuerpo: la corrida recien creada esta en Location.
    desde_cab = str(cab.get("Location", "")).rstrip("/").split("/")[-1]
    return str(cuerpo.get("id") or desde_cab or "aceptada"), ""


def estado_de_corrida(ws_nombre, item_id, corrida_id):
    """La corrida concreta, por su id: una llamada pequena, no el historial entero.

    historial() baja TODAS las paginas de corridas del item y se queda con una. Hacer eso
    cada 15 s durante media hora es mucha memoria y muchas llamadas para nada, y ademas
    coge «la mas reciente», que no tiene por que ser la nuestra si alguien lanzo el mismo
    pipeline a la vez. Devuelve None si el id no sirve, y entonces se cae al historial."""
    wid = workspaces().get(ws_nombre)
    if not wid or not corrida_id or corrida_id == "aceptada":
        return None
    codigo, cuerpo = api(f"workspaces/{wid}/items/{item_id}/jobs/instances/{corrida_id}")
    return cuerpo if codigo == 200 and isinstance(cuerpo, dict) else None


def esperar(ws_nombre, item_id, tope=None, intervalo=15, sospecha=None, corrida_id=None):
    """Bloquea hasta que la corrida alcanza un estado terminal. NUNCA da None.

    Dos cosas cambian respecto a la version anterior, las dos por el mismo
    motivo —que el tiempo no se pierda nunca:

      · Pasado `sospecha` avisa, pero NO para. Una corrida larga sigue siendo
        una corrida.
      · Al agotar el techo ya no cae del bucle devolviendo None —que borraba la
        corrida del reporte y dejaba el flujo sin duracion—: devuelve el ultimo
        estado conocido con los segundos que llevaba y la marca de que se dejo
        de esperar.
    """
    tope = TOPE_ESPERA if tope is None else tope
    sospecha = TOPE_SOSPECHA if sospecha is None else sospecha
    inicio = time.time()
    ultima, avisado = {}, False
    while True:
        # Primero la corrida por su id; solo si no hay id usable, el historial.
        directa = estado_de_corrida(ws_nombre, item_id, corrida_id)
        js = [directa] if directa else historial(ws_nombre, item_id, tope=1)
        transcurrido = time.time() - inicio
        if js:
            ultima = js[0]
            if str(ultima.get("status", "")).lower() not in ("inprogress", "notstarted"):
                ultima["_segundos_medidos"] = int(transcurrido)
                return ultima
        if transcurrido > sospecha and not avisado:
            print(f"   AVISO lleva {_hms(transcurrido)}, por encima de "
                  f"{_hms(sospecha)}. Es mucho — pero se sigue esperando y "
                  f"contando.")
            avisado = True
        if transcurrido >= tope:
            r = dict(ultima)
            r["status"] = ultima.get("status") or "EnCurso"
            r["_segundos_medidos"] = int(transcurrido)
            r["_se_dejo_de_esperar"] = True
            print(f"   se deja de esperar a los {_hms(transcurrido)} — "
                  f"la corrida sigue viva en Fabric, no se cancelo")
            return r
        time.sleep(intervalo)
    return None

## Paso 3 · PASOS — no editar

In [ ]:
import threading
# ════════════════════════════════════════════════════════════════════════════
#  PASO 3 · PASOS — no editar
# ════════════════════════════════════════════════════════════════════════════
# =============================================== 2 . RESOLVER
def normalizar(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())


def _nombre_ws(wid):
    for n, i in workspaces().items():
        if i == wid:
            return n
    return ""


def resolver(nombre, ambiente):
    """Encuentra el flujo en los workspaces del ambiente. Prefiere el pipeline."""
    clave, exactos, parciales, problemas = normalizar(nombre), [], [], []
    for ws in sorted(workspaces_de(ambiente)):
        try:
            lista = items(ws)
        except FabricNoResponde as e:
            # Un workspace que no se pudo listar no autoriza a decir "no existe".
            problemas.append(f"{ws}: {e}")
            continue
        for it in lista:
            if it.get("type") not in EJECUTABLES:
                continue
            n = normalizar(it.get("displayName"))
            reg = {"ws": ws, "id": it["id"], "nombre": it["displayName"],
                   "tipo": it["type"]}
            if n == clave:
                exactos.append(reg)
            elif clave in n or n in clave:
                parciales.append(reg)
    hallados = exactos or parciales
    # El Data Pipeline es el punto de entrada real: es el unico que pasa los
    # parametros al job. Lanzar el SparkJobDefinition suelto falla siempre.
    hallados.sort(key=lambda r: EJECUTABLES.index(r["tipo"]))
    return hallados, bool(exactos), problemas


# --- pipelines de analitica: el I/O viaja en los argumentos del Spark Job ----
def alias_de_biblioteca(defi):
    """{alias en el pipeline: nombre real de la variable}.

    El pipeline referencia `libraryVariables.vl_x_mi_variable`, pero en la
    Variable Library la variable se llama `mi_variable`. El bloque
    `libraryVariables` del descriptor es el que traduce.
    """
    decl = (defi.get("properties") or defi).get("libraryVariables") or {}
    return {a: str(v.get("variableName") or a) for a, v in decl.items()
            if isinstance(v, dict)}


def resolver_alias(ali, alias, vars_):
    """Valor de una variable referida por su alias, y el nombre con que se hallo.

    El bloque `libraryVariables` traduce el alias, pero en los pipelines de
    analitica de QA ese bloque viene vacio. Alli el alias es
    `<biblioteca>_<variable>` y la variable se llama solo `<variable>`: sin
    quitar el prefijo, TODA bandera de analitica queda «sin resolver» y el
    validador reporta que no hay salidas cuando si las hay.
    """
    if ali in (getattr(vars_, "origen", None) or {}):     # v9 · resuelto por biblioteca
        return ali, str(vars_[ali])
    if ali in alias:                       # el descriptor lo declara
        real = alias[ali]
        if real in vars_:
            return real, str(vars_[real])
    if ali in vars_:                       # el alias es el nombre mismo
        return ali, str(vars_[ali])
    # el alias trae el nombre de la biblioteca por delante
    partes = ali.split("_")
    for corte in range(1, len(partes)):
        cand = "_".join(partes[corte:])
        if cand in vars_:
            return cand, str(vars_[cand])
    return ali, "{" + str(ali) + "}"


# Banderas que no transportan datos: modo de escritura, particiones,
# credenciales, el zip del codigo. Se descartan por el final del nombre.
_OPCION = ("write-mode", "partitions", "batchsize", "isolation-level", "truncate",
           "encoding", "delimiter", "header", "repartition-count", "flow-code-zip",
           "create-schema-if-not-exists", "newline-substitution", "client-id",
           "client-secret", "tenant-id", "display-name", "access-token",
           "password", "url", "schema")
# El rol se decide por las palabras del nombre, no por el sufijo: el prefijo
# lleva el nombre del objeto y cambia en cada flujo.
_ES_SALIDA = ("output", "target", "sink", "destino", "rechazados")
_ES_ENTRADA = ("path", "table", "input", "source", "origen")
# Tablas de apoyo del framework: la bitacora de migraciones y compania. Llevan
# "table" en el nombre, asi que sin esto se clasificaban como ENTRADA y se
# comparaban contra el dato del negocio — que no significa nada, y bastaba con
# que una no se pudiera leer para dejar toda la transformacion en NO_MEDIBLE.
# Se miden y se muestran, pero no son entrada ni salida del flujo.
_ES_CONTROL = ("migrations-history", "migration-history", "migrations_history",
               "control-table", "audit-table", "bitacora")


_MODOS_REESCRITURA = ("overwrite", "rewrite", "truncate", "replace")


# El rol va al final del nombre de la bandera y el objeto va delante. Ese
# prefijo es lo que hermana una salida con SU modo de escritura:
#   --account-dq-output-table          <-> --account-dq-write-mode
#   --account-registros-rechazados-table <-> --account-registros-rechazados-write-mode
_SUFIJOS_DE_ROL = sorted(
    ("-write-mode", "-write_mode", "-output-table", "-output-path", "-input-path",
     "-output", "-input", "-table", "-path"), key=len, reverse=True)


def _prefijo_de_bandera(bandera):
    """El nombre del objeto, sin el rol: `adr2-sap-crm-output-table` -> `adr2-sap-crm`."""
    b = str(bandera).strip("-").lower()
    for suf in _SUFIJOS_DE_ROL:
        if b.endswith(suf):
            return b[:-len(suf)]
    return b


def modos_de_escritura(crudo, alias, vars_):
    """{prefijo: modo} de una actividad. Una actividad puede traer VARIOS.

    Los flujos reales declaran un modo POR SALIDA —`--account-dq-write-mode` y
    `--account-registros-rechazados-write-mode` en la misma actividad—, asi que
    quedarse con uno solo asignaria a una salida el modo de otra.

    El valor casi nunca es literal: viene por la Variable Library, igual que las
    rutas, y hay que resolverlo con el mismo mecanismo.
    """
    fuera = {}
    for bandera, ali in _banderas_del_argumento(crudo):
        b = str(bandera).strip("-").lower()
        if not (b.endswith("write-mode") or b.endswith("write_mode")):
            continue
        if ali:
            _real, valor = resolver_alias(ali, alias, vars_)
        else:
            m = re.search(rf"--{re.escape(bandera)}\s+([^\s',]+)", texto_crudo(crudo))
            valor = m.group(1) if m else ""
        fuera[_prefijo_de_bandera(bandera)] = valor
    return fuera


def modo_de_la_salida(bandera, modos):
    """El modo que le toca a ESTA salida. Devuelve (modo, como_se_decidio).

    Lo normal es que el nombre de la salida y el de su modo compartan prefijo
    (`--account-dq-output-table` con `--account-dq-write-mode`). Cuando no lo
    comparten —hay flujos que nombran el modo por el destino, `postgres` o
    `parquet`, y la salida por la ruta— no se adivina: si en la actividad hay un
    unico modo se le atribuye y se dice que fue por descarte, y si hay varios se
    deja sin modo antes que colgarle a una salida el modo de otra.
    """
    if not modos:
        return None, "la actividad no declara modo"
    pref = _prefijo_de_bandera(bandera)
    if pref in modos:
        return modos[pref], "emparejado por nombre"
    if len(modos) == 1:
        return list(modos.values())[0], "unico modo de la actividad, por descarte"
    return None, (f"la actividad declara {len(modos)} modos "
                  f"({', '.join(sorted(modos))}) y ninguno casa con «{pref}»")


def modo_de_escritura(crudo):
    """El modo de escritura si —y solo si— viene como literal en el descriptor.

    Se conserva para el caso literal, pero lo normal es que el valor llegue por
    la Variable Library: para eso esta `modos_de_escritura`, que ademas devuelve
    uno por salida.
    """
    t = texto_crudo(crudo)
    m = re.search(r"--[\w-]*write[-_]mode\s+([A-Za-z_]+)", t)
    return m.group(1) if m else None


def es_reescritura(modo):
    m = str(modo or "").strip().lower()
    return any(x in m for x in _MODOS_REESCRITURA) if m else None


def _rol_de_bandera(bandera):
    b = bandera.strip("-").lower()
    if any(b.endswith(o) for o in _OPCION):
        return None
    if any(p in b for p in _ES_CONTROL):
        return "control"
    if any(p in b for p in _ES_SALIDA):
        return "salida"
    if any(p in b for p in _ES_ENTRADA):
        return "entrada"
    return None


def _banderas_del_argumento(crudo):
    """[(bandera, alias_de_variable_o_literal)] leyendo la expresion cruda.

    Se parea en la expresion y no en el texto ya resuelto porque asi se sabe
    QUE variable alimenta cada bandera, que es lo que hace auditable la ruta.
    """
    txt = texto_crudo(crudo)
    pares = []
    # forma: '--mi-bandera ', pipeline().libraryVariables.ALIAS
    for m in re.finditer(
            r"--([A-Za-z0-9][\w-]*)\s*'\s*,\s*pipeline\(\)\.libraryVariables\.(\w+)",
            txt):
        pares.append((m.group(1), m.group(2)))
    # forma: --mi-bandera valor-literal (dentro de la misma cadena)
    for m in re.finditer(r"--([A-Za-z0-9][\w-]*)\s+([^\s',]+)", txt):
        if m.group(1) not in {b for b, _ in pares}:
            pares.append((m.group(1), None))
    return pares


def io_spark_job(defi, vars_):
    """Entradas y salidas de las actividades FabricSparkJobDefinition."""
    alias = alias_de_biblioteca(defi)
    vars_padre, alias_padre = vars_, alias
    fuera = []
    # Se recorre el arbol entero: un Spark job dentro de un ForEach o de un
    # flujo invocado es tan real como uno de primer nivel, y antes se saltaba.
    for _info in actividades_planas((defi.get("properties") or defi)
                                    .get("activities", [])):
        a = _info["actividad"]
        if a.get("type") != "FabricSparkJobDefinition":
            continue
        # v9 · un Spark job dentro de un flujo invocado usa las variables del hijo
        vars_ = vars_de_actividad(_info, vars_padre)
        alias = (alias_de_biblioteca(_DEF_HIJOS[_info["origen_pipeline"]])
                 if _info.get("origen_pipeline") in _DEF_HIJOS else alias_padre)
        tp = a.get("typeProperties") or {}
        crudo = tp.get("commandLineArguments")
        if not crudo:
            continue
        sjd_ws = str(tp.get("workspaceId") or "")
        sjd_id = str(tp.get("sparkJobDefinitionId") or "")
        lh_ws, lh_id = (lakehouse_del_spark_job(sjd_ws, sjd_id)
                        if sjd_ws and sjd_id else ("", ""))
        # El modo de escritura es de ESTA actividad: un flujo con varias salidas
        # puede reescribir una y acumular en otra, y antes la bandera se
        # descartaba por completo, asi que no quedaba en ninguna parte.
        _modos = modos_de_escritura(crudo, alias, vars_)
        for bandera, ali in _banderas_del_argumento(crudo):
            rol = _rol_de_bandera(bandera)
            if rol is None:
                continue
            if ali:
                real, valor = resolver_alias(ali, alias, vars_)
                plantilla = f"pipeline().libraryVariables.{ali}"
                usadas = {real: vars_.get(real)}
            else:
                # literal en la propia expresion
                m = re.search(rf"--{re.escape(bandera)}\s+([^\s',]+)", texto_crudo(crudo))
                valor = m.group(1) if m else ""
                plantilla = "literal en el descriptor"
                usadas = {}
            anclado = anclar_en_lakehouse(valor, lh_ws, lh_id)
            fuera.append({
                "actividad": a.get("name", ""), "rol": rol, "tipo": bandera,
                "dentro_de": _info["dentro_de"], "rama": _info["rama"],
                "iteraciones": _info["iteraciones"],
                "modo_escritura": (modo_de_la_salida(bandera, _modos)[0]
                                   if rol == "salida" else None),
                "modo_origen": (modo_de_la_salida(bandera, _modos)[1]
                                if rol == "salida" else None),
                "modos_en_la_actividad": (dict(_modos) if rol == "salida" else None),
                "reescribe": (es_reescritura(modo_de_la_salida(bandera, _modos)[0])
                              if rol == "salida" else None),
                "plantilla_carpeta": plantilla, "plantilla_archivo": "",
                "carpeta": valor.rstrip("/"),
                "archivo": valor.rstrip("/").split("/")[-1],
                "ruta": valor, "variables": usadas,
                "origen_variables": ({ali: (getattr(vars_, "origen", {}) or {}).get(ali)
                                      or (getattr(vars_, "faltantes", {}) or {}).get(ali)}
                                     if ali else {}),                          # v9
                "sin_resolver": rastro_sin_resolver(valor),                     # v9
                # v9 · `esquema.tabla` que no es del lakehouse: la lee el Spark por
                # su propia conexion (Postgres). No se busca en Tables/.
                "tabla_externa": bool(valor and not anclado and not rastro_sin_resolver(valor)
                                      and re.match(r"^[A-Za-z_]\w*\.[A-Za-z_]\w*$", valor)),
                "lakehouse": _nombre_lakehouse(lh_ws, lh_id),
                "ws_id": lh_ws, "artifact_id": lh_id,
                "abfss": anclado,
            })
        # El zip del codigo no es un dato, pero dice a que lakehouse apunta el
        # despliegue, asi que se anota con rol propio para no contarlo como I/O.
        m = re.search(r"--flow-code-zip\s+(\S+)", texto_crudo(crudo))
        if m:
            z = m.group(1)
            fuera.append({
                "actividad": a.get("name", ""), "rol": "codigo",
                "tipo": "flow-code-zip", "plantilla_carpeta": "literal en el descriptor",
                "plantilla_archivo": "", "carpeta": z.rsplit("/", 1)[0],
                "archivo": z.rsplit("/", 1)[-1], "ruta": z, "variables": {},
                "lakehouse": "", "ws_id": "", "artifact_id": "",
                "abfss": z if z.startswith("abfss://") else "",
            })
    return fuera


# =============================================== 3 . ENTRADAS Y SALIDAS
def contrato_io(obj, ambiente):
    """Entradas y salidas declaradas, leidas del descriptor del propio item.

    Las rutas del descriptor vienen como expresiones sobre la Variable Library;
    aqui se resuelven contra los valores del ambiente pedido, de modo que lo
    que se devuelve es lo que el flujo va a leer y escribir de verdad.
    """
    if obj["tipo"] == "DataPipeline":
        defi = definicion(obj["ws"], obj["id"], "pipeline-content.json")
        if not defi:
            return []
        usadas = bibliotecas_del_pipeline(defi)
        obj["bibliotecas"] = sorted(usadas)
        # v9 · cada alias contra SU biblioteca y SU variable, en el workspace del
        # flujo (orchestration o ingestion), con el value set activo
        vars_ = variables_del_pipeline(defi, ambiente, obj["ws"])
        obj["_vars"] = vars_
        obj["variables_origen"] = dict(vars_.origen)
        obj["variables_faltantes"] = dict(vars_.faltantes)
        # Ingesta declara el I/O en actividades Copy; analitica, en los
        # argumentos del Spark Job. Un pipeline puede tener de las dos.
        return io_pipeline(defi, vars_) + io_spark_job(defi, vars_)
    if obj["tipo"] == "SparkJobDefinition":
        vars_ = variables(ambiente)
        defi = definicion(obj["ws"], obj["id"], "sparkjobdefinitionv1.json") or {}
        fuera = []
        for m in re.finditer(r"--(\S+)\s+(\S+)", str(defi.get("commandLineArguments", ""))):
            bandera, valor = m.group(1), lit(m.group(2), vars_)
            rol = "salida" if re.search(r"load|sink|target|output|destino", bandera) else "entrada"
            fuera.append({"actividad": "spark", "rol": rol, "tipo": bandera,
                          "plantilla_carpeta": m.group(2), "plantilla_archivo": "",
                          "carpeta": valor, "archivo": valor.rstrip("/").split("/")[-1],
                          "ruta": valor,
                          "variables": {n: vars_.get(n)
                                        for n in variables_referidas(m.group(2))},
                          "lakehouse": "", "ws_id": "",
                          "artifact_id": "", "abfss": ""})
        return fuera
    return []


# =============================================== 4 . PREVUELO
def prevuelo(obj, io, ambiente):
    """Cinco comprobaciones antes de tocar nada. True si es ejecutable."""
    ok, et = True, obj["nombre"]

    # P1 . el objeto esta desplegado en el ambiente pedido
    if not obj["ws"].endswith(f"-{ambiente}"):
        anotar("BLOQUEANTE", et, f"resuelto en {obj['ws']}, que no es de {ambiente}",
               prueba="despliegue_en_ambiente", esperado=f"workspace de {ambiente}",
               obtenido=obj["ws"])
        ok = False
    else:
        anotar("OK", et, f"desplegado en {obj['ws']} como {obj['tipo']}",
               prueba="despliegue_en_ambiente", esperado=f"workspace de {ambiente}",
               obtenido=obj["ws"])

    # P2 . punto de entrada correcto
    if obj["tipo"] == "SparkJobDefinition":
        anotar("ADVERTENCIA", et,
               "el punto de entrada es un Spark Job Definition; quien pasa los "
               "parametros es el Data Pipeline que lo invoca",
               prueba="punto_de_entrada", esperado="DataPipeline",
               obtenido=obj["tipo"])
    else:
        anotar("OK", et, f"punto de entrada ejecutable: {obj['tipo']}",
               prueba="punto_de_entrada", esperado="DataPipeline",
               obtenido=obj["tipo"])

    # P3 . ninguna ruta quedo a medio resolver
    crudas = [(r, rastro_sin_resolver(r["ruta"])) for r in io]
    malas = [(r, m) for r, m in crudas if m]
    # v9 · dos casos que antes se anotaban igual. Si la variable no existe en la
    # biblioteca, o vale «placeholder», el flujo tampoco la va a encontrar: eso
    # bloquea. Si solo es que el validador no supo evaluar la expresion, el flujo
    # si puede correr: se avisa y se verifica tras la corrida.
    faltan = obj.get("variables_faltantes") or {}
    bloquean = 0
    for r, motivos in malas:
        vars_ruta = set(re.findall(r"\{([A-Za-z_][A-Za-z_0-9]*)\}", r["ruta"]))
        real = ("valor de relleno sin configurar" in motivos
                or any(v in faltan for v in vars_ruta))
        detalle = "; ".join(f"{v}: {faltan[v]}" for v in sorted(vars_ruta) if v in faltan)
        anotar("BLOQUEANTE" if real else "ADVERTENCIA", et,
               f"{r['rol']} sin resolver ({'; '.join(motivos)}): {r['ruta'][:120]}"
               + (f" · {detalle}" if detalle else "")
               + ("" if real else " · el validador no pudo evaluarla; el flujo si "
                                   "puede correr y se verifica tras la corrida"),
               prueba="rutas_resueltas", esperado="ruta sin expresiones pendientes",
               obtenido=r["ruta"][:200])
        bloquean += bool(real)
    if bloquean:
        ok = False
    elif io:
        anotar("OK", et, f"{len(io)} ruta(s) resueltas contra la Variable Library "
                         f"de {ambiente}",
               prueba="rutas_resueltas", esperado=f"{len(io)} rutas resueltas",
               obtenido=f"{len(io)} resueltas")

    # P4 . ningun destino apunta a otro ambiente
    fugas = [r for r in io if r["ws_id"] and _nombre_ws(r["ws_id"])
             and not _nombre_ws(r["ws_id"]).endswith(f"-{ambiente}")]
    for r in fugas:
        anotar("BLOQUEANTE", et,
               f"{r['rol']} apunta a {_nombre_ws(r['ws_id'])}, fuera de {ambiente}",
               prueba="aislamiento_de_ambiente", esperado=f"lakehouse de {ambiente}",
               obtenido=_nombre_ws(r["ws_id"]))
    if fugas:
        ok = False
    elif any(r["ws_id"] for r in io):
        anotar("OK", et, f"lakehouses de {ambiente} en todas las rutas ancladas",
               prueba="aislamiento_de_ambiente", esperado=f"lakehouse de {ambiente}",
               obtenido=sorted({r["lakehouse"] for r in io if r["lakehouse"]}))

    # P5 . hay al menos una salida identificada
    salidas = [r for r in io if r["rol"] == "salida"]
    if salidas:
        anotar("OK", et, f"{len(salidas)} salida(s) declarada(s)",
               prueba="salidas_declaradas", esperado=">= 1 salida",
               obtenido=[r["ruta"] for r in salidas])
    else:
        anotar("OMITIDO", et, "sin salidas declaradas en el descriptor: la "
                              "verificacion de datos no aplica a este objeto",
               prueba="salidas_declaradas", esperado=">= 1 salida", obtenido=0)
    return ok


# =============================================== 5 . DATOS (Spark)
# Caida de volumen tolerada entre una corrida y la siguiente antes de avisar.
CAIDA_MAXIMA = 0.30


def _spark():
    try:
        from pyspark.sql import SparkSession
        return SparkSession.builder.getOrCreate()
    except Exception:
        return None


# v5.2 · el formato se decide por DONDE vive la ruta, no solo por su extension.
# Lo que cuelga de Tables/ es Delta. Lo que cuelga de Files/ lo escribe la ingesta
# en parquet y NO tiene _delta_log: abrirlo con el lector Delta devolvia
# DELTA_TABLE_NOT_FOUND —"no existe"— sobre carpetas que si estaban, con filas.
# Ahora se intenta en orden y solo se reporta si fallan todos los intentos.
NO_ES_DELTA = re.compile(r"DELTA_TABLE_NOT_FOUND|DELTA_MISSING_DELTA_TABLE|is not a Delta table",
                         re.I)


def formatos_a_intentar(ruta):
    """Orden de lectores para esta ruta. El primero es el esperado."""
    r = str(ruta or "")
    if ".parquet" in r:
        return ["parquet"]
    if ".csv" in r:
        return ["csv"]
    if ".json" in r:
        return ["json"]
    if "/Tables/" in r:
        return ["delta", "parquet"]
    if "/Files/" in r:
        # Una carpeta de Files/ puede ser parquet (lo normal en bronze) o Delta
        # (silver escrita como tabla gestionada dentro de Files).
        return ["parquet", "delta"]
    return ["delta", "parquet"]


def leer(ruta):
    """DataFrame de una ruta abfss, probando los lectores que la ruta admite."""
    s = _spark()
    if s is None or not ruta:
        return None
    errores = []
    for fmt in formatos_a_intentar(ruta):
        try:
            lector = s.read.format(fmt)
            if fmt == "csv":
                lector = lector.option("header", "true")
            df = lector.load(ruta)
            FORMATO_LEIDO[ruta] = fmt
            return df
        except Exception as e:
            texto = str(e)
            causas = re.findall(r"Caused by:\s*([^\n]+)", texto)
            errores.append((fmt, (causas[-1].strip() if causas
                                  else f"{type(e).__name__}: {texto}")[:220]))
    # El motivo se GUARDA, no se traga: "Illegal Parquet type", "Path does not exist" o
    # un 403 son diagnosticos distintos y el reporte tiene que decir cual fue.
    detalle = " · ".join(f"como {f}: {m}" for f, m in errores)
    MOTIVOS[ruta] = f"Spark no pudo leerla: {detalle}"[:400]
    # Que ningun lector sirva NO prueba que la ruta falte: eso lo dice OneLake.
    SOLO_NO_DELTA[ruta] = all(NO_ES_DELTA.search(m) for _, m in errores)
    return None



# --- lectura de datos: Spark dentro de Fabric, OneLake REST fuera de el -----
DFS = "https://onelake.dfs.fabric.microsoft.com"
MOTIVOS = {}          # ruta -> por que no se pudo perfilar
FORMATO_LEIDO = {}    # ruta -> con que lector se pudo leer (v5.2)
SOLO_NO_DELTA = {}    # ruta -> el fallo fue "no es Delta", que no prueba ausencia (v5.2)


def _partes_abfss(url):
    """abfss://ws@host/artifact/resto  ->  (ws, artifact, resto)."""
    m = re.match(r"abfss://([^@]+)@[^/]+/([^/]+)/(.+)$", str(url or ""))
    return m.groups() if m else (None, None, None)


def _onelake(url, binario=False, timeout=180):
    pet = urllib.request.Request(
        url, headers={"Authorization": f"Bearer {_token('storage')}"})
    try:
        with urllib.request.urlopen(pet, timeout=timeout) as r:
            crudo = r.read()
            return r.status, (crudo if binario else json.loads(crudo.decode() or "{}"))
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()[:200]
    except Exception as e:
        return 0, str(e)


def _listar_onelake(ws, artifact, resto):
    """[(nombre, bytes)] de los archivos bajo la ruta. Vacio si es un archivo."""
    url = (f"{DFS}/{ws}?recursive=true&resource=filesystem"
           f"&directory={artifact}/{resto}")
    codigo, cuerpo = _onelake(url)
    if codigo != 200 or not isinstance(cuerpo, dict):
        return []
    return [(p["name"], int(p.get("contentLength") or 0))
            for p in cuerpo.get("paths", []) if p.get("contentLength")]


def _perfil_arrow(tablas, columnas_orden):
    """Mismo perfil que la rama Spark, calculado con pyarrow."""
    import hashlib
    import pyarrow as pa
    t = pa.concat_tables(tablas, promote_options="permissive") if len(tablas) > 1 else tablas[0]
    cols = sorted(t.column_names)
    filas = t.num_rows
    firma = ""
    if filas:
        # Suma de CRC32 por fila: igual que en Spark, no depende del orden.
        import zlib
        acc = 0
        datos = {c: t.column(c).to_pylist() for c in cols}
        for i in range(filas):
            linea = "".join(
                (" " if datos[c][i] is None else str(datos[c][i])) for c in cols)
            acc += zlib.crc32(linea.encode())
        firma = str(acc)
    tipos = {f.name: str(f.type) for f in t.schema}
    esquema = hashlib.sha256(
        "|".join(f"{c}:{_tipo_comun(tipos[c])}" for c in cols).encode()).hexdigest()[:16]
    return {"filas": filas, "columnas": len(cols), "esquema": esquema,
            "contenido": firma, "nombres": cols}


def _tipo_comun(t):
    """Normaliza el nombre del tipo para que Spark y Arrow coincidan."""
    t = str(t).lower()
    for arrow, comun in (("string", "string"), ("large_string", "string"),
                         ("int64", "bigint"), ("int32", "int"),
                         ("double", "double"), ("float", "float"),
                         ("bool", "boolean"), ("timestamp", "timestamp"),
                         ("date", "date")):
        if t.startswith(arrow):
            return comun
    return t


def _huella_onelake(url):
    """Perfil de la ruta leyendo OneLake por REST. None si no se pudo."""
    ws, artifact, resto = _partes_abfss(url)
    if not ws:
        MOTIVOS[url] = "la ruta no es abfss"
        return None
    try:
        import pyarrow.parquet as pq
    except ImportError:
        MOTIVOS[url] = "sin Spark ni pyarrow en este entorno"
        return None

    archivos = _listar_onelake(ws, artifact, resto)
    if not archivos:
        # Puede ser un archivo suelto y no una carpeta.
        codigo, datos = _onelake(f"{DFS}/{ws}/{artifact}/{resto}", binario=True)
        if codigo != 200:
            MOTIVOS[url] = f"no existe o sin acceso (HTTP {codigo})"
            return None
        archivos = [(f"{artifact}/{resto}", len(datos))]

    if any("_delta_log" in n for n, _ in archivos):
        # Leer todos los parquet de una tabla Delta contaria filas ya borradas.
        MOTIVOS[url] = ("tabla Delta: requiere Spark para respetar el log de "
                        "transacciones; no se perfila para no reportar de mas")
        return None

    partes = [n for n, tam in archivos
              if tam and n.endswith((".parquet", ".csv", ".json"))]
    if not partes:
        MOTIVOS[url] = f"{len(archivos)} archivo(s), ninguno legible como tabla"
        return None
    # Tope de bytes: esto se lee ENTERO en la memoria del proceso. Fuera de Fabric no hay
    # Spark y es la unica via, pero no a cualquier precio: pasado el tope se dice y no se lee.
    _tope = int(globals().get("LIMITE_MB_LECTURA_REST") or 512) * 1024 * 1024
    _bytes = sum(int(tam or 0) for n, tam in archivos if n in partes)
    if _bytes > _tope:
        MOTIVOS[url] = (f"{_bytes / 1048576:,.0f} MB en {len(partes)} archivo(s): demasiado "
                        f"para leer por REST en memoria (tope LIMITE_MB_LECTURA_REST = "
                        f"{_tope // 1048576} MB); requiere Spark")
        return None

    import io as _io
    tablas = []
    for nombre in sorted(partes):
        codigo, datos = _onelake(f"{DFS}/{ws}/{nombre}", binario=True)
        if codigo != 200:
            continue
        try:
            if nombre.endswith(".parquet"):
                tablas.append(pq.read_table(_io.BytesIO(datos)))
            elif nombre.endswith(".csv"):
                import pyarrow.csv as pcsv
                tablas.append(pcsv.read_csv(_io.BytesIO(datos)))
            else:
                import pyarrow.json as pjson
                tablas.append(pjson.read_json(_io.BytesIO(datos)))
        except Exception as e:
            MOTIVOS[url] = f"{nombre.split('/')[-1]} no se pudo abrir: {e}"
    if not tablas:
        return None
    return _perfil_arrow(tablas, None)


def _huella_spark(df, checksum=True):
    import hashlib
    from pyspark.sql import functions as F
    cols = sorted(df.columns)
    filas = df.count()
    firma = ""
    if filas and checksum:
        # Suma de CRC32 por fila: no depende del orden de las filas.
        cat = F.concat_ws("", *[F.coalesce(F.col(f"`{c}`").cast("string"), F.lit(" "))
                                for c in cols])
        firma = str(df.select(F.sum(F.crc32(cat).cast("decimal(38,0)"))).first()[0])
    tipos = {f.name: _tipo_comun(f.dataType.simpleString()) for f in df.schema.fields}
    esquema = hashlib.sha256(
        "|".join(f"{c}:{tipos[c]}" for c in cols).encode()).hexdigest()[:16]
    return {"filas": filas, "columnas": len(cols), "esquema": esquema,
            "contenido": firma, "nombres": cols}


_CACHE_HUELLA = {}
_LOCK_HUELLA = threading.Lock()


def huella(ruta, checksum=True):
    """Conteo, esquema y checksum del contenido. None si no se pudo perfilar.

    checksum=False se salta el CRC32 sobre todas las columnas concatenadas, que es la
    mitad del trabajo. Las ENTRADAS solo necesitan saber que existen y traen filas
    —revisar_entrada solo lee h["filas"] y h["columnas"]—, asi que ahi el checksum se
    calculaba para nada.

    El resultado se cachea por (ruta, checksum): una tabla que es salida de un flujo y
    entrada de otro se medía dos veces, y ahora una.

    Con Spark disponible se usa Spark. Fuera de Fabric se cae a OneLake por
    REST, que da el mismo perfil para parquet, csv y json.
    """
    if not ruta:
        return None
    _clave = (ruta, bool(checksum))
    with _LOCK_HUELLA:
        if _clave in _CACHE_HUELLA:
            return _CACHE_HUELLA[_clave]
        # Si ya se midio CON checksum, ese resultado sirve para quien lo pide sin el.
        if not checksum and (ruta, True) in _CACHE_HUELLA:
            return _CACHE_HUELLA[(ruta, True)]

    def _guardar(v):
        with _LOCK_HUELLA:
            _CACHE_HUELLA[_clave] = v
        return v

    if _spark() is not None:
        df = leer(ruta)
        if df is not None:
            return _guardar(_huella_spark(df, checksum))
        # Spark ESTA y no pudo leerla: el motivo ya quedo en MOTIVOS. Aqui NO se cae a REST.
        # La lectura por REST baja todos los archivos de la carpeta a la RAM del kernel y los
        # abre con pyarrow; con una entrada de bronze de varios GB el nodo mata el kernel
        # ("Ipython kernel exits with code -9") y se pierde la sesion entera. Dentro de Fabric
        # el motor es Spark; si Spark no puede, se reporta y se sigue.
        return _guardar(None)
    return _guardar(_huella_onelake(ruta))


def existe_en_onelake(abfss):
    """(existe, n_archivos, bytes) segun el listado de OneLake. v5.2.

    Es la unica fuente que distingue "la ruta no esta" de "la ruta esta pero el
    lector que probe no le sirve". Sin esto, un fallo de lectura se reportaba
    como ausencia.
    """
    ws, artifact, resto = _partes_abfss(abfss)
    if not ws:
        return None, 0, 0
    try:
        archivos = _listar_onelake(ws, artifact, resto)
    except Exception:
        return None, 0, 0
    if archivos:
        return True, len(archivos), sum(b for _, b in archivos)
    # Sin hijos puede ser un archivo suelto: se pregunta por el padre.
    padre = resto.rsplit("/", 1)[0] if "/" in resto else resto
    try:
        hermanos = _listar_onelake(ws, artifact, padre)
    except Exception:
        return None, 0, 0
    propio = [(n, b) for n, b in hermanos if n.endswith(resto)]
    if propio:
        return True, len(propio), sum(b for _, b in propio)
    return False, 0, 0


def revisar_entrada(r, et):
    """Una entrada debe existir y traer filas antes de ejecutar."""
    if r.get("sin_resolver"):
        # v9 · antes se media la plantilla literal y salia «no existe en OneLake»
        anotar("OMITIDO", f"{et} / entrada",
               f"{r['ruta']}: ruta sin resolver ({'; '.join(r['sin_resolver'])}); "
               "ya se informo en el prevuelo y no se mide un nombre que no existe",
               prueba="entrada_legible", esperado="ruta resuelta", obtenido="sin resolver")
        return
    if r.get("tabla_externa"):
        # v9 · `esquema.tabla` cuyo esquema no existe en el lakehouse: la lee el Spark
        # por su propia conexion (Postgres). Antes se buscaba en Tables/ y daba bloqueo.
        anotar("OMITIDO", f"{et} / entrada",
               f"{r['ruta']}: tabla fuera del lakehouse (el esquema no existe en Tables/); "
               "la lee el flujo por su conexion, p. ej. Postgres. Se verifica tras la corrida",
               prueba="entrada_legible", esperado="tabla del lakehouse",
               obtenido="tabla externa")
        return
    if not r["abfss"]:
        anotar("OMITIDO", f"{et} / entrada",
               f"origen {r['tipo']} externo ({r['ruta']}): no es legible desde "
               "aqui, lo resuelve la conexion del pipeline. Queda verificado "
               "tras la corrida con las filas leidas que reporta la actividad",
               prueba="entrada_legible", esperado="lectura directa del origen",
               obtenido="no accesible desde el motor de validacion")
        return
    h = huella(r["abfss"], checksum=False)
    if h is None:
        motivo = MOTIVOS.get(r["abfss"], "no legible")
        # v5.2 · antes de llamarlo bloqueo, se comprueba si la ruta existe.
        # Un fallo de lectura no es una ausencia: son dos hallazgos distintos.
        existe, n_arch, _bytes = existe_en_onelake(r["abfss"])
        if "requiere Spark" in motivo or "sin Spark" in motivo:
            nivel = "OMITIDO"
        elif existe is True:
            nivel = "ADVERTENCIA"
            motivo = (f"existe en OneLake ({n_arch} archivo(s), "
                      f"{_bytes / 1048576:,.1f} MB) pero el motor no la pudo perfilar: "
                      + ("no es una tabla Delta y ningun lector sirvio"
                         if SOLO_NO_DELTA.get(r["abfss"]) else motivo))
        elif existe is False:
            nivel = "BLOQUEANTE"
            motivo = f"no existe en OneLake · {motivo}"
        else:
            nivel = "ADVERTENCIA"
            motivo = f"no se pudo confirmar si existe · {motivo}"
        anotar(nivel, f"{et} / entrada", f"{r['ruta']}: {motivo}")
    elif h["filas"] == 0:
        anotar("ADVERTENCIA", f"{et} / entrada", f"existe pero esta vacia: {r['ruta']}")
    else:
        anotar("OK", f"{et} / entrada",
               f"{h['filas']} filas, {h['columnas']} columnas - {r['ruta']}",
               prueba="entrada_legible", esperado="> 0 filas",
               obtenido={"filas": h["filas"], "columnas": h["columnas"]})


def comparar(antes, despues, et, escritura_confirmada=None):
    """Contrasta las dos huellas de una salida y anota lo que cambio.

    `escritura_confirmada` viene del detalle de la corrida. Sin ese dato, una
    salida con el mismo contenido de antes es sospechosa. Con el, se sabe que
    el flujo si escribio y que lo identico es que el origen no habia cambiado.
    """
    if antes is None and despues is None:
        anotar("OMITIDO", et, "no perfilable antes ni despues; sin comparacion")
        return
    if antes is None:
        anotar("OK", et, f"salida creada: {despues['filas']} filas, "
                         f"{despues['columnas']} columnas",
               prueba="salida_creada", esperado="la salida no existia antes",
               obtenido={"filas": despues["filas"], "columnas": despues["columnas"]})
        return
    if despues is None:
        anotar("BLOQUEANTE", et, "la salida existia antes y ahora no se puede leer")
        return

    if despues["esquema"] != antes["esquema"]:
        nuevas = sorted(set(despues["nombres"]) - set(antes["nombres"]))
        idas = sorted(set(antes["nombres"]) - set(despues["nombres"]))
        detalle = " ".join(x for x in (f"+{nuevas}" if nuevas else "",
                                       f"-{idas}" if idas else "") if x)
        anotar("BLOQUEANTE", et, "el esquema cambio " + (detalle or "(tipos)"),
               prueba="esquema_estable", esperado=antes["nombres"],
               obtenido=despues["nombres"])
    else:
        anotar("OK", et, f"esquema estable: {despues['columnas']} columnas",
               prueba="esquema_estable", esperado=antes["nombres"],
               obtenido=despues["nombres"])

    delta = despues["filas"] - antes["filas"]
    if despues["filas"] == 0:
        anotar("BLOQUEANTE", et, "la salida quedo vacia",
               prueba="salida_con_datos", esperado="> 0 filas", obtenido=0)
    elif despues["contenido"] == antes["contenido"]:
        if escritura_confirmada:
            anotar("OK", et, f"{despues['filas']} filas reescritas con el mismo "
                             "contenido: el origen no habia cambiado",
                   prueba="contenido_actualizado",
                   esperado="escritura confirmada por la corrida",
                   obtenido={"filas": despues["filas"], "contenido": "identico"})
        else:
            anotar("ADVERTENCIA", et,
                   f"{despues['filas']} filas, contenido identico al de antes: "
                   "no hay senal de que la corrida escribiera",
                   prueba="contenido_actualizado",
                   esperado="senal de escritura",
                   obtenido={"filas": despues["filas"], "contenido": "identico"})
    elif antes["filas"] and delta < 0 and abs(delta) / antes["filas"] > CAIDA_MAXIMA:
        # Una carga que trunca deja la salida legible y con esquema intacto:
        # el unico rastro es el volumen, asi que aqui no puede pasar en silencio.
        anotar("ADVERTENCIA", et,
               f"el volumen cayo {abs(delta) / antes['filas']:.0%}: "
               f"{antes['filas']} -> {despues['filas']} filas ({delta:+d})",
               prueba="volumen_sin_caida", esperado=f"caida <= {CAIDA_MAXIMA:.0%}",
               obtenido={"antes": antes["filas"], "despues": despues["filas"]})
    else:
        anotar("OK", et, f"contenido actualizado: {antes['filas']} -> "
                         f"{despues['filas']} filas ({delta:+d})",
               prueba="volumen_sin_caida", esperado=f"caida <= {CAIDA_MAXIMA:.0%}",
               obtenido={"antes": antes["filas"], "despues": despues["filas"]})


def revisar_copia(acts, et, filas_salida=None):
    """Contrasta lo que el pipeline dice haber movido con lo que hay escrito.

    Cierra el circuito de la ingesta: filas en el origen -> filas copiadas ->
    filas que quedaron en el lakehouse. Si los tres numeros no coinciden, algo
    se perdio por el camino aunque la corrida figure como Completed.
    """
    if not acts:
        anotar("OMITIDO", et, "sin detalle por actividad para esta corrida")
        return
    for a in acts:
        nombre = a.get("activityName") or "actividad"
        estado = str(a.get("status", ""))
        o = a.get("output") or {}
        leidas, copiadas = o.get("rowsRead"), o.get("rowsCopied")
        etq = f"{et} / {nombre}"

        if estado.lower() != "succeeded":
            anotar("BLOQUEANTE", etq, f"la actividad termino en {estado}")
            continue

        if leidas is None and copiadas is None:
            anotar("OMITIDO", etq,
                   f"{estado} en {a.get('durationInMs', '?')} ms; la actividad no "
                   "reporta filas (no es una copia de datos)")
            continue

        # Entrada: es lo unico que dice cuanto trajo del origen.
        if leidas == 0:
            anotar("ADVERTENCIA", etq, "el origen no devolvio ninguna fila",
                   prueba="entrada_con_datos", esperado="> 0 filas", obtenido=0)
        else:
            anotar("OK", etq,
                   f"entrada: {leidas} filas leidas del origen "
                   f"({o.get('filesRead', '?')} archivo(s), "
                   f"{o.get('dataRead', '?')} bytes)",
                   prueba="entrada_con_datos", esperado="> 0 filas",
                   obtenido={"filas": leidas, "archivos": o.get("filesRead"),
                             "bytes": o.get("dataRead")})

        # Transito: leidas frente a copiadas.
        if leidas is not None and copiadas is not None and leidas != copiadas:
            anotar("BLOQUEANTE", etq,
                   f"se perdieron filas en la copia: {leidas} leidas, "
                   f"{copiadas} copiadas",
                   prueba="copia_sin_perdida", esperado=leidas, obtenido=copiadas)
        elif copiadas is not None:
            anotar("OK", etq, f"copiadas las {copiadas} filas sin perdida",
                   prueba="copia_sin_perdida", esperado=leidas, obtenido=copiadas)

        # Cierre: lo copiado frente a lo que hay de verdad en el destino.
        if filas_salida is not None and copiadas is not None:
            if copiadas == filas_salida:
                anotar("OK", etq,
                       f"cuadra con el destino: {filas_salida} filas escritas",
                       prueba="cuadre_origen_destino", esperado=copiadas,
                       obtenido=filas_salida)
            else:
                anotar("BLOQUEANTE", etq,
                       f"el pipeline dice {copiadas} filas copiadas pero el "
                       f"destino tiene {filas_salida}",
                       prueba="cuadre_origen_destino", esperado=copiadas,
                       obtenido=filas_salida)


def hubo_escritura(acts):
    """True si alguna actividad de la corrida reporta datos escritos."""
    for a in acts:
        o = a.get("output") or {}
        if (o.get("rowsCopied") or 0) > 0 or (o.get("filesWritten") or 0) > 0:
            return True
    return False

## Paso 4 · Resolver los flujos

Busca cada nombre entre todos los workspaces del ambiente. Prefiere el Data
Pipeline sobre el Spark Job Definition porque el pipeline es el que pasa los
parametros al job.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 4 · Resolver los flujos
# ════════════════════════════════════════════════════════════════════════════
OBJETOS = []
print(f"ambiente: {AMBIENTE}   workspaces visibles: {len(workspaces_de(AMBIENTE))}")
for ws in sorted(workspaces_de(AMBIENTE)):
    try:
        print(f"   {ws:44} {len(items(ws)):3} items")
    except FabricNoResponde as e:
        print(f"   {ws:44}  no legible: {e}")

print()
for nombre in FLUJOS:
    en_flujo(nombre)
    _t = time.time()
    hallados, exacto, problemas = resolver(nombre, AMBIENTE)
    cronometrar(nombre, "resolucion", time.time() - _t)
    for p in problemas:
        anotar("BLOQUEANTE", nombre, f"workspace no verificable - {p}",
               prueba="resolucion", esperado="workspace legible", obtenido=str(p))
    if not hallados:
        falta = (f"no existe ningun objeto ejecutable en {AMBIENTE}" if not problemas
                 else f"no encontrado, y {len(problemas)} workspace(s) no se pudieron leer")
        anotar("BLOQUEANTE", nombre, falta, prueba="resolucion",
               esperado=f"un objeto ejecutable en {AMBIENTE}", obtenido="ninguno")
        continue
    elegido = hallados[0]
    elegido["solicitado"] = nombre
    OBJETOS.append(elegido)
    coincide = "exacta" if exacto else "aproximada"
    print(f"{nombre}")
    print(f"   -> {elegido['nombre']}  [{elegido['tipo']}]  ({coincide})")
    print(f"      {elegido['ws']}")
    for otro in hallados[1:]:
        print(f"      tambien: {otro['nombre']} [{otro['tipo']}] en {otro['ws']}")

## Paso 5 · Entradas y salidas declaradas

Lee el descriptor del propio objeto en Fabric y resuelve cada ruta contra la
Variable Library del ambiente. Lo que aparece aqui es lo que el flujo va a
leer y escribir de verdad, ya con los valores de `{AMBIENTE}` sustituidos.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 5 · Entradas y salidas declaradas
# ════════════════════════════════════════════════════════════════════════════
IO = {}
for obj in OBJETOS:
    en_flujo(obj["solicitado"])
    _t = time.time()
    try:
        io = contrato_io(obj, AMBIENTE)
    except Exception as e:
        io = []
        anotar("BLOQUEANTE", obj["nombre"], f"no se pudo leer el descriptor: {e}",
               prueba="lectura_descriptor", esperado="descriptor legible",
               obtenido=f"{type(e).__name__}: {e}")
    IO[obj["id"]] = io
    cronometrar(obj["solicitado"], "lectura_descriptor", time.time() - _t)
    print(f"\n{obj['nombre']}  [{obj['tipo']}]")
    if obj.get("bibliotecas"):
        print(f"   variables desde: {', '.join(obj['bibliotecas'])}")
    for _ali, _por in sorted((obj.get("variables_faltantes") or {}).items()):
        print(f"   !! variable no encontrada: {_ali} -> {_por}")
    if not io:
        print("   sin entradas ni salidas declaradas en el descriptor")
    for r in io:
        print(f"\n   {r['rol'].upper():8} {r['tipo']}")
        # Las dos caras: como lo declara el descriptor y a que resuelve.
        print(f"      plantilla carpeta : {r['plantilla_carpeta'] or '(vacia)'}")
        print(f"      plantilla archivo : {r['plantilla_archivo'] or '(vacia)'}")
        if r["variables"]:
            print("      variables         :")
            for n, v in sorted(r["variables"].items()):
                marca = v if v not in (None, "") else "<<SIN VALOR EN QA>>"
                print(f"          {n:34} = {marca}")
                _org = (r.get("origen_variables") or {}).get(n)
                if _org:
                    print(f"          {'':34}   desde {_org}")
        print(f"      carpeta real      : {r['carpeta'] or '(raiz)'}")
        if r.get("por_partes"):
            print(f"      ARCHIVO REAL      : {r['patron_archivo']}  "
                  "(se escribe en partes: se mide la carpeta completa)")
        else:
            print(f"      ARCHIVO REAL      : {r['archivo'] or '(carpeta completa)'}")
        if r.get("intermedia"):
            print("      NOTA              : salida intermedia (staging): el flujo la "
                  "consume y la reescribe; no permanece tras la corrida")
        print(f"      ruta completa     : {r['ruta']}")
        if r["lakehouse"]:
            print(f"      lakehouse         : {r['lakehouse']}  "
                  f"({_nombre_ws(r['ws_id']) or r['ws_id']})")
        if r["abfss"]:
            print(f"      abfss             : {r['abfss']}")

## Paso 6 · Prevuelo

Cinco comprobaciones, ninguna escribe nada:

1. el objeto esta desplegado en el ambiente pedido
2. el punto de entrada es el correcto
3. toda variable de ruta quedo resuelta contra la Variable Library
4. ningun destino apunta a otro ambiente
5. hay al menos una salida identificada

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 6 · Prevuelo
# ════════════════════════════════════════════════════════════════════════════
LISTOS = []
for obj in OBJETOS:
    en_flujo(obj["solicitado"])
    print(f"\n{obj['nombre']}")
    _t = time.time()
    if prevuelo(obj, IO.get(obj["id"], []), AMBIENTE):
        LISTOS.append(obj)
    cronometrar(obj["solicitado"], "prevuelo", time.time() - _t)

print("\n" + "-" * 70)
print(f"EJECUTABLES: {len(LISTOS)} de {len(OBJETOS)}")
for obj in LISTOS:
    print(f"   {obj['nombre']}")

## Paso 7 · Línea base

Antes de ejecutar: revisa que las entradas existan y traigan filas, y toma la
huella de cada salida (conteo, esquema y checksum de contenido). Esa huella es
lo que despues permite afirmar que la corrida cambio los datos.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

# ════════════════════════════════════════════════════════════════════════════
#  PASO 7 · Línea base  ·  EN PARALELO
# ════════════════════════════════════════════════════════════════════════════
# Antes: un flujo detras de otro, y por cada uno un count + un CRC32 sobre todas las
# columnas de cada entrada y de cada salida. Dos recorridos completos por tabla, en serie.
#
# Ahora, tres cosas:
#   1. las ENTRADAS se miden sin checksum: revisar_entrada solo mira filas y columnas
#   2. cada RUTA se mide UNA vez aunque sea salida de un flujo y entrada de otro (cache)
#   3. las mediciones van en paralelo; los print y los anotar, en serie y en orden
BASE = {}
_t_p7 = time.time()

# 1 · todas las rutas a medir, sin repetir
_rutas_entrada, _rutas_salida = [], []
for obj in LISTOS:
    for r in IO.get(obj["id"], []):
        if r["rol"] == "entrada" and r.get("abfss"):
            _rutas_entrada.append(r["abfss"])
        elif r["rol"] == "salida" and r.get("abfss"):
            _rutas_salida.append(r["abfss"])
_unicas = [(x, False) for x in dict.fromkeys(_rutas_entrada)] + \
          [(x, True) for x in dict.fromkeys(_rutas_salida)]
_ahorro = (len(_rutas_entrada) + len(_rutas_salida)) - len(_unicas)

_hl = max(1, int(globals().get("HILOS_LINEA_BASE") or 1))
print("=" * 78)
print(f"LINEA BASE · {len(_unicas)} ruta(s) unica(s) · {_hl} en paralelo")
if _ahorro > 0:
    print(f"   {_ahorro} medicion(es) ahorrada(s): rutas que se repiten entre flujos")
print(f"   las entradas van SIN checksum (solo se comprueba que existan y traigan filas)")
print("=" * 78)

# 2 · medir en paralelo. Solo red y Spark, nada de estado compartido.
_hechas = 0
_lock_p7 = threading.Lock()


def _medir_ruta(par):
    global _hechas
    ruta, con_checksum = par
    h = huella(ruta, checksum=con_checksum)
    with _lock_p7:
        globals()["_hechas"] = globals().get("_hechas", 0) + 1
        n = globals()["_hechas"]
        _et = "sin datos" if h is None else f"{h['filas']} filas"
        print(f"   [{n:>3}/{len(_unicas)}] {_et:<16} {str(ruta).split('/')[-1][:52]}",
              flush=True)
    return h


if len(_unicas) > 1 and _hl > 1:
    with ThreadPoolExecutor(max_workers=min(_hl, len(_unicas)),
                            thread_name_prefix="base") as _pool:
        list(_pool.map(_medir_ruta, _unicas))
else:
    for _p in _unicas:
        _medir_ruta(_p)
print(f"\nmedicion terminada en {_hms(time.time() - _t_p7)}\n")

# 3 · el informe, en serie y en el orden de siempre. Todo sale ya de la cache.
for obj in LISTOS:
    en_flujo(obj["solicitado"])
    et = obj["nombre"]
    print(f"\n{et}")
    _t = time.time()
    for r in IO.get(obj["id"], []):
        if r["rol"] == "entrada":
            revisar_entrada(r, et)
    BASE[obj["id"]] = {}
    for r in IO.get(obj["id"], []):
        if r["rol"] != "salida":
            continue
        h = huella(r["abfss"])
        BASE[obj["id"]][r["ruta"]] = h
        if h is None:
            print(f"   [ base ] salida aun no existe: {r['ruta']}")
        else:
            print(f"   [ base ] {h['filas']} filas, {h['columnas']} columnas "
                  f"- {r['ruta']}")
    cronometrar(obj["solicitado"], "linea_base", time.time() - _t)

print(f"\n{'=' * 78}")
print(f"PASO 7 COMPLETO: {_hms(time.time() - _t_p7)}")
print("=" * 78)


## Paso 8 · Ejecutar

**Esta es la unica celda que actua.** Requiere `EJECUTAR = True` en la celda 1.
Lanza el objeto, espera a que alcance un estado terminal y reporta duracion e
identificador de corrida.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

# ════════════════════════════════════════════════════════════════════════════
#  PASO 8 · Ejecutar  ·  POR OLAS
# ════════════════════════════════════════════════════════════════════════════
CORRIDAS = {}
_LOCK_OLA = threading.Lock()

# ── MOTOR DE OLAS · quien depende de quien va en serie, el resto en paralelo ──────────
# El criterio es el mismo que usa grafo_es() en el Paso 8.2: una SALIDA de A es ENTRADA de
# B si las rutas normalizadas coinciden o una es prefijo de la otra (carpeta contra archivo,
# o particion dentro). Se reimplementa aqui, y no se reutiliza grafo_es, porque aquella se
# define en el Paso 8.2 —despues de este— y moverla arrastraria clave_ruta y _graphlib.
#
# Es CONSERVADOR en un sentido y explicito en el otro:
#   · si se puede probar que B lee algo que escribe A, B espera a A. Siempre.
#   · si un flujo no declara E/S, NO se puede probar dependencia, asi que va en paralelo.
#     Eso se DICE en pantalla, porque es la unica suposicion que hace el motor.
def _clave_ruta_ola(v):
    """Normaliza una ruta para compararla. Minusculas, sin barras sobrantes."""
    s = str(v or "").strip().lower().replace("\\", "/")
    while "//" in s:
        s = s.replace("//", "/")
    return s.strip("/")


def _es_de(obj, rol):
    """Las rutas de un flujo en ese rol. El codigo (el zip) no cuenta como dato."""
    return {_clave_ruta_ola(r.get("ruta")) for r in IO.get(obj["id"], [])
            if r.get("rol") == rol and r.get("ruta")}


def _depende(rutas_entrada, rutas_salida):
    """True si algo de `rutas_entrada` lo escribe `rutas_salida`."""
    for e in rutas_entrada:
        for s in rutas_salida:
            if e == s or e.startswith(s + "/") or s.startswith(e + "/"):
                return True
    return False


def plan_de_olas(listos):
    """Agrupa los flujos en OLAS. Dentro de una ola nada depende de nada: van a la vez.

    Devuelve (olas, sin_es, aristas) donde `olas` es una lista de listas de objetos."""
    ent = {o["id"]: _es_de(o, "entrada") for o in listos}
    sal = {o["id"]: _es_de(o, "salida") for o in listos}
    sin_es = [o for o in listos if not ent[o["id"]] and not sal[o["id"]]]

    espera = {o["id"]: set() for o in listos}
    aristas = []
    for b in listos:
        for a in listos:
            if a["id"] == b["id"]:
                continue
            if _depende(ent[b["id"]], sal[a["id"]]):
                espera[b["id"]].add(a["id"])
                aristas.append((a["solicitado"], b["solicitado"]))

    olas, pendientes, hechos = [], list(listos), set()
    while pendientes:
        ola = [o for o in pendientes if not (espera[o["id"]] - hechos)]
        if not ola:
            # Ciclo: nadie puede empezar. Se rompe metiendo lo que queda en UNA ola en
            # serie, que es como se comportaba antes, y se avisa.
            print(f"   ⚠️ {len(pendientes)} flujo(s) en ciclo de dependencias: se lanzan "
                  f"en serie, en el orden de FLUJOS.")
            olas.extend([[o] for o in pendientes])
            break
        olas.append(ola)
        for o in ola:
            hechos.add(o["id"])
        pendientes = [o for o in pendientes if o["id"] not in hechos]
    return olas, sin_es, aristas


def _lanzar_y_esperar(obj):
    """Solo red: arranca el flujo y espera. NADA de anotar() ni de estado compartido.

    La contabilidad (anotar, cronometrar, los print del detalle) se hace fuera, en serie y
    en orden, para no tener que poner cerrojos por todas partes ni entrelazar la salida."""
    t0 = time.time()
    previas = historial(obj["ws"], obj["id"], tope=1)
    corrida_id, error = lanzar(obj["ws"], obj["id"], obj["tipo"])
    if corrida_id is None:
        return {"obj": obj, "previas": previas, "corrida_id": None, "error": error,
                "final": None, "segundos": time.time() - t0}
    with _LOCK_OLA:
        print(f"   ▶ lanzado   {obj['nombre'][:58]:58} {corrida_id}", flush=True)
    final = esperar(obj["ws"], obj["id"], tope=TOPE_ESPERA,
                    sospecha=TOPE_SOSPECHA, corrida_id=corrida_id)
    seg = time.time() - t0
    with _LOCK_OLA:
        _e = str((final or {}).get("status", "")) or "sin corrida"
        _ico = "✅" if _e.lower() == "completed" else "❌"
        print(f"   {_ico} {obj['nombre'][:58]:58} {_e:12} {_hms(seg)}", flush=True)
    return {"obj": obj, "previas": previas, "corrida_id": corrida_id, "error": None,
            "final": final, "segundos": seg}


def _contabilizar(r):
    """Todo lo que toca estado compartido, en serie y en el orden de la ola."""
    obj, final = r["obj"], r["final"]
    et = obj["nombre"]
    en_flujo(obj["solicitado"])
    print(f"\n{et}  [{obj['tipo']}] en {obj['ws']}")
    if r["previas"]:
        print(f"   ultima corrida: {r['previas'][0].get('status')} "
              f"{r['previas'][0].get('startTimeUtc')} ({duracion(r['previas'][0])})")
    if r["corrida_id"] is None:
        anotar("BLOQUEANTE", et, f"no arranco - {r['error']}",
               prueba="lanzamiento", esperado="corrida creada", obtenido=r["error"])
        return
    CORRIDAS[obj["id"]] = final
    cronometrar(obj["solicitado"], "ejecucion_total", r["segundos"])
    print(f"   {texto_duracion(final)}")
    if not final:
        anotar("ADVERTENCIA", et, "no se pudo leer ninguna corrida",
               prueba="corrida_exitosa", esperado="Completed", obtenido="sin corrida")
        return
    estado = str(final.get("status", ""))
    if final.get("_se_dejo_de_esperar"):
        anotar("ADVERTENCIA", et,
               f"seguia en «{estado or 'EnCurso'}» tras "
               f"{_hms(duracion_segundos(final))}; el validador dejo de "
               f"esperar (TOPE_ESPERA = {TOPE_ESPERA}s). La corrida NO se "
               f"cancelo.",
               prueba="corrida_exitosa", esperado="Completed",
               obtenido=estado or "sin terminar")
        return
    if estado.lower() == "completed":
        anotar("OK", et, f"corrida Completed en {duracion(final)}",
               prueba="corrida_exitosa", esperado="Completed", obtenido=estado)
    else:
        anotar("BLOQUEANTE", et,
               f"corrida termino en {estado}: "
               f"{json.dumps(final.get('failureReason') or {})[:250]}",
               prueba="corrida_exitosa", esperado="Completed", obtenido=estado)


if not EJECUTAR:
    print("EJECUTAR = False -> no se lanza nada.")
    print("Revisa el prevuelo y, si esta conforme, pon EJECUTAR = True en la celda 1.")
elif not LISTOS:
    print("Ningun flujo paso el prevuelo: no hay nada que lanzar.")
else:
    _hilos_f = max(1, int(globals().get("HILOS_FLUJOS") or 1))
    if not globals().get("ORDEN_AUTOMATICO", True) or _hilos_f <= 1:
        _olas, _sin_es, _aristas = [[o] for o in LISTOS], [], []
        print(f"Orden de FLUJOS, de uno en uno ({len(LISTOS)} flujo(s)).")
    else:
        _olas, _sin_es, _aristas = plan_de_olas(LISTOS)
        print("=" * 78)
        print(f"PLAN DE EJECUCION · {len(LISTOS)} flujo(s) en {len(_olas)} ola(s) · "
              f"{_hilos_f} a la vez")
        print("=" * 78)
        print(f"   {len(_aristas)} dependencia(s) detectada(s) entre los flujos")
        for _a, _b in _aristas[:12]:
            print(f"      {_b}  espera a  {_a}")
        if len(_aristas) > 12:
            print(f"      … y {len(_aristas) - 12} mas")
        if _sin_es:
            print(f"   ℹ️ {len(_sin_es)} flujo(s) no declaran entradas ni salidas: NO se puede")
            print(f"      probar dependencia, asi que van en paralelo. Es la unica suposicion")
            print(f"      que hace el plan. Si alguno depende de otro, se vera al fallar.")
        for _i, _ola in enumerate(_olas, start=1):
            print(f"   ola {_i}: {len(_ola)} flujo(s)")
        print("=" * 78)

    _t_total = time.time()
    for _i, _ola in enumerate(_olas, start=1):
        print(f"\n{'─' * 78}")
        print(f"OLA {_i}/{len(_olas)} · {len(_ola)} flujo(s) · "
              f"{min(_hilos_f, len(_ola))} a la vez")
        print("─" * 78)
        _t_ola = time.time()
        if len(_ola) == 1 or _hilos_f <= 1:
            _res = [_lanzar_y_esperar(_o) for _o in _ola]
        else:
            _res = [None] * len(_ola)
            with ThreadPoolExecutor(max_workers=min(_hilos_f, len(_ola)),
                                    thread_name_prefix="flujo") as _pool:
                _futs = {_pool.submit(_lanzar_y_esperar, _o): _k
                         for _k, _o in enumerate(_ola)}
                for _f in _futs:
                    _res[_futs[_f]] = _f.result()
        _ok = sum(1 for _r in _res
                  if str((_r["final"] or {}).get("status", "")).lower() == "completed")
        print(f"\nOLA {_i} terminada: {_ok} OK · {len(_res) - _ok} con problema · "
              f"{_hms(time.time() - _t_ola)}")
        # La contabilidad va aparte, en serie: asi el informe sale en orden y no hace
        # falta un cerrojo por cada cosa que anotar() toca.
        for _r in _res:
            _contabilizar(_r)
    print(f"\n{'=' * 78}")
    print(f"TODAS LAS OLAS: {_hms(time.time() - _t_total)}")
    print("=" * 78)


### Paso 8.1 · Entradas y salidas tras la corrida

Igual que el paso 5.1, pero **despues** de ejecutar: mide las rutas que cada flujo declara —sus entradas y sus salidas, no otras— y las enmarca en una tabla.

Cada ruta se mide una sola vez aunque la compartan varios flujos. No recorre el lakehouse ni busca datos donde el descriptor no dijo que los hubiera.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 8.1 · Entradas y salidas DESPUES de la corrida
# ════════════════════════════════════════════════════════════════════════════
# El paso 5.1 mira el grafo antes de ejecutar: quien depende de quien y si las rutas existen.
# Este mira lo mismo DESPUES, con los datos ya escritos, y lo pone en una tabla.
#
# Solo se miran las rutas que el flujo DECLARA en su descriptor. Si flujoA declara la entrada A,
# la entrada A0 y la salida B, se miden esas tres y ninguna mas: no se recorre el lakehouse ni
# se buscan datos donde el flujo no dijo que los hubiera. Cada ruta se mide UNA vez aunque la
# compartan varios flujos, que es lo normal cuando la salida de uno es la entrada del siguiente.
ES_TRAS_CORRIDA, _HUELLAS_CACHE = {}, {}


def _ruta_corta(r, ancho=46):
    """La ruta sin el prefijo del workspace, que es igual en todas y no distingue nada."""
    t = str(r.get("ruta") or r.get("abfss") or "")
    t = t.replace("\\", "/").strip("/")
    return t if len(t) <= ancho else "…" + t[-(ancho - 1):]


def _miles(n):
    return f"{n:,}".replace(",", ".") if isinstance(n, int) else "—"


def _tabla(cabeceras, filas):
    """Una tabla de texto con las columnas al ancho del contenido."""
    if not filas:
        return ["   (sin filas)"]
    cols = list(zip(*([cabeceras] + [[str(x) for x in f] for f in filas])))
    ancho = [min(max(len(str(x)) for x in c), 60) for c in cols]
    # Los numeros a la derecha; el resto a la izquierda.
    der = {i for i in range(len(cabeceras))
           if all(str(f[i]).replace(".", "").replace("—", "").isdigit() for f in filas)}
    def linea(vals):
        return "   " + "  ".join(
            (str(v)[:ancho[i]].rjust(ancho[i]) if i in der else str(v)[:ancho[i]].ljust(ancho[i]))
            for i, v in enumerate(vals))
    return [linea(cabeceras), "   " + "  ".join("─" * a for a in ancho)] + [linea(f) for f in filas]


def _medir_una(r):
    """La huella de una ruta declarada. Cacheada: la misma ruta no se mide dos veces."""
    url = r.get("abfss")
    if not url:
        return {"estado": "EXTERNO", "filas": None, "columnas": None,
                "motivo": (f"origen {r.get('tipo')} fuera de OneLake: lo resuelve la conexion "
                           "del pipeline, no se puede medir desde aqui")}
    if url in _HUELLAS_CACHE:
        return _HUELLAS_CACHE[url]
    try:
        h = huella(url)
    except Exception as e:
        h = None
        MOTIVOS[url] = f"{type(e).__name__}: {str(e)[:90]}"
    if h is None:
        d = {"estado": "NO_LEGIBLE", "filas": None, "columnas": None,
             "motivo": MOTIVOS.get(url, "no existe o sin acceso")}
    elif not h.get("filas"):
        d = {"estado": "VACIA", "filas": 0, "columnas": h.get("columnas"),
             "esquema": h.get("esquema"), "motivo": "existe pero no trae filas"}
    else:
        d = {"estado": "OK", "filas": h["filas"], "columnas": h["columnas"],
             "esquema": h.get("esquema"), "contenido": h.get("contenido"), "motivo": None}
    _HUELLAS_CACHE[url] = d
    return d


if not CORRIDAS:
    print("Sin corridas en esta sesion: no hay nada que medir despues de ejecutar.")
else:
    _resumen_global = []
    for obj in LISTOS:
        if obj["id"] not in CORRIDAS:
            continue
        en_flujo(obj["solicitado"])
        _t = time.time()
        et = obj["nombre"]
        declaradas = IO.get(obj["id"], [])
        entradas = [r for r in declaradas if r["rol"] == "entrada"]
        salidas = [r for r in declaradas if r["rol"] == "salida"]

        _acts = (globals().get("ACTIVIDADES") or {}).get(obj["id"])
        if _acts is None:
            _acts = actividades(obj["ws"], (CORRIDAS[obj["id"]] or {}).get("id"))
        escribio = hubo_escritura(_acts)

        print()
        print("╭" + "─" * 92 + "╮")
        print(f"│ {et[:66]:<66} [{str(obj['tipo'])[:12]:<12}] {'ESCRIBIO' if escribio else '—':>9} │")
        print("╰" + "─" * 92 + "╯")
        if not declaradas:
            print("   el descriptor no declara ninguna entrada ni salida: nada que medir")
            ES_TRAS_CORRIDA[obj["solicitado"]] = {"entradas": [], "salidas": [],
                                                  "escribio": escribio, "medido": False}
            continue

        filas_tabla, detalle = [], {"entradas": [], "salidas": [], "escribio": escribio,
                                    "medido": True}
        for r in entradas + salidas:
            d = _medir_una(r)
            filas_tabla.append([r["rol"].upper(), str(r.get("tipo") or "")[:14],
                                str(r.get("lakehouse") or "—")[:18], _ruta_corta(r),
                                _miles(d["filas"]),
                                d["columnas"] if d["columnas"] is not None else "—",
                                d["estado"]])
            detalle[r["rol"] + "s"].append(dict(d, ruta=r.get("ruta"), abfss=r.get("abfss"),
                                                tipo=r.get("tipo"), lakehouse=r.get("lakehouse")))
        for l in _tabla(["ROL", "TIPO", "ARTEFACTO", "RUTA", "FILAS", "COLS", "ESTADO"],
                        filas_tabla):
            print(l)

        # Lo que la tabla deja ver, dicho como hallazgo para que entre en el informe.
        for r, d in zip(entradas, detalle["entradas"]):
            if d["estado"] == "NO_LEGIBLE":
                anotar("ADVERTENCIA", f"{et} / entrada", f"{r['ruta']}: {d['motivo']}",
                       prueba="entrada_tras_corrida", esperado="legible", obtenido=d["estado"])
            elif d["estado"] == "VACIA":
                anotar("ADVERTENCIA", f"{et} / entrada",
                       f"la entrada quedo vacia tras la corrida: {r['ruta']}",
                       prueba="entrada_tras_corrida", esperado="> 0 filas", obtenido=0)
        for r, d in zip(salidas, detalle["salidas"]):
            if d["estado"] == "NO_LEGIBLE":
                # v5.2 · tres casos distintos que antes se anotaban igual, como bloqueo.
                if r.get("intermedia"):
                    anotar("ADVERTENCIA", f"{et} / salida",
                           f"{r['ruta']}: salida intermedia en staging; el flujo la consume "
                           "y la reescribe particionada, asi que no permanece. Verificar el "
                           "destino final de la actividad de particion",
                           prueba="salida_tras_corrida", esperado="intermedia, no permanente",
                           obtenido=d["estado"])
                    continue
                existe, n_arch, _bytes = existe_en_onelake(r["abfss"])
                if existe is True:
                    anotar("ADVERTENCIA", f"{et} / salida",
                           f"{r['ruta']}: la carpeta existe en OneLake ({n_arch} archivo(s), "
                           f"{_bytes / 1048576:,.1f} MB) pero el motor no la pudo perfilar: "
                           + str(d["motivo"]),
                           prueba="salida_tras_corrida", esperado="legible", obtenido=d["estado"])
                elif existe is False:
                    anotar("BLOQUEANTE", f"{et} / salida",
                           f"la salida declarada no existe en OneLake tras la corrida: "
                           f"{r['ruta']} ({d['motivo']})",
                           prueba="salida_tras_corrida", esperado="legible", obtenido=d["estado"])
                else:
                    anotar("ADVERTENCIA", f"{et} / salida",
                           f"{r['ruta']}: no se pudo confirmar si existe ({d['motivo']})",
                           prueba="salida_tras_corrida", esperado="legible", obtenido=d["estado"])
            elif d["estado"] == "VACIA":
                anotar("ADVERTENCIA", f"{et} / salida",
                       f"la salida existe pero quedo sin filas: {r['ruta']}",
                       prueba="salida_tras_corrida", esperado="> 0 filas", obtenido=0)

        _f_ent = [d["filas"] for d in detalle["entradas"] if isinstance(d["filas"], int)]
        _f_sal = [d["filas"] for d in detalle["salidas"] if isinstance(d["filas"], int)]
        detalle["filas_entrada_total"] = sum(_f_ent) if _f_ent else None
        detalle["filas_salida_total"] = sum(_f_sal) if _f_sal else None
        # Un flujo que lee y no escribe nada es el fallo que mas caro sale y menos ruido hace.
        if _f_ent and sum(_f_ent) > 0 and _f_sal and sum(_f_sal) == 0:
            anotar("BLOQUEANTE", et,
                   f"leyo {sum(_f_ent):,} fila(s) de sus entradas y sus salidas quedaron en 0",
                   prueba="entrada_sin_salida", esperado="salida > 0",
                   obtenido={"entrada": sum(_f_ent), "salida": 0})
        ES_TRAS_CORRIDA[obj["solicitado"]] = detalle
        _resumen_global.append([et[:34], len(entradas), len(salidas),
                                _miles(detalle["filas_entrada_total"]),
                                _miles(detalle["filas_salida_total"]),
                                "si" if escribio else "no"])
        cronometrar(obj["solicitado"], "entradas_y_salidas", time.time() - _t)

    if _resumen_global:
        print("\n" + "═" * 96)
        print("  RESUMEN · entradas y salidas declaradas, medidas tras la corrida")
        print("═" * 96)
        for l in _tabla(["FLUJO", "ENTRADAS", "SALIDAS", "FILAS ENT", "FILAS SAL", "ESCRIBIO"],
                        _resumen_global):
            print(l)
        print(f"\n   {len(_HUELLAS_CACHE)} ruta(s) distinta(s) medida(s) · solo las declaradas "
              f"por los descriptores")

### Paso 8.2 · Grafo de dependencias entre flujos

Un interruptor al comienzo de la celda: `GRAFO = True` la ejecuta, `GRAFO = False` la salta. Cruza las rutas ya declaradas en `IO` (sin leer OneLake) y usa lo medido en 8.1 para decir si un flujo dependio de otro que dejo la salida vacia o ilegible.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 8.2 · Grafo de dependencias entre flujos
# ════════════════════════════════════════════════════════════════════════════
# Un solo interruptor. Con GRAFO = False esta celda no hace nada y el informe lo dice.
GRAFO = True

# Quien produce lo que otro consume: la salida de un flujo es (o contiene, o esta contenida
# en) la entrada de otro. Se calcula cruzando las rutas que ya estan en IO —sin leer nada de
# OneLake ni de la API—, asi que cuesta milisegundos. La existencia real de esas rutas ya la
# midio el paso 8.1 con datos; aqui se usa ese resultado para decir si un flujo dependio de
# otro que dejo la salida vacia o ilegible.
import re as _re51
import graphlib as _graphlib


def clave_ruta(reg):
    """Clave comparable de una entrada/salida: la ruta sin esquema ni host, en minusculas,
    sin barras repetidas ni barra final. Vale para abfss://, /lakehouse/default/Files/... y
    rutas relativas. Dos registros con la misma clave (o uno prefijo del otro) son la misma
    cosa fisica."""
    ruta = str(reg.get("abfss") or reg.get("ruta") or "").strip()
    if not ruta:
        return ""
    ruta = ruta.lower()
    ruta = _re51.sub(r"^[a-z]+://", "", ruta)                       # abfss://, https://
    ruta = _re51.sub(r"^[^/]*@onelake\.dfs\.fabric\.microsoft\.com/", "", ruta)
    ruta = _re51.sub(r"^onelake\.dfs\.fabric\.microsoft\.com/", "", ruta)
    ruta = ruta.replace("/lakehouse/default/", "")
    ruta = _re51.sub(r"/+", "/", ruta).strip("/")
    return ruta


def grafo_es(io_por_flujo):
    """Aristas productor -> consumidor: la salida de uno es (o contiene, o esta contenida en)
    la entrada de otro. Devuelve nodos, aristas, upstream/downstream por flujo, orden
    topologico y ciclos."""
    salidas, entradas = {}, {}
    for flujo, regs in io_por_flujo.items():
        for r in regs or []:
            k = clave_ruta(r)
            if not k:
                continue
            (salidas if r.get("rol") == "salida" else entradas).setdefault(k, set()).add(flujo)
    aristas = []
    for ks, prods in salidas.items():
        for ke, cons in entradas.items():
            if ks == ke or ke.startswith(ks + "/") or ks.startswith(ke + "/"):
                for a in prods:
                    for b in cons:
                        if a != b:
                            aristas.append({"productor": a, "consumidor": b, "ruta": ks,
                                            "clase": "exacta" if ks == ke else "prefijo"})
    vistas, unicas = set(), []
    for x in aristas:
        t = (x["productor"], x["consumidor"], x["ruta"])
        if t not in vistas:
            vistas.add(t)
            unicas.append(x)
    up, down = {}, {}
    for x in unicas:
        up.setdefault(x["consumidor"], set()).add(x["productor"])
        down.setdefault(x["productor"], set()).add(x["consumidor"])
    nodos = sorted(io_por_flujo)
    ciclos = []
    try:
        ts = _graphlib.TopologicalSorter({n: set(up.get(n, ())) for n in nodos})
        orden = list(ts.static_order())
    except _graphlib.CycleError as e:
        orden = []
        ciclos = [list(e.args[1])] if len(e.args) > 1 else [["(ciclo)"]]
    return {"nodos": nodos, "aristas": unicas,
            "upstream": {n: sorted(up.get(n, ())) for n in nodos},
            "downstream": {n: sorted(down.get(n, ())) for n in nodos},
            "orden_topologico": orden, "ciclos": ciclos,
            "cobertura": {"flujos": len(nodos),
                          "salidas_con_ruta": sum(len(v) for v in salidas.values()),
                          "entradas_con_ruta": sum(len(v) for v in entradas.values()),
                          "aristas": len(unicas)},
            "criterio": ("una salida de A es entrada de B si la ruta normalizada coincide o "
                         "una es prefijo de la otra (carpeta vs archivo o particion dentro)")}

DEPENDENCIAS_POR_FLUJO, EXISTENCIA_POR_FLUJO, RED_DEPENDENCIAS = {}, {}, None
if not GRAFO:
    print("GRAFO = False: no se calcula el grafo de dependencias.")
else:
    RED_DEPENDENCIAS = grafo_es({o["solicitado"]: IO.get(o["id"], []) for o in OBJETOS})
    _es = globals().get("ES_TRAS_CORRIDA") or {}
    for obj in OBJETOS:
        n = obj["solicitado"]
        DEPENDENCIAS_POR_FLUJO[n] = {
            "upstream": RED_DEPENDENCIAS["upstream"].get(n, []),
            "downstream": RED_DEPENDENCIAS["downstream"].get(n, []),
            "aristas_entrada": [x for x in RED_DEPENDENCIAS["aristas"] if x["consumidor"] == n],
            "aristas_salida": [x for x in RED_DEPENDENCIAS["aristas"] if x["productor"] == n],
            "en_ciclo": any(n in c for c in RED_DEPENDENCIAS["ciclos"]),
        }
        # La existencia sale de lo MEDIDO en 8.1, no de una consulta aparte.
        m = _es.get(n) or {}
        EXISTENCIA_POR_FLUJO[n] = {
            "verificada": bool(m.get("medido")),
            "entradas_faltantes": [d.get("ruta") for d in m.get("entradas") or []
                                   if d.get("estado") == "NO_LEGIBLE"],
            "salidas_faltantes": [d.get("ruta") for d in m.get("salidas") or []
                                  if d.get("estado") == "NO_LEGIBLE"],
            "sin_resolver": [d.get("ruta") for d in (m.get("entradas") or []) + (m.get("salidas") or [])
                             if d.get("estado") == "EXTERNO"],
        }
    print(f"grafo: {RED_DEPENDENCIAS['cobertura']['flujos']} flujo(s) · "
          f"{RED_DEPENDENCIAS['cobertura']['aristas']} dependencia(s)"
          + (f" · {len(RED_DEPENDENCIAS['ciclos'])} CICLO(s)" if RED_DEPENDENCIAS["ciclos"] else ""))
    if RED_DEPENDENCIAS["orden_topologico"]:
        print("   orden de ejecucion sugerido: " + "  ->  ".join(RED_DEPENDENCIAS["orden_topologico"]))
    for n, d in DEPENDENCIAS_POR_FLUJO.items():
        if d["upstream"] or d["downstream"]:
            print(f"   {n}: lee de {d['upstream'] or '—'} · alimenta a {d['downstream'] or '—'}")
        if d["en_ciclo"]:
            anotar("ADVERTENCIA", n, "el flujo esta en un ciclo de dependencias",
                   prueba="dependencias", esperado="grafo aciclico", obtenido="ciclo")
        for u in d["upstream"]:
            _mu = _es.get(u) or {}
            _malas = [x.get("ruta") for x in _mu.get("salidas") or []
                      if x.get("estado") in ("NO_LEGIBLE", "VACIA")]
            if _malas:
                anotar("ADVERTENCIA", n,
                       f"depende de {u}, cuyas salidas quedaron vacias o ilegibles tras la "
                       f"corrida: {', '.join(str(x).split('/')[-1] for x in _malas[:3])}",
                       prueba="upstream_listo", esperado="salidas del upstream con datos",
                       obtenido="vacias/ilegibles")

## Paso 9 · Verificar el resultado

Vuelve a medir cada salida y la contrasta contra la linea base: si el esquema
se movio, si quedo vacia, si el contenido efectivamente cambio.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 9 · Verificar el resultado
# ════════════════════════════════════════════════════════════════════════════
AHORA, ACTIVIDADES = {}, {}
if not CORRIDAS:
    print("Sin corridas en esta sesion: no hay nada que comparar.")
else:
    for obj in LISTOS:
        if obj["id"] not in CORRIDAS:
            continue
        en_flujo(obj["solicitado"])
        et = obj["nombre"]
        print(f"\n{et}")
        _t = time.time()
        salidas = [r for r in IO.get(obj["id"], []) if r["rol"] == "salida"]
        if not salidas:
            anotar("OMITIDO", et, "sin salidas declaradas: nada que verificar",
                   prueba="salidas_declaradas", esperado=">= 1 salida", obtenido=0)

        # El detalle de la corrida va primero: dice si el flujo escribio, y eso
        # cambia como hay que leer una salida de contenido identico.
        corrida = CORRIDAS[obj["id"]] or {}
        acts = actividades(obj["ws"], corrida.get("id"))
        escribio = hubo_escritura(acts)

        ahora = {}
        for r in salidas:
            ahora[r["ruta"]] = huella(r["abfss"])
            comparar(BASE[obj["id"]].get(r["ruta"]), ahora[r["ruta"]],
                     f"{et} / {r['ruta'].split('/')[-1]}", escribio)

        perfiles = [h for h in ahora.values() if h]
        revisar_copia(acts, et, perfiles[0]["filas"] if len(perfiles) == 1 else None)
        AHORA[obj["id"]] = ahora
        ACTIVIDADES[obj["id"]] = acts
        cronometrar(obj["solicitado"], "verificacion", time.time() - _t)

## Paso 10 · Informe

Dos salidas. El **reporte general de la corrida** queda en la carpeta del dia y
resume los N flujos validados juntos. Los **veredictos por flujo** van a una
carpeta paralela, un archivo por flujo con el id de la corrida, y ahi si esta el
detalle: que se probo, que se esperaba, que se obtuvo y cuanto tardo.

```
resultados/flujos/<fecha>/corrida_<id>_<N>flujos.json
resultados/flujos/veredictos/<flujo>__<id>.json
```

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 10 · Informe
# ════════════════════════════════════════════════════════════════════════════
import os
from collections import Counter

# Identificador de esta corrida de validacion: agrupa todos los flujos que se
# validaron juntos, y es lo que enlaza el informe general con los individuales.
CORRIDA = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
FECHA = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# REPORTES_V2: corridas/ y flujos/<flujo>/. El veredicto por flujo se escribe UNA vez.
DIR_CORRIDAS = f"{RUTA_RESULTADOS}/corridas"
DIR_VEREDICTOS = f"{RUTA_RESULTADOS}/flujos"
DIR_DEPENDENCIAS = f"{RUTA_RESULTADOS}/dependencias"
FECHA_V3 = datetime.now(timezone.utc).strftime("%Y-%m-%d")


def _seguro_v3(nombre):
    return "".join(c if c.isalnum() or c in "-_" else "-" for c in str(nombre))[:80] or "x"


def siguiente_version_v3(carpeta, prefijo, clave) -> int:
    """La version que toca para <prefijo>_<clave>_<fecha>_v<N>.json en esa carpeta: la mayor
    que ya exista, +1. Cuenta por flujo a lo largo de toda su historia, no por dia."""
    import glob as _glob
    patron = os.path.join(carpeta, f"{prefijo}_{_seguro_v3(clave)}_*_v*.json")
    vistos = [0]
    for ruta in _glob.glob(patron):
        m = re.search(r"_v(\d+)\.json$", os.path.basename(ruta))
        if m:
            vistos.append(int(m.group(1)))
    return max(vistos) + 1


def nombre_v3(carpeta, prefijo, clave) -> str:
    return (f"{prefijo}_{_seguro_v3(clave)}_{FECHA_V3}_"
            f"v{siguiente_version_v3(carpeta, prefijo, clave)}.json")


def _detalle_io(r):
    # Las dos caras de una ruta: la plantilla y el valor real.
    return {"rol": r["rol"], "tipo": r["tipo"], "actividad": r["actividad"],
            "plantilla_carpeta": r["plantilla_carpeta"],
            "plantilla_archivo": r["plantilla_archivo"],
            "carpeta_real": r["carpeta"], "archivo_real": r["archivo"],
            "ruta_completa": r["ruta"], "variables": r["variables"],
            "lakehouse": r["lakehouse"], "abfss": r["abfss"],
            "existe": r.get("existe")}


def _metricas(acts):
    # Filas y bytes que reporta la propia corrida, por actividad.
    fuera = []
    for a in acts or ():
        o = a.get("output") or {}
        fuera.append({"actividad": a.get("activityName"),
                      "estado": a.get("status"),
                      "duracion_ms": a.get("durationInMs"),
                      "filas_leidas": o.get("rowsRead"),
                      "filas_copiadas": o.get("rowsCopied"),
                      "archivos_leidos": o.get("filesRead"),
                      "archivos_escritos": o.get("filesWritten"),
                      "bytes_leidos": o.get("dataRead"),
                      "bytes_escritos": o.get("dataWritten")})
    return fuera


def _escribir(carpeta, nombre, datos):
    # Se relee lo escrito: que open() no falle no prueba que el archivo quedo.
    ruta = f"{carpeta}/{nombre}"
    try:
        os.makedirs(carpeta, exist_ok=True)
        with open(ruta, "w", encoding="utf-8") as f:
            json.dump(datos, f, ensure_ascii=False, indent=2)
        with open(ruta, encoding="utf-8") as f:
            json.load(f)
        return ruta, os.path.getsize(ruta), ""
    except Exception as e:
        return ruta, 0, f"{type(e).__name__}: {e}"


# ---------------------------------------------------------------- por flujo
POR_FLUJO = {}
for nombre in FLUJOS:
    obj = next((o for o in OBJETOS if o.get("solicitado") == nombre), None)
    hall = de_flujo(nombre)
    corrida_fabric = CORRIDAS.get(obj["id"]) if obj else None
    # Con ejecucion, el perfil de despues; en pasada de lectura, la linea base.
    # Asi el informe dice cuantas filas tiene el destino en los dos casos.
    medidas = (AHORA.get(obj["id"]) or {}) if obj else {}
    momento = "despues_de_ejecutar"
    if not any(medidas.values()):
        medidas = (BASE.get(obj["id"]) or {}) if obj else {}
        momento = "antes_de_ejecutar"
    salidas_ahora = medidas
    conteo_f = Counter(h["nivel"] for h in hall)

    POR_FLUJO[nombre] = {
        "corrida_validacion": CORRIDA,
        "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "flujo_solicitado": nombre,
        "flujo_resuelto": obj["nombre"] if obj else None,
        "tipo": obj["tipo"] if obj else None,
        "workspace": obj["ws"] if obj else None,
        "ambiente": AMBIENTE,
        "se_ejecuto": bool(corrida_fabric),

        "resultado": resultado(hall),
        "veredicto": veredicto(hall) if hall else "OMITIDO",
        "pasa_ejecucion": bool(corrida_fabric) and
                          str((corrida_fabric or {}).get("status", "")).lower() == "completed",
        "pasa_validacion": pasa(hall) and bool(hall),
        "resumen": dict(conteo_f),

        "pruebas": [{"prueba": h["prueba"] or "(sin nombre)", "nivel": h["nivel"],
                     "objeto": h["objeto"], "esperado": h["esperado"],
                     "obtenido": h["obtenido"], "detalle": h["mensaje"]}
                    for h in hall],

        "entradas": [_detalle_io(r) for r in (IO.get(obj["id"], []) if obj else [])
                     if r["rol"] == "entrada"],
        "salidas": [_detalle_io(r) for r in (IO.get(obj["id"], []) if obj else [])
                    if r["rol"] == "salida"],

        "datos_salida": {ruta: h for ruta, h in salidas_ahora.items()},
        "datos_salida_momento": momento,
        "metricas_corrida": _metricas(ACTIVIDADES.get(obj["id"]) if obj else None),

        "corrida_fabric": {k: (corrida_fabric or {}).get(k)
                           for k in ("id", "status", "startTimeUtc", "endTimeUtc")}
                          if corrida_fabric else None,
        "duracion_corrida": duracion(corrida_fabric) if corrida_fabric else None,
        # En segundos y con su procedencia, para que el numero se pueda discutir
        # y no dependa de que la corrida trajera endTimeUtc.
        "duracion_segundos": duracion_segundos(corrida_fabric),
        "duracion_texto": texto_duracion(corrida_fabric),
        "duracion_origen": duracion_detallada(corrida_fabric)["origen"],
        "duracion_parcial": duracion_detallada(corrida_fabric)["parcial"],
        "duracion_sospechosa": bool((duracion_segundos(corrida_fabric) or 0)
                                    > TOPE_SOSPECHA),
        "tiempos_segundos": TIEMPOS.get(nombre, {}),
        "tiempo_total_segundos": round(sum(TIEMPOS.get(nombre, {}).values()), 2),
        # Paso 5.1: de quien depende, a quien alimenta, y que existe.
        "dependencias": (globals().get("DEPENDENCIAS_POR_FLUJO") or {}).get(nombre, {
            "upstream": [], "downstream": [], "aristas_entrada": [], "aristas_salida": [],
            "en_ciclo": False, "nota": "el paso 8.2 no se ejecuto (GRAFO = False o sin corridas)"}),
        "existencia": (globals().get("EXISTENCIA_POR_FLUJO") or {}).get(nombre, {
            "verificada": False, "entradas_faltantes": [], "salidas_faltantes": [],
            "sin_resolver": []}),
        # Paso 8.1: lo que traian de verdad las entradas y las salidas DECLARADAS por este
        # flujo, medido despues de la corrida. Si el paso no se ejecuto, queda dicho.
        "entradas_y_salidas_tras_corrida": (globals().get("ES_TRAS_CORRIDA") or {}).get(nombre, {
            "medido": False, "entradas": [], "salidas": [], "escribio": None,
            "filas_entrada_total": None, "filas_salida_total": None,
            "nota": "el paso 8.1 no se ejecuto en esta sesion"}),
        "n_entradas": sum(1 for r in (IO.get(obj["id"], []) if obj else []) if r["rol"] == "entrada"),
        "n_salidas_declaradas": sum(1 for r in (IO.get(obj["id"], []) if obj else []) if r["rol"] == "salida"),
    }

# ---------------------------------------------------------------- consolidado
conteo = Counter(h["nivel"] for h in HALLAZGOS)
final = veredicto()
pasaron = [n for n, f in POR_FLUJO.items() if f["pasa_validacion"]]
ejecutaron = [n for n, f in POR_FLUJO.items() if f["pasa_ejecucion"]]
bloqueados = [n for n, f in POR_FLUJO.items() if f["veredicto"] == "BLOQUEANTE"]
sin_resolver = [n for n, f in POR_FLUJO.items() if f["flujo_resuelto"] is None]

consolidado = {
    "version_validador": VERSION,
    "corrida_validacion": CORRIDA,
    "fecha": FECHA,
    "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "ambiente": AMBIENTE,
    "ejecutar": EJECUTAR,
    "veredicto_global": final,
    "resultado_global": resultado(HALLAZGOS),
    "totales": {
        "solicitados": len(FLUJOS),
        "resueltos": len(OBJETOS),
        "sin_resolver": len(sin_resolver),
        "aptos_para_ejecutar": len(LISTOS),
        "ejecutados_ok": len(ejecutaron),
        "pasan_validacion": len(pasaron),
        "bloqueados": len(bloqueados),
        "por_resultado": dict(Counter(f["resultado"] for f in POR_FLUJO.values())),
    },
    "resumen_hallazgos": dict(conteo),
    "filas_totales_destino": sum(
        h["filas"] for f in POR_FLUJO.values()
        for h in f["datos_salida"].values() if h),
    "flujos": [{
        "flujo": n,
        "resultado": f["resultado"],
        "resuelto": f["flujo_resuelto"],
        "workspace": f["workspace"],
        "veredicto": f["veredicto"],
        "pasa_ejecucion": f["pasa_ejecucion"],
        "pasa_validacion": f["pasa_validacion"],
        "corrida_fabric": (f["corrida_fabric"] or {}).get("id"),
        "duracion": f["duracion_corrida"],
        "duracion_segundos": f.get("duracion_segundos"),
        # Suma, no la primera: un pipeline con varias copias mueve varias
        # tablas y quedarse con una sola subrepresenta el volumen.
        "n_salidas": len(f["datos_salida"]),
        "n_entradas": f.get("n_entradas"),
        "depende_de": (f.get("dependencias") or {}).get("upstream", []),
        "alimenta_a": (f.get("dependencias") or {}).get("downstream", []),
        "entradas_faltantes": len((f.get("existencia") or {}).get("entradas_faltantes", [])),
        "tiempo_total_segundos": f.get("tiempo_total_segundos"),
        "filas_destino": (sum(h["filas"] for h in f["datos_salida"].values() if h)
                          if any(f["datos_salida"].values()) else None),
        "resumen": f["resumen"],
    } for n, f in POR_FLUJO.items()],
    "sin_resolver": sin_resolver,
    "bloqueados": bloqueados,
}

# ---------------------------------------------------------------- en pantalla
print("=" * 78)
print(f"CORRIDA {CORRIDA}   ambiente {AMBIENTE}   "
      f"resultado: {resultado(HALLAZGOS)}   (peor nivel: {final})")
t = consolidado["totales"]
print(f"  solicitados {t['solicitados']}   resueltos {t['resueltos']}   "
      f"aptos {t['aptos_para_ejecutar']}   ejecutados ok {t['ejecutados_ok']}   "
      f"pasan {t['pasan_validacion']}   bloqueados {t['bloqueados']}")
print(f"  filas en destino: {consolidado['filas_totales_destino']:,}")
print("  " + "   ".join(f"{k}: {v}" for k, v in sorted(t["por_resultado"].items())))
print("=" * 78)
print(f"{'FLUJO':40} {'RESULTADO':19} {'EJEC':5} {'FILAS':>11} {'DUR':>6}  OK/OMI/ADV/BLQ")
for f in consolidado["flujos"]:
    r = f["resumen"]
    print(f"{f['flujo'][:40]:40} {f['resultado']:19} "
          f"{'si' if f['pasa_ejecucion'] else '-':5} "
          f"{(f['filas_destino'] if f['filas_destino'] is not None else '-'):>11} "
          f"{(f['duracion'] or _hms(f.get('duracion_segundos')) or '-'):>6}  "
          f"{r.get('OK', 0)}/{r.get('OMITIDO', 0)}/"
          f"{r.get('ADVERTENCIA', 0)}/{r.get('BLOQUEANTE', 0)}"
          + (f"   ({f['n_salidas']} salidas)" if f["n_salidas"] > 1 else ""))

# Que quedo sin verificar, agrupado: es lo unico que separa PASA de
# PASA_CON_OMISIONES, y conviene verlo junto y no flujo por flujo.
omitidas = Counter(h["prueba"] for h in HALLAZGOS if h["nivel"] == "OMITIDO")
if omitidas:
    print("\nsin verificar (no reprueban, pero no se comprobaron):")
    for prueba, veces in omitidas.most_common():
        print(f"   {prueba:28} {veces} vez(ces)")

for nivel in ("BLOQUEANTE", "ADVERTENCIA"):
    filas = [h for h in HALLAZGOS if h["nivel"] == nivel]
    if not filas:
        continue
    print(f"\n{nivel}  ({len(filas)})")
    for h in filas:
        print(f"   {h['objeto']}")
        print(f"      {h['mensaje']}")

# ---------------------------------------------------------------- a disco
# El consolidado por corrida ya no es la unidad: se conserva como indice de la corrida.
ruta_gen, tam_gen, err_gen = _escribir(
    DIR_CORRIDAS, nombre_v3(DIR_CORRIDAS, "corrida", f"{len(FLUJOS)}flujos"), consolidado)
print(f"\nreporte general : {ruta_gen}" + (f"  ({tam_gen} bytes)" if not err_gen
                                          else f"  FALLO {err_gen}"))

escritos, fallidos = 0, []
for nombre, datos in POR_FLUJO.items():
    seguro = "".join(c if c.isalnum() or c in "-_" else "-" for c in nombre)[:80]
    # flujos/<flujo>/<CORRIDA>.json: una carpeta por flujo, un archivo por corrida
    _carp = f"{DIR_VEREDICTOS}/{seguro}"
    _, tam, err = _escribir(_carp, nombre_v3(_carp, "analitica", nombre), datos)
    if err:
        fallidos.append((nombre, err))
    else:
        escritos += 1
print(f"veredictos      : {DIR_VEREDICTOS}/<flujo>/{CORRIDA}.json  ({escritos} de {len(POR_FLUJO)})")
for nombre, err in fallidos:
    print(f"   FALLO {nombre}: {err}")

if err_gen or fallidos:
    print("\nla escritura fallo; el consolidado esta en la variable `consolidado` "
          "y el detalle en `POR_FLUJO`")
    print("\n--- consolidado (copiable) ---")
    print(json.dumps(consolidado, ensure_ascii=False, indent=2))

## Paso 11 · Analítica — revisión profunda

Solo para flujos de analítica; los de ingesta se saltan y el paso lo dice.

Un flujo de analítica no copia: transforma. Verificar que «escribió algo» no
basta, así que estos pasos abren lo que escribió y lo contrastan con lo que
leyó.

| Paso | Qué hace |
|---|---|
| 11.1 | Lectura del pie de los parquet |
| 11.2 | Medición de entradas y salidas |
| 11.3 | Veredictos |
| 11.4 | La revisión: variables, ambiente, medición, salida y transformación |

Los pasos 11.1 a 11.3 solo definen funciones. El que produce el informe es
el 11.4.

### Paso 11.1 · Lectura del pie de los parquet

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 11.1 · Analítica · lectura del pie de los parquet
# ════════════════════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════════════════
#  ANALITICA · lectura profunda de la salida
#
#  Un flujo de analitica no copia: transforma. Verificar que "escribio algo" no
#  basta; hay que abrir lo que escribio y contrastarlo con lo que leyo.
#
#  Estas celdas anaden al validador:
#    · conteo y esquema reales de la tabla de salida, leyendo el registro Delta
#    · correspondencia entre las filas que entran y las que salen
#    · revision de cada variable del flujo, una por una
#    · comprobacion de que el apuntamiento no se escape del ambiente
# ═══════════════════════════════════════════════════════════════════════════
import struct

# Familias de workspace por ambiente. Un flujo de QA que apunte a un lakehouse
# de desarrollo o de produccion es un defecto de despliegue, no un detalle.
SUFIJO_AMBIENTE = {"qa": "-qa", "dev": "-dev", "prod": "-prod"}


def _trozo(ws, ruta, desde, hasta):
    """Un rango de bytes de un archivo de OneLake.

    Los desplazamientos van absolutos: OneLake ignora el sufijo `bytes=-N` y
    devuelve el archivo completo, con lo que se leeria la cabecera en vez del
    pie y el conteo saldria en cero.
    """
    url = f"{DFS}/{ws}/{urllib.parse.quote(ruta)}"
    req = urllib.request.Request(url, headers={
        "Authorization": f"Bearer {_token('storage')}",
        "Range": f"bytes={desde}-{hasta}"})
    with urllib.request.urlopen(req, timeout=300) as x:
        return x.read()


def _varint(b, i):
    r = s = 0
    while True:
        x = b[i]; i += 1
        r |= (x & 0x7F) << s
        if not x & 0x80:
            return r, i
        s += 7


def _zig(n):
    return (n >> 1) ^ -(n & 1)


def _saltar(b, i, tipo):
    """Avanza sobre un valor thrift sin interpretarlo."""
    if tipo in (1, 2):
        return i
    if tipo == 3:
        return i + 1
    if tipo in (4, 5, 6):
        return _varint(b, i)[1]
    if tipo == 7:
        return i + 8
    if tipo == 8:
        n, i = _varint(b, i)
        return i + n
    if tipo in (9, 10):
        h = b[i]; i += 1
        n = h >> 4
        if n == 15:
            n, i = _varint(b, i)
        for _ in range(n):
            i = _saltar(b, i, h & 0x0F)
        return i
    if tipo == 11:
        n, i = _varint(b, i)
        if n == 0:
            return i
        kv = b[i]; i += 1
        for _ in range(n):
            i = _saltar(b, i, kv >> 4)
            i = _saltar(b, i, kv & 0x0F)
        return i
    if tipo == 12:
        while True:
            h = b[i]; i += 1
            if h == 0:
                return i
            d, t = h >> 4, h & 0x0F
            if not d:
                i = _varint(b, i)[1]
            i = _saltar(b, i, t)
    raise ValueError(f"tipo thrift {tipo}")


_TIPO_PQ = {0: "BOOLEAN", 1: "INT32", 2: "INT64", 3: "INT96", 4: "FLOAT",
            5: "DOUBLE", 6: "BYTE_ARRAY", 7: "FIXED_LEN_BYTE_ARRAY"}


def _pie_parquet(datos):
    """(filas, [(columna, tipo)]) del FileMetaData de un parquet."""
    i, campo, filas, cols = 0, 0, None, []
    while i < len(datos):
        h = datos[i]; i += 1
        if h == 0:
            break
        delta, tipo = h >> 4, h & 0x0F
        if delta:
            campo += delta
        else:
            v, i = _varint(datos, i)
            campo = _zig(v)
        if campo == 3 and tipo in (4, 5, 6):
            v, i = _varint(datos, i)
            filas = _zig(v)
            continue
        if campo == 2 and tipo in (9, 10):     # lista de SchemaElement
            h2 = datos[i]; i += 1
            n = h2 >> 4
            if n == 15:
                n, i = _varint(datos, i)
            for k in range(n):
                j = i
                elem, campo2 = {}, 0
                while True:
                    h3 = datos[j]; j += 1
                    if h3 == 0:
                        break
                    d3, t3 = h3 >> 4, h3 & 0x0F
                    if d3:
                        campo2 += d3
                    else:
                        v3, j = _varint(datos, j)
                        campo2 = _zig(v3)
                    if campo2 == 1 and t3 in (4, 5, 6):
                        v3, j = _varint(datos, j)
                        elem["tipo"] = _zig(v3)
                    elif campo2 == 4 and t3 == 8:
                        n4, j = _varint(datos, j)
                        elem["nombre"] = datos[j:j + n4].decode("utf8", "replace")
                        j += n4
                    else:
                        j = _saltar(datos, j, t3)
                i = j
                if k and elem.get("nombre"):    # el primero es la raiz
                    cols.append((elem["nombre"], _TIPO_PQ.get(elem.get("tipo"),
                                                              "grupo")))
            continue
        i = _saltar(datos, i, tipo)
    return filas, cols


def leer_pie(ws, ruta, tam):
    """Filas y columnas declaradas en el pie de un parquet."""
    if not tam or tam < 12:
        return None, []
    cola = _trozo(ws, ruta, tam - 8, tam - 1)
    if len(cola) != 8 or cola[4:8] != b"PAR1":
        return None, []
    largo = struct.unpack("<I", cola[:4])[0]
    if largo <= 0 or largo + 8 > tam:
        return None, []
    return _pie_parquet(_trozo(ws, ruta, tam - 8 - largo, tam - 9))

### Paso 11.2 · Medición de entradas y salidas

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 11.2 · Analítica · medición de entradas y salidas
# ════════════════════════════════════════════════════════════════════════════
# ── Medicion de la salida de un flujo de analitica ──────────────────────────

import threading
from concurrent.futures import ThreadPoolExecutor


def _partes_ruta(ruta, lh_ws, lh_id):
    """De un valor de bandera a (workspace, ruta dentro del lakehouse).

    Acepta tres formas: una abfss completa, `Files/...` o `Tables/...`, y un
    nombre de tabla `esquema.tabla`.
    """
    v = (ruta or "").strip()
    if v.startswith("abfss://"):
        ws, art, resto = _partes_abfss(v)
        return ws, f"{art}/{resto}"
    v = v.strip("/")
    if v.lower().startswith(("files/", "tables/")):
        return lh_ws, f"{lh_id}/{v}"
    if "." in v and "/" not in v:                 # [lakehouse.]esquema.tabla
        # Hay flujos que nombran la tabla en TRES partes, con el lakehouse
        # delante: `lh_transversal.control.migrations_history`. Partiendo solo
        # por el primer punto, el lakehouse se tomaba por esquema y la tabla se
        # buscaba en `Tables/lh_transversal/control.migrations_history`, que no
        # existe — y una tabla que si esta se reportaba como ausente, dejando el
        # veredicto de transformacion en NO_MEDIBLE.
        # En OneLake una tabla siempre cuelga de `Tables/<esquema>/<tabla>`, asi
        # que los dos ultimos segmentos son los buenos, haya o no prefijo.
        partes = [p for p in v.split(".") if p]
        return lh_ws, f"{lh_id}/Tables/" + "/".join(partes[-2:])
    return lh_ws, f"{lh_id}/Files/{v}"


def _archivos(ws, ruta):
    """[(nombre, bytes, modificado)] bajo esa ruta. None si la ruta no existe."""
    url = (f"{DFS}/{ws}?recursive=true&resource=filesystem"
           f"&directory={urllib.parse.quote(ruta)}")
    c, b = _onelake(url)
    if c == 404:
        return None
    if c != 200:
        raise FabricNoResponde(f"listar {ruta}: HTTP {c} · {str(b)[:160]}")
    return [(p["name"], int(p.get("contentLength") or 0), p.get("lastModified", ""))
            for p in (b or {}).get("paths", []) if p.get("contentLength")]


def _solo_datos(arch):
    """Parquet de datos.

    Se excluye lo que cuelga de `_delta_log` y `_metadata`: el archivo de
    estadisticas de Delta lleva EL MISMO nombre que el de datos, de modo que
    filtrar por nombre los confunde y duplica el conteo.
    """
    return [x for x in arch if x[0].endswith(".parquet")
            and "/_delta_log/" not in x[0] and "/_metadata/" not in x[0]]


def registro_delta(ws, arch):
    """Lo que declara el registro Delta, que es la verdad de la tabla.

    Delta ya lleva el conteo en `stats.numRecords` y el esquema en `metaData`.
    Leerlos evita abrir los parquet y da la cifra que la propia tabla considera
    valida.
    """
    logs = sorted(x[0] for x in arch
                  if x[0].endswith(".json") and "/_delta_log/" in x[0])
    if not logs:
        return None
    add, rem, esquema, cometido = {}, set(), None, None
    for l in logs:
        c, d = _onelake(f"{DFS}/{ws}/{urllib.parse.quote(l)}", binario=True)
        if c != 200:
            continue
        for linea in d.decode("utf8", "replace").splitlines():
            if not linea.strip():
                continue
            try:
                j = json.loads(linea)
            except Exception:
                continue
            if "add" in j:
                a = j["add"]
                st = {}
                try:
                    st = json.loads(a.get("stats") or "{}")
                except Exception:
                    pass
                add[a["path"].split("/")[-1]] = st.get("numRecords")
            if "remove" in j:
                rem.add(j["remove"]["path"].split("/")[-1])
            if "metaData" in j:
                try:
                    e = json.loads(j["metaData"].get("schemaString") or "{}")
                    esquema = [(f["name"], f.get("type"), f.get("nullable"))
                               for f in e.get("fields", [])]
                except Exception:
                    pass
            if "commitInfo" in j:
                cometido = j["commitInfo"]
    vivos = {k: v for k, v in add.items() if k not in rem}
    filas = None if any(v is None for v in vivos.values()) else sum(vivos.values())
    return {"versiones": len(logs), "archivos": len(vivos), "filas": filas,
            "esquema": esquema, "retirados": len(rem),
            "operacion": (cometido or {}).get("operation"),
            "modo": ((cometido or {}).get("operationParameters") or {}).get("mode"),
            "cometido_ms": (cometido or {}).get("timestamp")}


_SEM_RED = threading.Semaphore(max(1, int(globals().get("HILOS_RED") or 1)))


def _leer_pies(ws, lote):
    """Los pies de una lista de parquet, en paralelo y EN ORDEN.

    Devuelve una lista alineada con `lote`, asi que quien la consume la recorre igual que
    antes. El semaforo es global: aunque varios flujos midan a la vez, el numero de
    peticiones de pie en vuelo nunca pasa de HILOS_RED."""
    hilos = max(1, min(int(globals().get("HILOS_RED") or 1), len(lote) or 1))
    if hilos <= 1 or len(lote) <= 1:
        return [leer_pie(ws, n, tam) for n, tam, _m in lote]

    def _uno(par):
        n, tam = par
        with _SEM_RED:
            return leer_pie(ws, n, tam)

    salida = [None] * len(lote)
    with ThreadPoolExecutor(max_workers=hilos, thread_name_prefix="pie") as pool:
        futs = {pool.submit(_uno, (n, tam)): i
                for i, (n, tam, _m) in enumerate(lote)}
        for f in futs:
            salida[futs[f]] = f.result()
    return salida


def medir(valor, lh_ws, lh_id):
    """Todo lo medible de una entrada o de una salida."""
    ws, ruta = _partes_ruta(valor, lh_ws, lh_id)
    fuera = {"valor": valor, "ws": ws, "ruta": ruta, "existe": False}
    if not ws:
        fuera["nota"] = "no se pudo determinar el workspace del lakehouse"
        return fuera
    arch = _archivos(ws, ruta)
    if arch is None:
        return fuera
    datos = _solo_datos(arch)
    fuera.update(existe=True, archivos=len(arch), parquet=len(datos),
                 bytes=sum(x[1] for x in arch),
                 mb=round(sum(x[1] for x in arch) / 1048576, 2),
                 ultima_escritura=max((x[2] for x in arch), default=""))
    d = registro_delta(ws, arch)
    if d:
        fuera["delta"] = d
        fuera["filas"] = d["filas"]
        fuera["columnas"] = [c[0] for c in (d["esquema"] or [])]
        fuera["fuente_conteo"] = "registro Delta"
    if fuera.get("filas") is None and datos:
        # Cada leer_pie() son DOS peticiones de rango. Con 60 parquet eran 120 viajes de
        # red en fila. Se lanzan a la vez, acotados por el semaforo global.
        _lote = datos[:60]
        _pies = _leer_pies(ws, _lote)
        # Se pliega en el ORDEN ORIGINAL a proposito: asi el resultado no depende de que
        # hilo termino antes, y un fallo corta donde cortaba en serie.
        total, cols = 0, []
        for f, c in _pies:
            if f is None:
                total = None
                break
            total += f
            cols = cols or c
        fuera["filas"] = total
        fuera["columnas"] = fuera.get("columnas") or [c[0] for c in cols]
        fuera["fuente_conteo"] = "pie de los parquet"
    return fuera

### Paso 11.3 · Veredictos de analítica

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 11.3 · Analítica · veredictos
# ════════════════════════════════════════════════════════════════════════════
# ── Veredictos de un flujo de analitica ─────────────────────────────────────

def revisar_variables(io, vars_):
    """Cada bandera del flujo, con la variable que la alimenta y su estado.

    Una bandera puede estar «resuelta» y aun asi no servir: un valor de relleno
    pasa como texto valido y el flujo termina escribiendo donde no debe.
    """
    filas, malas = [], 0
    for r in io:
        valor = r.get("ruta") or ""
        motivo = rastro_sin_resolver(valor)
        variable = next(iter(r.get("variables") or {}), "")
        filas.append({"rol": r["rol"], "bandera": r["tipo"], "variable": variable,
                      "valor": valor, "problema": motivo})
        if motivo and r["rol"] != "codigo":
            malas += 1
    return {"veredicto": "RESUELTA" if not malas else "SIN_RESOLVER",
            "banderas": len(filas), "con_problema": malas, "detalle": filas}


def revisar_ambiente(io, ambiente):
    """El flujo no debe apuntar fuera de su ambiente.

    Un pipeline de QA que lee o escribe en un lakehouse de desarrollo o de
    produccion mezcla ambientes: lo que se valide aqui no dice nada de lo que
    pasara alla, y en el peor caso escribe sobre datos que no le pertenecen.
    """
    sufijo = SUFIJO_AMBIENTE.get(ambiente, f"-{ambiente}")
    filas, fuera = [], 0
    for r in io:
        if r["rol"] == "codigo":
            continue
        ws_nombre = _nombre_ws(r.get("ws_id")) or ""
        ok = bool(ws_nombre) and ws_nombre.endswith(sufijo)
        if ws_nombre and not ok:
            fuera += 1
        filas.append({"rol": r["rol"], "bandera": r["tipo"],
                      "workspace": ws_nombre or "(no resuelto)",
                      "lakehouse": r.get("lakehouse") or "",
                      "correcto": ok})
    return {"veredicto": "EN_AMBIENTE" if not fuera else "FUERA_DE_AMBIENTE",
            "ambiente": ambiente, "sufijo": sufijo, "fuera": fuera,
            "detalle": filas}


def revisar_salida(med, corrida):
    """La salida debe existir, tener datos y ser de esta corrida."""
    if not med.get("existe"):
        return {"veredicto": "AUSENTE", "nota": f"no existe {med['ruta']}"}
    if not med.get("parquet"):
        return {"veredicto": "VACIA",
                "nota": "la ruta existe pero no tiene archivos de datos"}
    v = {"veredicto": "ESCRITA", "filas": med.get("filas"),
         "columnas": len(med.get("columnas") or []), "mb": med.get("mb"),
         "archivos": med.get("parquet"),
         "fuente_conteo": med.get("fuente_conteo")}
    d = med.get("delta") or {}
    if d:
        v.update(delta=True, versiones=d["versiones"], operacion=d.get("operacion"),
                 modo=d.get("modo"))
    fin = (corrida or {}).get("endTimeUtc") or (corrida or {}).get("fin")
    if fin and med.get("ultima_escritura"):
        try:
            from email.utils import parsedate_to_datetime
            esc = parsedate_to_datetime(med["ultima_escritura"])
            # _limpiar_ts, no un replace suelto: si la API no manda zona, esta comparacion
            # reventaba en silencio dentro del except y 'escrita_por_la_corrida' no se
            # calculaba nunca.
            ter = datetime.fromisoformat(_limpiar_ts(fin))
            # margen de un minuto: el cierre del trabajo y el de los archivos no
            # son el mismo instante
            v["escrita_por_la_corrida"] = esc >= ter - timedelta(minutes=1)
            if not v["escrita_por_la_corrida"]:
                v["veredicto"] = "ANTERIOR_A_LA_CORRIDA"
                v["nota"] = ("la salida no se modifico con esta corrida: es de "
                             "antes, asi que puede ser una escritura previa")
        except Exception:
            pass
    return v


def revisar_transformacion(ent, sal):
    """Que lo que sale se corresponda con lo que entra.

    Un flujo de analitica puede legitimamente devolver menos filas que las que
    lee, y casi siempre devuelve otras columnas. Lo que no puede es salir vacio
    ni traer mas filas sin una razon.
    """
    fe = sum(e.get("filas") or 0 for e in ent if e.get("filas") is not None)
    fs = sum(s.get("filas") or 0 for s in sal if s.get("filas") is not None)
    medibles = all(e.get("filas") is not None for e in ent) and \
               all(s.get("filas") is not None for s in sal)
    ce = sorted({c for e in ent for c in (e.get("columnas") or [])})
    cs = sorted({c for s in sal for c in (s.get("columnas") or [])})
    v = {"filas_entrada": fe if medibles else None,
         "filas_salida": fs if medibles else None,
         "columnas_entrada": ce, "columnas_salida": cs,
         "conservadas": sorted(set(ce) & set(cs)),
         "descartadas": sorted(set(ce) - set(cs)),
         "derivadas": sorted(set(cs) - set(ce))}
    if not medibles:
        v["veredicto"] = "NO_MEDIBLE"
        v["nota"] = "no se pudo contar alguno de los dos lados"
        return v
    v["delta"] = fs - fe
    v["pct_conservado"] = round(100 * fs / fe, 4) if fe else None
    if fs == 0:
        v["veredicto"] = "SALIDA_VACIA"
    elif fs == fe:
        v["veredicto"] = "CONTEO_IGUAL"
    elif fs > fe:
        v["veredicto"] = "SALIDA_MAYOR"
        v["nota"] = ("la salida trae mas filas que la entrada: revisar si el "
                     "flujo cruza con otra fuente o si duplica")
    else:
        v["veredicto"] = "SALIDA_MENOR"
        v["nota"] = ("la salida trae menos filas que la entrada, lo que es "
                     "esperable en un flujo de limpieza o de filtrado")
    return v


def veredicto_analitica(cfg, amb, sal_v, trans, eje):
    """El peor de los aspectos manda."""
    if (eje or {}).get("veredicto") == "FALLA":
        return "FALLA_EN_EJECUCION"
    if cfg["veredicto"] != "RESUELTA":
        return "CONFIGURACION_INCOMPLETA"
    if amb["veredicto"] != "EN_AMBIENTE":
        return "APUNTA_FUERA_DEL_AMBIENTE"
    if any(v["veredicto"] in ("AUSENTE", "VACIA") for v in sal_v):
        return "SALIDA_NO_ESCRITA"
    if any(v["veredicto"] == "ANTERIOR_A_LA_CORRIDA" for v in sal_v):
        return "SALIDA_NO_ACTUALIZADA"
    if trans["veredicto"] in ("SALIDA_VACIA", "SALIDA_MAYOR"):
        return "REVISAR_TRANSFORMACION"
    if trans["veredicto"] == "NO_MEDIBLE":
        return "SALIDA_NO_MEDIBLE"
    return "OK"

### Paso 11.4 · Ejecutar la revisión de analítica

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 11.4 · Analítica · ejecutar la revisión
# ════════════════════════════════════════════════════════════════════════════
# ── Revision completa de los flujos de analitica del lote ───────────────────
ANALITICA = {}
print("REVISION PROFUNDA DE FLUJOS DE ANALITICA")
print("=" * 78)

def es_analitica(obj):
    """Un flujo es de analitica si su I/O salio de un Spark Job Definition.

    No se mira `obj["tipo"]`: ahi va el tipo de Fabric (DataPipeline), que es el
    mismo para ingesta y para analitica. Lo que los separa es que el pipeline de
    analitica lleva una actividad FabricSparkJobDefinition, y de ella sale una
    entrada con rol `codigo` que apunta al zip del flujo.
    """
    return any(r.get("tipo") == "flow-code-zip" for r in IO.get(obj["id"], []))


for obj in OBJETOS:
    if not es_analitica(obj):
        print(f"{obj['nombre']}: es de ingesta, no aplica esta revisión")
        continue
    en_flujo(obj["solicitado"])
    io = IO.get(obj["id"], [])
    vars_ = obj.get("_vars") or variables(AMBIENTE, obj.get("bibliotecas"))   # v9
    print(f"\n{obj['nombre']}")
    print("-" * 78)

    # 1 · variables, una por una
    cfg = revisar_variables(io, vars_)
    print(f"1 · VARIABLES   [{cfg['veredicto']}]   "
          f"{cfg['banderas']} banderas · {cfg['con_problema']} con problema")
    for f in cfg["detalle"]:
        marca = " " if not f["problema"] else "!"
        print(f"  {marca} {f['rol']:8} {f['bandera'][:34]:34} {f['valor'][:56]}")
        if f["variable"]:
            print(f"       desde  {f['variable']}")
        if f["problema"]:
            print(f"       PROBLEMA: {f['problema']}")
            anotar("BLOQUEANTE", obj["nombre"],
                   f"{f['bandera']}: {f['problema']}",
                   prueba="variable_resuelta", esperado="valor utilizable",
                   obtenido=f["valor"])

    # 2 · apuntamiento entre ambientes
    amb = revisar_ambiente(io, AMBIENTE)
    print(f"\n2 · AMBIENTE    [{amb['veredicto']}]   "
          f"esperado workspace terminado en «{amb['sufijo']}»")
    for f in amb["detalle"]:
        marca = " " if f["correcto"] else "!"
        print(f"  {marca} {f['rol']:8} {f['bandera'][:30]:30} "
              f"{f['workspace'][:36]:36} {f['lakehouse']}")
        if not f["correcto"] and f["workspace"] != "(no resuelto)":
            anotar("BLOQUEANTE", obj["nombre"],
                   f"{f['bandera']} apunta a {f['workspace']}, fuera de {AMBIENTE}",
                   prueba="apuntamiento_ambiente",
                   esperado=f"workspace de {AMBIENTE}", obtenido=f["workspace"])

    # 3 · medicion de entradas y salidas
    print("\n3 · MEDICION EN ONELAKE")
    ent, sal, ctrl = [], [], []
    # Las entradas y salidas de un flujo no dependen unas de otras: se miden a la vez.
    # Los PRINT siguen saliendo en el orden de `io`, para que el informe se lea igual.
    _a_medir = [r for r in io if r["rol"] != "codigo"]
    _nh = max(1, min(int(globals().get("HILOS_RED") or 1), len(_a_medir) or 1))
    if _nh > 1 and len(_a_medir) > 1:
        with ThreadPoolExecutor(max_workers=_nh, thread_name_prefix="medir") as _pool:
            _futs = {_pool.submit(medir, r["ruta"], r.get("ws_id"),
                                  r.get("artifact_id")): i
                     for i, r in enumerate(_a_medir)}
            _medidas = [None] * len(_a_medir)
            for _f in _futs:
                _medidas[_futs[_f]] = _f.result()
    else:
        _medidas = [medir(r["ruta"], r.get("ws_id"), r.get("artifact_id"))
                    for r in _a_medir]
    for r, m in zip(_a_medir, _medidas):
        m["bandera"] = r["tipo"]
        # El control es andamiaje: no entra en la correspondencia entrada->salida.
        # Se mide y se muestra, pero no arrastra el veredicto.
        if r["rol"] == "control":
            ctrl.append(m)
        else:
            (sal if r["rol"] == "salida" else ent).append(m)
        estado = "existe" if m.get("existe") else "NO EXISTE"
        filas = m.get("filas")
        txt = f"{filas:,}".replace(",", ".") if filas is not None else "no medible"
        print(f"  {r['rol']:8} {r['ruta'][:46]:46} {estado:10} "
              f"{m.get('mb', 0):>8} MB  {txt:>14} filas")
        if m.get("delta"):
            d = m["delta"]
            print(f"           Delta v{d['versiones'] - 1} · {d['archivos']} archivos "
                  f"· operacion {d.get('operacion')} · modo {d.get('modo')}")
        if m.get("columnas"):
            print(f"           columnas ({len(m['columnas'])}): "
                  f"{', '.join(m['columnas'][:10])}"
                  f"{' …' if len(m['columnas']) > 10 else ''}")

    # 4 · estado de la salida
    ultima = (historial(obj["ws"], obj["id"], 1) or [None])[0]
    sal_v = [revisar_salida(m, ultima) for m in sal]
    print("\n4 · SALIDA")
    for m, v in zip(sal, sal_v):
        print(f"  [{v['veredicto']}]  {m['valor']}")
        if v.get("nota"):
            print(f"      {v['nota']}")
        if "escrita_por_la_corrida" in v:
            print(f"      escrita por la ultima corrida: "
                  f"{'si' if v['escrita_por_la_corrida'] else 'NO'}")
        if v["veredicto"] in ("AUSENTE", "VACIA", "ANTERIOR_A_LA_CORRIDA"):
            anotar("BLOQUEANTE", obj["nombre"],
                   f"salida {m['valor']}: {v['veredicto']}",
                   prueba="salida_escrita", esperado="salida con datos de la corrida",
                   obtenido=v["veredicto"])

    if ctrl:
        print(f"  ({len(ctrl)} tabla(s) de control medidas aparte: no son dato del "
              f"flujo y no entran en la correspondencia)")

    # 5 · transformacion
    trans = revisar_transformacion(ent, sal)
    print(f"\n5 · TRANSFORMACION   [{trans['veredicto']}]")
    if trans.get("filas_entrada") is not None:
        print(f"  entrada {trans['filas_entrada']:,} filas  →  "
              f"salida {trans['filas_salida']:,} filas   "
              f"(delta {trans['delta']:,} · conserva {trans['pct_conservado']} %)"
              .replace(",", "."))
    print(f"  columnas conservadas ({len(trans['conservadas'])}): "
          f"{', '.join(trans['conservadas'][:12]) or 'ninguna'}")
    print(f"  columnas descartadas ({len(trans['descartadas'])}): "
          f"{', '.join(trans['descartadas'][:12]) or 'ninguna'}")
    print(f"  columnas derivadas   ({len(trans['derivadas'])}): "
          f"{', '.join(trans['derivadas'][:12]) or 'ninguna'}")
    if trans.get("nota"):
        print(f"  {trans['nota']}")
    if trans["veredicto"] in ("SALIDA_VACIA", "SALIDA_MAYOR"):
        anotar("BLOQUEANTE", obj["nombre"],
               f"transformacion: {trans['veredicto']}",
               prueba="correspondencia_entrada_salida",
               esperado="salida coherente con la entrada",
               obtenido=trans["veredicto"])

    eje = {"veredicto": "OK" if (ultima or {}).get("status") == "Completed"
           else ("FALLA" if ultima else "SIN_CORRIDA"),
           "estado": (ultima or {}).get("status"),
           "corrida": (ultima or {}).get("id"),
           "fin": (ultima or {}).get("endTimeUtc")}
    glob = veredicto_analitica(cfg, amb, sal_v, trans, eje)
    print(f"\n>>> VEREDICTO: {glob}")

    ANALITICA[obj["id"]] = {
        "flujo": obj["nombre"], "veredicto": glob, "ejecucion": eje,
        "variables": cfg, "ambiente": amb, "entradas": ent, "salidas": sal,
        "control": ctrl,
        "estado_salidas": sal_v, "transformacion": trans,
    }

print("\n" + "=" * 78)
if ANALITICA:
    from collections import Counter
    c = Counter(v["veredicto"] for v in ANALITICA.values())
    print(f"flujos de analitica revisados: {len(ANALITICA)}")
    for k, n in c.most_common():
        print(f"   {k:32} {n}")
else:
    print("no se revisó ningún flujo de analítica: ninguno de los flujos "
          "solicitados tiene actividades de Spark Job Definition")

## Paso 12 · Cierre — inventario de todo lo producido

Reúne lo que dejaron las celdas anteriores, guarda lo que faltaba por guardar y
deja un índice con la ruta de cada archivo.

Produce:

| Carpeta | Qué queda |
|---|---|
| `flujos/<fecha>/` | el consolidado de la corrida |
| `flujos/veredictos/` | el veredicto de cada flujo, plano |
| `flujos/veredictos/<fecha>/` | los mismos, archivados por fecha |
| `analitica/<fecha>/` | la revisión individual de cada flujo de analítica y la general del lote |
| `informes/<fecha>/` | el índice: cifras de la corrida y ruta de todo lo demás |

Ejecútela al final, después de la revisión de analítica.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PASO 12 · Cierre · inventario de todo lo producido
# ════════════════════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════════════════
#  CIERRE · inventario de todo lo producido
#
#  Reune lo que dejaron las celdas anteriores, guarda lo que faltaba por
#  guardar y deja un indice con la ruta de cada archivo, para armar el informe
#  sin tener que buscarlos uno por uno.
# ═══════════════════════════════════════════════════════════════════════════
import os
from collections import Counter

_ANALITICA = globals().get("ANALITICA") or {}
DIR_ANALITICA = f"{RUTA_RESULTADOS}/analitica"          # analitica/<flujo>/<CORRIDA>.json

ARCHIVOS = []          # todo lo que existe en disco, con su tamaño
PENDIENTES = []        # lo que se esperaba y no quedó


def _anotar_archivo(clase, flujo, ruta, descripcion):
    """Registra un archivo COMPROBANDO que exista: que se haya escrito antes no
    prueba que siga ahí, y un informe que promete un archivo ausente es peor
    que uno que lo omite."""
    existe = os.path.exists(ruta)
    tam = os.path.getsize(ruta) if existe else 0
    reg = {"clase": clase, "flujo": flujo, "ruta": ruta, "bytes": tam,
           "existe": existe, "contiene": descripcion}
    (ARCHIVOS if existe else PENDIENTES).append(reg)
    return reg


def _seguro(nombre):
    return "".join(c if c.isalnum() or c in "-_" else "-" for c in nombre)[:80]


def _guardar(clase, flujo, carpeta, archivo, datos, descripcion):
    ruta, tam, err = _escribir(carpeta, archivo, datos)
    if err:
        PENDIENTES.append({"clase": clase, "flujo": flujo, "ruta": ruta,
                           "bytes": 0, "existe": False,
                           "contiene": f"no se pudo escribir: {err}"})
        return None
    return _anotar_archivo(clase, flujo, ruta, descripcion)


_SELLO = {"corrida_validacion": CORRIDA, "fecha": FECHA, "ambiente": AMBIENTE,
          "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}

# ── 1 · la revisión de analítica aún no estaba en disco ─────────────────────
if _ANALITICA:
    # individual, una por flujo
    for rev in _ANALITICA.values():
        _carp_a = f"{DIR_ANALITICA}/{_seguro(rev['flujo'])}"
        _guardar("analitica", rev["flujo"], _carp_a,
                 nombre_v3(_carp_a, "analitica", rev["flujo"]),
                 {**_SELLO, **rev},
                 "revisión individual: variables, ambiente, medición de "
                 "entradas y salidas, estado de la salida y transformación")

    # general, la revisión de todos los flujos de analítica de esta corrida
    _cuenta = Counter(r["veredicto"] for r in _ANALITICA.values())
    revision_general = {
        **_SELLO,
        "flujos_revisados": len(_ANALITICA),
        "por_veredicto": dict(_cuenta),
        "veredicto_global": ("OK" if set(_cuenta) == {"OK"}
                             else sorted(_cuenta, key=lambda k: k != "OK")[-1]),
        "flujos": [{
            "flujo": r["flujo"],
            "veredicto": r["veredicto"],
            "ejecucion": (r.get("ejecucion") or {}).get("veredicto"),
            "corrida_fabric": (r.get("ejecucion") or {}).get("corrida"),
            "variables": (r.get("variables") or {}).get("veredicto"),
            "banderas": (r.get("variables") or {}).get("banderas"),
            "banderas_con_problema": (r.get("variables") or {}).get("con_problema"),
            "ambiente": (r.get("ambiente") or {}).get("veredicto"),
            "salidas": [v.get("veredicto") for v in r.get("estado_salidas", [])],
            "transformacion": (r.get("transformacion") or {}).get("veredicto"),
            "filas_entrada": (r.get("transformacion") or {}).get("filas_entrada"),
            "filas_salida": (r.get("transformacion") or {}).get("filas_salida"),
            "delta": (r.get("transformacion") or {}).get("delta"),
            "pct_conservado": (r.get("transformacion") or {}).get("pct_conservado"),
            "columnas_conservadas": (r.get("transformacion") or {}).get("conservadas"),
            "columnas_descartadas": (r.get("transformacion") or {}).get("descartadas"),
            "columnas_derivadas": (r.get("transformacion") or {}).get("derivadas"),
        } for r in _ANALITICA.values()],
        "detalle": list(_ANALITICA.values()),
    }
    _guardar("analitica_general", "(todos)", f"{DIR_ANALITICA}/_totales",
             f"revision_{CORRIDA}_{len(_ANALITICA)}flujos.json", revision_general,
             "revisión general de analítica: una línea por flujo y el detalle "
             "completo de cada uno")

# ── 2 · lo que escribieron las celdas anteriores ────────────────────────────
_anotar_archivo("corrida", "(todos)",
                f"{DIR_CORRIDAS}/corrida_{CORRIDA}_{len(FLUJOS)}flujos.json",
                "consolidado de la corrida: totales, veredicto global y una "
                "línea por flujo")
for nombre in POR_FLUJO:
    _anotar_archivo("veredicto", nombre,
                    f"{DIR_VEREDICTOS}/{_seguro(nombre)}/{CORRIDA}.json",
                    "veredicto del flujo: pruebas, entradas y salidas "
                    "declaradas, datos del destino y métricas de la corrida")

# ── 2b · el grafo de dependencias de la corrida ────────────────────────────
# En REPORTES_V2 el veredicto por flujo se escribe UNA sola vez (flujos/<flujo>/), asi que
# ya no hay copia por fecha. DIR_VEREDICTOS_FECHA se conserva por compatibilidad.
DIR_VEREDICTOS_FECHA = DIR_VEREDICTOS
# OJO: la guarda mira RED_DEPENDENCIAS, no GRAFO. GRAFO es el INTERRUPTOR del paso 8.2
# (un bool); el grafo es RED_DEPENDENCIAS. Con `if GRAFO:` y `**GRAFO` esto reventaba con
# "'bool' object is not a mapping" en cuanto el interruptor estaba en True.
# Mirar RED_DEPENDENCIAS cubre los tres casos de una vez: None si GRAFO = False, None si el
# paso 8.2 no se ejecuto, y el dict cuando de verdad hay grafo.
if isinstance(globals().get("RED_DEPENDENCIAS"), dict) and RED_DEPENDENCIAS:
    _guardar("dependencias", "(todos)", DIR_DEPENDENCIAS,
             nombre_v3(DIR_DEPENDENCIAS, "dependencias", "grafo"),
             {**_SELLO, **RED_DEPENDENCIAS,
              "flujos_solicitados": [o["solicitado"] for o in OBJETOS],
              "por_flujo": globals().get("DEPENDENCIAS_POR_FLUJO") or {},
              "existencia": globals().get("EXISTENCIA_POR_FLUJO") or {}},
             "grafo de dependencias: nodos, aristas salida->entrada, orden topologico, "
             "y existencia de cada entrada/salida")


# ── 3 · el índice, que es lo que se pega en el informe ──────────────────────
def _cifras_flujo(nombre):
    f = POR_FLUJO.get(nombre, {})
    rev = next((r for r in _ANALITICA.values() if r["flujo"] == f.get(
        "flujo_resuelto")), None)
    filas_destino = (sum(h["filas"] for h in (f.get("datos_salida") or {}).values()
                         if h) if any((f.get("datos_salida") or {}).values())
                     else None)
    d = {
        "flujo": nombre,
        "resuelto": f.get("flujo_resuelto"),
        "tipo": "analítica" if rev else "ingesta",
        "workspace": f.get("workspace"),
        "resultado": f.get("resultado"),
        "veredicto": f.get("veredicto"),
        "se_ejecuto": f.get("se_ejecuto"),
        "corrida_fabric": (f.get("corrida_fabric") or {}).get("id"),
        "duracion": f.get("duracion_corrida"),
        "filas_destino": filas_destino,
        "pruebas": f.get("resumen", {}),
        "n_entradas": len(f.get("entradas") or []),
        "n_salidas": len(f.get("salidas") or []),
    }
    if rev:
        t = rev.get("transformacion") or {}
        d["analitica"] = {
            "veredicto": rev.get("veredicto"),
            "variables": rev.get("variables", {}).get("veredicto"),
            "banderas_con_problema": rev.get("variables", {}).get("con_problema"),
            "ambiente": rev.get("ambiente", {}).get("veredicto"),
            "salidas": [v.get("veredicto") for v in rev.get("estado_salidas", [])],
            "filas_entrada": t.get("filas_entrada"),
            "filas_salida": t.get("filas_salida"),
            "delta": t.get("delta"),
            "pct_conservado": t.get("pct_conservado"),
            "columnas_conservadas": len(t.get("conservadas") or []),
            "columnas_descartadas": len(t.get("descartadas") or []),
            "columnas_derivadas": len(t.get("derivadas") or []),
            "transformacion": t.get("veredicto"),
        }
    return d


INFORME = {
    "version_validador": VERSION,
    "corrida_validacion": CORRIDA,
    "fecha": FECHA,
    "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "ambiente": AMBIENTE,
    "se_ejecutaron_los_flujos": bool(EJECUTAR),
    "veredicto_global": consolidado["veredicto_global"],
    "resultado_global": consolidado["resultado_global"],
    "totales": consolidado["totales"],
    "filas_totales_destino": consolidado["filas_totales_destino"],
    "resumen_hallazgos": consolidado["resumen_hallazgos"],
    "flujos": [_cifras_flujo(n) for n in FLUJOS],
    "archivos": ARCHIVOS,
    "archivos_faltantes": PENDIENTES,
    "carpetas": {
        "corrida": DIR_CORRIDAS,
        "veredictos": DIR_VEREDICTOS,
        "veredictos_por_fecha": DIR_VEREDICTOS_FECHA,
        "analitica": DIR_ANALITICA if _ANALITICA else None,
        "dependencias": DIR_DEPENDENCIAS,
        "informes": f"{RUTA_RESULTADOS}/informes",
        "raiz": RUTA_RESULTADOS,
    },
}

_carp_inf = f"{RUTA_RESULTADOS}/informes"
ruta_idx, tam_idx, err_idx = _escribir(
    _carp_inf, nombre_v3(_carp_inf, "informe", "corrida"), INFORME)
if not err_idx:
    _anotar_archivo("informe", "(todos)", ruta_idx,
                    "este índice: cifras de la corrida y ruta de cada archivo")


# ── 4 · en pantalla, listo para pegar en el informe ─────────────────────────
def _n(v):
    return f"{v:,}".replace(",", ".") if isinstance(v, int) else (
        "—" if v is None else str(v))


print("=" * 78)
print(f"INFORME DE LA CORRIDA {CORRIDA}")
print(f"{FECHA}   ambiente {AMBIENTE}   "
      f"{'con ejecución' if EJECUTAR else 'solo lectura, sin ejecutar'}")
print("=" * 78)

t = INFORME["totales"]
print(f"\nRESULTADO GLOBAL: {INFORME['resultado_global']}   "
      f"(peor nivel: {INFORME['veredicto_global']})")
print(f"  solicitados {t['solicitados']} · resueltos {t['resueltos']} · "
      f"aptos {t['aptos_para_ejecutar']} · ejecutados {t['ejecutados_ok']} · "
      f"pasan {t['pasan_validacion']} · bloqueados {t['bloqueados']}")
print(f"  filas en destino: {_n(INFORME['filas_totales_destino'])}")
print("  pruebas: " + " · ".join(f"{k} {v}" for k, v in
                                 sorted(INFORME["resumen_hallazgos"].items())))

print("\n" + "-" * 78)
print("POR FLUJO")
print("-" * 78)
for f in INFORME["flujos"]:
    print(f"\n{f['flujo']}   [{f['tipo']}]")
    print(f"  resuelto      {f['resuelto'] or '(no se resolvió)'}")
    print(f"  workspace     {f['workspace'] or '—'}")
    print(f"  resultado     {f['resultado']}   ·   veredicto {f['veredicto']}")
    if f["corrida_fabric"]:
        print(f"  corrida       {f['corrida_fabric']}   ({f['duracion'] or '—'})")
    print(f"  filas destino {_n(f['filas_destino'])}")
    print(f"  entradas {f['n_entradas']} · salidas {f['n_salidas']}   ·   "
          "pruebas " + "/".join(str(f["pruebas"].get(k, 0)) for k in
                                ("OK", "OMITIDO", "ADVERTENCIA", "BLOQUEANTE"))
          + "  (OK/OMI/ADV/BLQ)")
    a = f.get("analitica")
    if a:
        print(f"  ANALÍTICA     {a['veredicto']}")
        print(f"     variables  {a['variables']}  "
              f"({a['banderas_con_problema']} con problema)   ·   "
              f"ambiente {a['ambiente']}")
        print(f"     salidas    {', '.join(a['salidas']) or '—'}")
        print(f"     transforma {_n(a['filas_entrada'])} → "
              f"{_n(a['filas_salida'])} filas   "
              f"(delta {_n(a['delta'])} · conserva {a['pct_conservado']} %)")
        print(f"     columnas   {a['columnas_conservadas']} conservadas · "
              f"{a['columnas_descartadas']} descartadas · "
              f"{a['columnas_derivadas']} derivadas")

print("\n" + "-" * 78)
print(f"ARCHIVOS PRODUCIDOS  ({len(ARCHIVOS)})")
print("-" * 78)
for clase in ("informe", "corrida", "dependencias", "analitica_general", "analitica", "veredicto", "veredicto_fecha"):
    grupo = [a for a in ARCHIVOS if a["clase"] == clase]
    if not grupo:
        continue
    print(f"\n{clase.upper()}")
    for a in grupo:
        print(f"  {a['ruta']}")
        print(f"     {a['bytes']:,} bytes".replace(",", ".")
              + f"   ·   {a['contiene']}")

if PENDIENTES:
    print(f"\nNO QUEDARON EN DISCO  ({len(PENDIENTES)})")
    for a in PENDIENTES:
        print(f"  {a['ruta']}")
        print(f"     {a['contiene']}")
    print("\n  El contenido sigue en memoria: `consolidado`, `POR_FLUJO`"
          + (", `ANALITICA`" if _ANALITICA else "") + " e `INFORME`.")

print("\n" + "=" * 78)
print(f"índice del informe: {ruta_idx}"
      + (f"   ({tam_idx:,} bytes)".replace(",", ".") if not err_idx
         else f"   FALLÓ: {err_idx}"))
print("Ese archivo trae las cifras y la ruta de todo lo demás.")
print("=" * 78)